# DeepWeeds — notebook Kaggle độc lập

Chỉ upload **notebook này và dataset**; không cần upload code Python, eval.py hoặc clone repo.
Code đánh giá và huấn luyện được gói sẵn, tự ghi ra thư mục làm việc khi chạy.

Thêm dataset vào Input; ảnh và CSV có thể nằm chung một dataset hoặc ở hai dataset riêng.
Bật GPU và Internet (cần Internet khi tải trọng số ImageNet hoặc cài thư viện thiếu).
Chạy `setup` trước, sau đó `smoke`, `backbones`, `training`, `inference`, `final`, `export`.
Mọi lựa chọn dựa trên val; chỉ bước final dùng test.


## 0. Cấu hình Kaggle

Cell dưới chứa toàn bộ code nguồn. Khi chạy, notebook tự tạo code trong
`/kaggle/working/deepweeds-lab/` và tìm dataset trong `/kaggle/input/`.
Input chỉ được đọc; checkpoint, log và biểu đồ ghi vào working.

Nếu có nhiều bản dataset trùng nhau, điền MANUAL_IMAGES_DIR và MANUAL_LABELS_DIR.
Không cần đổi đường dẫn nếu chỉ gắn một bộ ảnh và một bộ CSV tương ứng.


In [ ]:
# Chỉ chỉnh các lựa chọn chạy tại đây.
STAGE = 'setup'                   # setup | smoke | backbones | training | inference | final | export
ALLOW_FINAL_TEST = False          # bật khi chốt cấu hình và chạy final
MANUAL_IMAGES_DIR = None          # ví dụ: '/kaggle/input/my-dataset/images'
MANUAL_LABELS_DIR = None          # ví dụ: '/kaggle/input/my-dataset/data/labels'
INSTALL_MISSING_DEPENDENCIES = True


In [ ]:
from pathlib import Path
import csv, importlib.util, subprocess, sys

KAGGLE_INPUT_ROOT = Path(globals().get('KAGGLE_INPUT_ROOT', '/kaggle/input'))
LAB_WORK_ROOT = Path(globals().get('LAB_WORK_ROOT', '/kaggle/working/deepweeds-lab'))
LAB_WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Source files are embedded in this notebook; no repo download is needed.
BUNDLED_FILES = {
  "eval.py": "#!/usr/bin/env python3\r\n\"\"\"eval.py - công cụ đánh giá chính thức của Lab Day 2 (DeepWeeds).\r\n\r\nFile này đã hoàn chỉnh: sinh viên KHÔNG sửa công thức chỉ số. Mọi con số trong\r\nresults.xlsx và báo cáo phải khớp với kết quả của file này (RUBRIC.md, mục I).\r\nChỉ phụ thuộc numpy và pandas.\r\n\r\nĐịnh nghĩa chỉ số (README.md, mục 2.2)\r\n--------------------------------------\r\n- top-1 accuracy : số ảnh đoán đúng / tổng số ảnh, không trọng số theo lớp.\r\n- macro-F1       : trung bình cộng F1 của 9 lớp (lớp không có dự đoán đúng có F1 = 0).\r\n- balanced acc.  : trung bình recall của các lớp có ảnh trong tập.\r\n- precision / recall / F1 theo lớp, ma trận nhầm lẫn (hàng = nhãn thật).\r\n- ECE            : 15 bin đều theo độ tin cậy (= max softmax); bin m là (b_{m-1}, b_m].\r\n                   ECE = sum_m (n_m / n) * |acc_m - conf_m|\r\n- mean +- std    : qua các seed, std mẫu (ddof = 1).\r\n\r\nĐịnh dạng file dự đoán (mỗi seed một file)\r\n------------------------------------------\r\n    Filename, y_true, y_pred, p0, p1, ..., p8\r\np0..p8 là xác suất softmax, thứ tự theo cột `Label` của labels.csv\r\n(0 = Chinee Apple ... 7 = Snake Weed, 8 = Negatives). Tên file chứa `seed<k>`,\r\nví dụ predictions/F01_seed0_test.csv. Hàm `save_predictions(...)` trong file này ghi\r\nđúng định dạng đó (`from eval import save_predictions`).\r\n\r\nCách dùng\r\n---------\r\n    # 1) Tính chỉ số cho một nhóm file (một cấu hình, nhiều seed)\r\n    python eval.py score --pred \"predictions/F01_seed*_test.csv\" \\\r\n        --test-csv labels/test_subset0.csv --labels labels/labels.csv --tag F01\r\n\r\n    # 2) Tự chấm phần I của RUBRIC (chung kết so với mốc)\r\n    python eval.py grade \\\r\n        --final    \"predictions/F01_seed*_test.csv\" \\\r\n        --baseline \"predictions/T00_seed*_test.csv\" \\\r\n        --uncal    \"predictions/F01_uncal_seed*_test.csv\" \\\r\n        --final-val \"predictions/F01_seed*_val.csv\" \\\r\n        --latency-p95-ms 41.5 --latency-method proper \\\r\n        --test-csv labels/test_subset0.csv --labels labels/labels.csv\r\n\r\n`--uncal` (cùng cấu hình nhưng chưa temperature scaling) và `--final-val` (dự đoán\r\ntrên val) là tùy chọn; thiếu thì các ý I4(a) hoặc I4(b) ghi là \"chưa chấm được\".\r\n\"\"\"\r\nfrom __future__ import annotations\r\n\r\nimport argparse\r\nimport glob\r\nimport json\r\nimport math\r\nimport re\r\nimport sys\r\nfrom dataclasses import dataclass\r\nfrom pathlib import Path\r\n\r\nimport numpy as np\r\nimport pandas as pd\r\n\r\n# --------------------------------------------------------------------------- #\r\n# Hằng số\r\n# --------------------------------------------------------------------------- #\r\nNUM_CLASSES = 9\r\nECE_BINS = 15\r\nPROB_SUM_TOL = 1e-3  # xác suất mỗi dòng phải cộng bằng 1 (sai số cho phép)\r\n\r\n# Thứ tự lớp theo cột `Label` của labels.csv (lấy từ deepweeds.py của tác giả).\r\nCLASS_NAMES = [\r\n    \"Chinee Apple\", \"Lantana\", \"Parkinsonia\", \"Parthenium\", \"Prickly Acacia\",\r\n    \"Rubber Vine\", \"Siam Weed\", \"Snake Weed\", \"Negatives\",\r\n]\r\n\r\n# Hai lớp khó nhất và mốc recall (%) từ bài báo gốc (README.md, mục 2.3).\r\nHARD_CLASSES = {\"Chinee Apple\": 88.5, \"Snake Weed\": 88.8}\r\n\r\n# ---- Ngưỡng chấm RUBRIC mục I (TẠM THỜI: giảng viên có thể chỉnh tại đây) ---- #\r\nACC_TIERS = [(95.7, 7), (95.1, 6), (94.0, 5), (92.0, 3), (90.0, 1)]  # I1: (acc %, điểm)\r\nI2_MIN_DELTA = 0.01          # I2: Delta macro-F1 tối thiểu để đạt 5 điểm\r\nHARD_FLOORS = [(85.0, 3), (80.0, 2)]  # I3: sàn recall (%) cho cả hai lớp khó\r\nI3_MIN_POINTS = 1            # I3: có báo cáo đầy đủ nhưng dưới 80%\r\nI4_MAX_VAL_TEST_GAP = 0.02   # I4(b): |macro-F1 val - macro-F1 test| tối đa\r\nI5_P95_BUDGET_MS = 100.0     # I5: ngân sách p95 ở batch 1\r\nMIN_SEEDS = 3\r\n\r\n\r\n# --------------------------------------------------------------------------- #\r\n# Đọc và kiểm tra file dự đoán\r\n# --------------------------------------------------------------------------- #\r\n@dataclass\r\nclass Pred:\r\n    path: str\r\n    seed: int | None\r\n    filenames: np.ndarray\r\n    y_true: np.ndarray\r\n    y_pred: np.ndarray\r\n    probs: np.ndarray\r\n\r\n\r\ndef parse_seed(path: str) -> int | None:\r\n    m = re.search(r\"seed[_-]?(\\d+)\", Path(path).name)\r\n    return int(m.group(1)) if m else None\r\n\r\n\r\ndef read_pred(path: str, num_classes: int = NUM_CLASSES) -> Pred:\r\n    \"\"\"Đọc một file dự đoán và kiểm tra định dạng. Ném ValueError nếu sai.\"\"\"\r\n    df = pd.read_csv(path)\r\n    prob_cols = [f\"p{i}\" for i in range(num_classes)]\r\n    missing = [c for c in [\"Filename\", \"y_true\", \"y_pred\", *prob_cols] if c not in df.columns]\r\n    if missing:\r\n        raise ValueError(f\"{path}: thiếu cột {missing}\")\r\n    extra_p = [c for c in df.columns if re.fullmatch(r\"p\\d+\", c) and c not in prob_cols]\r\n    if extra_p:\r\n        raise ValueError(f\"{path}: có cột xác suất ngoài p0..p{num_classes - 1}: {extra_p}\")\r\n    if len(df) == 0:\r\n        raise ValueError(f\"{path}: file rỗng\")\r\n    if df[\"Filename\"].duplicated().any():\r\n        raise ValueError(f\"{path}: có Filename bị trùng\")\r\n\r\n    probs = df[prob_cols].to_numpy(dtype=np.float64)\r\n    if not np.isfinite(probs).all() or (probs < 0).any() or (probs > 1 + 1e-6).any():\r\n        raise ValueError(f\"{path}: xác suất chứa NaN, âm hoặc lớn hơn 1; hãy lưu softmax, không lưu logit\")\r\n    bad_sum = np.abs(probs.sum(1) - 1.0) > PROB_SUM_TOL\r\n    if bad_sum.any():\r\n        raise ValueError(f\"{path}: {int(bad_sum.sum())} dòng có tổng xác suất khác 1 \"\r\n                         f\"(sai số > {PROB_SUM_TOL}); hãy lưu softmax, không lưu logit\")\r\n\r\n    ints = {}\r\n    for col in (\"y_true\", \"y_pred\"):\r\n        v = pd.to_numeric(df[col], errors=\"coerce\").to_numpy(dtype=np.float64)\r\n        if not np.isfinite(v).all() or (v != np.round(v)).any():\r\n            raise ValueError(f\"{path}: cột {col} phải là số nguyên\")\r\n        v = v.astype(np.int64)\r\n        if v.min() < 0 or v.max() >= num_classes:\r\n            raise ValueError(f\"{path}: cột {col} ngoài khoảng 0..{num_classes - 1}\")\r\n        ints[col] = v\r\n\r\n    argmax = probs.argmax(1)\r\n    mismatch = int((argmax != ints[\"y_pred\"]).sum())\r\n    if mismatch:\r\n        raise ValueError(f\"{path}: {mismatch} dòng có y_pred khác argmax của p0..p{num_classes - 1}\")\r\n\r\n    return Pred(path, parse_seed(path), df[\"Filename\"].to_numpy(), ints[\"y_true\"], ints[\"y_pred\"], probs)\r\n\r\n\r\ndef save_predictions(path: str | Path, filenames, y_true, probs, num_classes: int = NUM_CLASSES) -> Path:\r\n    \"\"\"Ghi một file dự đoán đúng định dạng của eval.py: Filename, y_true, y_pred, p0..p{K-1}.\r\n\r\n    Dùng hàm này (hoặc ghi y hệt định dạng) để tạo predictions/<exp_id>_seed<k>_<split>.csv.\r\n    `probs` là XÁC SUẤT softmax (mỗi dòng cộng bằng 1), không phải logit.\r\n    y_pred được tính bằng argmax của probs. Kết quả luôn qua được `read_pred`.\r\n    \"\"\"\r\n    probs = np.asarray(probs, dtype=np.float64)\r\n    y_true = np.asarray(y_true, dtype=np.int64)\r\n    filenames = list(filenames)\r\n    if probs.ndim != 2 or probs.shape[1] != num_classes:\r\n        raise ValueError(f\"probs phải có dạng (N, {num_classes}), nhận {probs.shape}\")\r\n    if not (len(filenames) == len(y_true) == len(probs)):\r\n        raise ValueError(\"filenames, y_true, probs phải cùng độ dài\")\r\n    if np.abs(probs.sum(1) - 1.0).max() > PROB_SUM_TOL:\r\n        raise ValueError(\"probs chưa chuẩn hoá (tổng mỗi dòng khác 1); hãy áp dụng softmax trước khi lưu\")\r\n    df = pd.DataFrame({\"Filename\": filenames, \"y_true\": y_true, \"y_pred\": probs.argmax(1)})\r\n    for i in range(num_classes):\r\n        df[f\"p{i}\"] = probs[:, i]\r\n    path = Path(path)\r\n    path.parent.mkdir(parents=True, exist_ok=True)\r\n    df.to_csv(path, index=False, float_format=\"%.8g\")\r\n    return path\r\n\r\n\r\ndef check_against_csv(pred: Pred, csv_path: str, what: str = \"test\") -> None:\r\n    \"\"\"Đối chiếu tên file và nhãn thật với file CSV chia sẵn (Filename, Label).\"\"\"\r\n    ref = pd.read_csv(csv_path)\r\n    if not {\"Filename\", \"Label\"} <= set(ref.columns):\r\n        raise ValueError(f\"{csv_path}: cần cột Filename và Label\")\r\n    ref_map = dict(zip(ref[\"Filename\"], ref[\"Label\"].astype(int)))\r\n    names = set(pred.filenames)\r\n    if names != set(ref_map):\r\n        only_pred = len(names - set(ref_map))\r\n        only_ref = len(set(ref_map) - names)\r\n        raise ValueError(f\"{pred.path}: không khớp {what} CSV ({csv_path}): \"\r\n                         f\"thừa {only_pred} ảnh, thiếu {only_ref} ảnh\")\r\n    wrong = sum(int(ref_map[f]) != int(t) for f, t in zip(pred.filenames, pred.y_true))\r\n    if wrong:\r\n        raise ValueError(f\"{pred.path}: {wrong} dòng có y_true khác Label trong {csv_path}\")\r\n\r\n\r\ndef load_names(labels_csv: str | None) -> list[str]:\r\n    \"\"\"Lấy tên lớp từ labels.csv (cột Label, Species) nếu có; nếu không dùng mặc định.\"\"\"\r\n    if not labels_csv:\r\n        return list(CLASS_NAMES)\r\n    df = pd.read_csv(labels_csv)\r\n    if not {\"Label\", \"Species\"} <= set(df.columns):\r\n        raise ValueError(f\"{labels_csv}: cần cột Label và Species\")\r\n    m = df.drop_duplicates(\"Label\").sort_values(\"Label\")\r\n    if list(m[\"Label\"]) != list(range(NUM_CLASSES)):\r\n        raise ValueError(f\"{labels_csv}: Label phải là 0..{NUM_CLASSES - 1}\")\r\n    return [str(s) for s in m[\"Species\"]]\r\n\r\n\r\ndef _norm(name: str) -> str:\r\n    return re.sub(r\"[^a-z]\", \"\", name.lower())\r\n\r\n\r\ndef hard_class_indices(names: list[str]) -> dict[str, int]:\r\n    out = {}\r\n    for hard in HARD_CLASSES:\r\n        for i, n in enumerate(names):\r\n            if _norm(n) == _norm(hard):\r\n                out[hard] = i\r\n    missing = set(HARD_CLASSES) - set(out)\r\n    if missing:\r\n        raise ValueError(f\"không tìm thấy lớp {sorted(missing)} trong danh sách tên lớp {names}\")\r\n    return out\r\n\r\n\r\ndef expand(pattern: str | list[str]) -> list[str]:\r\n    patterns = [pattern] if isinstance(pattern, str) else list(pattern)\r\n    files: list[str] = []\r\n    for p in patterns:\r\n        files.extend(glob.glob(p))\r\n    files = sorted(set(files))\r\n    if not files:\r\n        raise FileNotFoundError(f\"không có file nào khớp: {patterns}\")\r\n    return files\r\n\r\n\r\n# --------------------------------------------------------------------------- #\r\n# Chỉ số\r\n# --------------------------------------------------------------------------- #\r\ndef confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, k: int = NUM_CLASSES) -> np.ndarray:\r\n    cm = np.zeros((k, k), dtype=np.int64)\r\n    np.add.at(cm, (y_true, y_pred), 1)\r\n    return cm\r\n\r\n\r\ndef per_class(cm: np.ndarray) -> dict[str, np.ndarray]:\r\n    tp = np.diag(cm).astype(np.float64)\r\n    support = cm.sum(1).astype(np.float64)\r\n    predicted = cm.sum(0).astype(np.float64)\r\n    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted > 0)\r\n    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)\r\n    denom = precision + recall\r\n    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)\r\n    return {\"precision\": precision, \"recall\": recall, \"f1\": f1, \"support\": support}\r\n\r\n\r\ndef ece_score(probs: np.ndarray, y_true: np.ndarray, bins: int = ECE_BINS) -> float:\r\n    \"\"\"ECE với `bins` bin đều; độ tin cậy = max softmax; bin m là (b_{m-1}, b_m].\"\"\"\r\n    conf = probs.max(1)\r\n    correct = (probs.argmax(1) == y_true).astype(np.float64)\r\n    idx = np.clip(np.ceil(conf * bins).astype(np.int64) - 1, 0, bins - 1)\r\n    n = len(conf)\r\n    ece = 0.0\r\n    for m in range(bins):\r\n        mask = idx == m\r\n        if mask.any():\r\n            ece += mask.sum() / n * abs(correct[mask].mean() - conf[mask].mean())\r\n    return float(ece)\r\n\r\n\r\ndef compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, probs: np.ndarray,\r\n                    k: int = NUM_CLASSES, bins: int = ECE_BINS) -> dict:\r\n    cm = confusion_matrix(y_true, y_pred, k)\r\n    pc = per_class(cm)\r\n    has_support = pc[\"support\"] > 0\r\n    nll = float(-np.log(np.clip(probs[np.arange(len(y_true)), y_true], 1e-12, None)).mean())\r\n    return {\r\n        \"n\": int(len(y_true)),\r\n        \"top1\": float((y_true == y_pred).mean()),\r\n        \"macro_f1\": float(pc[\"f1\"].mean()),\r\n        \"balanced_acc\": float(pc[\"recall\"][has_support].mean()),\r\n        \"ece\": ece_score(probs, y_true, bins),\r\n        \"nll\": nll,\r\n        \"precision\": pc[\"precision\"], \"recall\": pc[\"recall\"], \"f1\": pc[\"f1\"],\r\n        \"support\": pc[\"support\"], \"confusion\": cm,\r\n    }\r\n\r\n\r\nSCALARS = (\"top1\", \"macro_f1\", \"balanced_acc\", \"ece\", \"nll\")\r\nVECTORS = (\"precision\", \"recall\", \"f1\")\r\n\r\n\r\ndef mean_std(values) -> tuple[float, float]:\r\n    \"\"\"mean và std mẫu (ddof=1) theo trục seed; std là NaN nếu chỉ có 1 seed.\"\"\"\r\n    a = np.asarray(values, dtype=np.float64)\r\n    mean = a.mean(axis=0)\r\n    std = a.std(axis=0, ddof=1) if len(a) > 1 else np.full(a.shape[1:], np.nan)\r\n    if a.ndim == 1:\r\n        return float(mean), float(std)\r\n    return mean, std\r\n\r\n\r\n@dataclass\r\nclass Group:\r\n    preds: list[Pred]\r\n    metrics: list[dict]\r\n    summary: dict  # {\"top1\": (mean, std), ..., \"recall\": (vec_mean, vec_std), ...}\r\n\r\n    @property\r\n    def seeds(self) -> list[int | None]:\r\n        return [p.seed for p in self.preds]\r\n\r\n\r\ndef load_group(pattern, ref_csv: str | None, k: int = NUM_CLASSES, ref_what: str = \"test\") -> Group:\r\n    files = expand(pattern)\r\n    preds = [read_pred(f, k) for f in files]\r\n    if ref_csv:\r\n        for p in preds:\r\n            check_against_csv(p, ref_csv, ref_what)\r\n    base_names = set(preds[0].filenames)\r\n    for p in preds[1:]:\r\n        if set(p.filenames) != base_names:\r\n            raise ValueError(f\"{p.path}: tập Filename khác {preds[0].path}\")\r\n    seeds = [p.seed for p in preds if p.seed is not None]\r\n    if len(seeds) != len(set(seeds)):\r\n        raise ValueError(f\"seed bị trùng giữa các file: {files}\")\r\n    metrics = [compute_metrics(p.y_true, p.y_pred, p.probs, k) for p in preds]\r\n    summary = {key: mean_std([m[key] for m in metrics]) for key in (*SCALARS, *VECTORS)}\r\n    return Group(preds, metrics, summary)\r\n\r\n\r\n# --------------------------------------------------------------------------- #\r\n# Định dạng đầu ra\r\n# --------------------------------------------------------------------------- #\r\ndef fmt(mean: float, std: float, digits: int = 4) -> str:\r\n    if std is None or (isinstance(std, float) and math.isnan(std)):\r\n        return f\"{mean:.{digits}f} (1 seed)\"\r\n    return f\"{mean:.{digits}f} ± {std:.{digits}f}\"\r\n\r\n\r\ndef report_group(name: str, g: Group, names: list[str]) -> str:\r\n    lines = [f\"### {name} ({len(g.preds)} seed: {g.seeds}; {g.metrics[0]['n']} ảnh)\", \"\"]\r\n    if len(g.preds) < MIN_SEEDS:\r\n        lines += [f\"> CẢNH BÁO: chỉ có {len(g.preds)} seed, yêu cầu tối thiểu {MIN_SEEDS}.\", \"\"]\r\n    lines += [\"| Chỉ số | mean ± std |\", \"|---|---|\"]\r\n    labels = {\"top1\": \"top-1 accuracy\", \"macro_f1\": \"macro-F1\", \"balanced_acc\": \"balanced accuracy\",\r\n              \"ece\": \"ECE (15 bin)\", \"nll\": \"NLL\"}\r\n    for key in SCALARS:\r\n        lines.append(f\"| {labels[key]} | {fmt(*g.summary[key])} |\")\r\n    lines += [\"\", \"| Lớp | Precision | Recall | F1 | Số ảnh |\", \"|---|---|---|---|---|\"]\r\n    pm, ps = g.summary[\"precision\"]\r\n    rm, rs = g.summary[\"recall\"]\r\n    fm, fs = g.summary[\"f1\"]\r\n    for i, n in enumerate(names):\r\n        lines.append(f\"| {n} | {fmt(pm[i], ps[i], 3)} | {fmt(rm[i], rs[i], 3)} | \"\r\n                     f\"{fmt(fm[i], fs[i], 3)} | {int(g.metrics[0]['support'][i])} |\")\r\n    return \"\\n\".join(lines)\r\n\r\n\r\ndef save_group(out_dir: Path, tag: str, g: Group, names: list[str]) -> None:\r\n    out_dir.mkdir(parents=True, exist_ok=True)\r\n    rows = []\r\n    for p, m in zip(g.preds, g.metrics):\r\n        rows.append({\"file\": Path(p.path).name, \"seed\": p.seed,\r\n                     **{k: m[k] for k in (\"n\", *SCALARS)}})\r\n    pd.DataFrame(rows).to_csv(out_dir / f\"{tag}_per_seed.csv\", index=False)\r\n    pc = pd.DataFrame({\"class\": names})\r\n    for key in VECTORS:\r\n        mean, std = g.summary[key]\r\n        pc[f\"{key}_mean\"], pc[f\"{key}_std\"] = mean, std\r\n    pc[\"support\"] = g.metrics[0][\"support\"].astype(int)\r\n    pc.to_csv(out_dir / f\"{tag}_per_class.csv\", index=False)\r\n    cm = sum(m[\"confusion\"] for m in g.metrics)\r\n    pd.DataFrame(cm, index=[f\"true_{n}\" for n in names],\r\n                 columns=[f\"pred_{n}\" for n in names]).to_csv(out_dir / f\"{tag}_confusion_sum.csv\")\r\n    summary = {\"seeds\": g.seeds,\r\n               **{k: {\"mean\": g.summary[k][0], \"std\": g.summary[k][1]} for k in SCALARS},\r\n               **{k: {\"mean\": g.summary[k][0].tolist(), \"std\": g.summary[k][1].tolist()} for k in VECTORS},\r\n               \"classes\": names}\r\n    (out_dir / f\"{tag}_summary.json\").write_text(json.dumps(summary, indent=2, ensure_ascii=False))\r\n\r\n\r\n# --------------------------------------------------------------------------- #\r\n# Lệnh `score`\r\n# --------------------------------------------------------------------------- #\r\ndef cmd_score(args) -> int:\r\n    names = load_names(args.labels)\r\n    g = load_group(args.pred, args.test_csv, ref_what=\"test\")\r\n    if not args.test_csv:\r\n        print(\"LƯU Ý: chưa đối chiếu với test CSV (thêm --test-csv labels/test_subset0.csv)\\n\")\r\n    print(report_group(args.tag, g, names))\r\n    if args.out:\r\n        save_group(Path(args.out), args.tag, g, names)\r\n        print(f\"\\nĐã lưu kết quả vào {args.out}/{args.tag}_*\")\r\n    return 0\r\n\r\n\r\n# --------------------------------------------------------------------------- #\r\n# Lệnh `grade` (RUBRIC mục I)\r\n# --------------------------------------------------------------------------- #\r\ndef points_i1(acc_pct: float) -> int:\r\n    for thr, pts in ACC_TIERS:\r\n        if acc_pct >= thr:\r\n            return pts\r\n    return 0\r\n\r\n\r\ndef points_i2(delta: float, s: float) -> int:\r\n    if not math.isfinite(delta) or delta <= 0:\r\n        return 0\r\n    s = 0.0 if not math.isfinite(s) else s\r\n    if delta > s and delta >= I2_MIN_DELTA:\r\n        return 5\r\n    return 4 if delta > s else 2\r\n\r\n\r\ndef points_i3(recalls_pct: dict[str, float]) -> int:\r\n    if all(recalls_pct[c] >= ref for c, ref in HARD_CLASSES.items()):\r\n        return 4\r\n    low = min(recalls_pct.values())\r\n    for floor, pts in HARD_FLOORS:\r\n        if low >= floor:\r\n            return pts\r\n    return I3_MIN_POINTS\r\n\r\n\r\ndef cmd_grade(args) -> int:\r\n    names = load_names(args.labels)\r\n    hard_idx = hard_class_indices(names)\r\n    final = load_group(args.final, args.test_csv, ref_what=\"test\")\r\n    base = load_group(args.baseline, args.test_csv, ref_what=\"test\")\r\n    if not args.test_csv:\r\n        print(\"LƯU Ý: chưa đối chiếu với test CSV (thêm --test-csv labels/test_subset0.csv)\\n\")\r\n    warnings: list[str] = []\r\n    for label, g in ((\"chung kết\", final), (\"mốc\", base)):\r\n        if len(g.preds) < MIN_SEEDS:\r\n            warnings.append(f\"nhóm {label} chỉ có {len(g.preds)} seed (< {MIN_SEEDS})\")\r\n    if set(final.preds[0].filenames) != set(base.preds[0].filenames):\r\n        warnings.append(\"tập Filename của chung kết và mốc khác nhau\")\r\n\r\n    items: list[tuple[str, str, int | None, int, str]] = []  # (mã, tiêu chí, điểm, tối đa, ghi chú)\r\n\r\n    # I1\r\n    acc = final.summary[\"top1\"][0] * 100\r\n    items.append((\"I1\", \"Top-1 accuracy test\", points_i1(acc), 7, f\"{acc:.2f}% (mean {len(final.preds)} seed)\"))\r\n\r\n    # I2\r\n    mf_f, sd_f = final.summary[\"macro_f1\"]\r\n    mf_b, sd_b = base.summary[\"macro_f1\"]\r\n    delta = mf_f - mf_b\r\n    s = max([v for v in (sd_f, sd_b) if math.isfinite(v)], default=math.nan)\r\n    items.append((\"I2\", \"Macro-F1 cải thiện so với mốc\", points_i2(delta, s), 5,\r\n                  f\"final {mf_f:.4f}, mốc {mf_b:.4f}, Δ={delta:+.4f}, s={s:.4f}\"))\r\n\r\n    # I3\r\n    rec_mean = final.summary[\"recall\"][0] * 100\r\n    recalls = {c: float(rec_mean[i]) for c, i in hard_idx.items()}\r\n    note = \", \".join(f\"{c} {recalls[c]:.1f}% (mốc {HARD_CLASSES[c]}%)\" for c in HARD_CLASSES)\r\n    items.append((\"I3\", \"Recall hai lớp khó\", points_i3(recalls), 4, note))\r\n\r\n    # I4(a)\r\n    if args.uncal:\r\n        uncal = load_group(args.uncal, args.test_csv, ref_what=\"test\")\r\n        e_after, e_before = final.summary[\"ece\"][0], uncal.summary[\"ece\"][0]\r\n        items.append((\"I4a\", \"ECE sau TS < ECE trước\", int(e_after < e_before), 1,\r\n                      f\"trước {e_before:.4f}, sau {e_after:.4f}\"))\r\n    else:\r\n        items.append((\"I4a\", \"ECE sau TS < ECE trước\", None, 1, \"chưa chấm được (thiếu --uncal)\"))\r\n\r\n    # I4(b)\r\n    if args.final_val:\r\n        fv = load_group(args.final_val, args.val_csv, ref_what=\"val\")\r\n        gap = abs(fv.summary[\"macro_f1\"][0] - mf_f)\r\n        items.append((\"I4b\", \"Chênh macro-F1 val/test <= %.2f\" % I4_MAX_VAL_TEST_GAP,\r\n                      int(gap <= I4_MAX_VAL_TEST_GAP), 1,\r\n                      f\"val {fv.summary['macro_f1'][0]:.4f}, test {mf_f:.4f}, chênh {gap:.4f}\"))\r\n    else:\r\n        items.append((\"I4b\", \"Chênh macro-F1 val/test <= %.2f\" % I4_MAX_VAL_TEST_GAP, None, 1,\r\n                      \"chưa chấm được (thiếu --final-val)\"))\r\n\r\n    # I5\r\n    if args.latency_p95_ms is None:\r\n        items.append((\"I5\", \"Cấu hình thời gian thực\", None, 2, \"chưa chấm được (thiếu --latency-p95-ms)\"))\r\n    else:\r\n        ok = args.latency_p95_ms <= I5_P95_BUDGET_MS\r\n        pts = 0 if not ok else (2 if args.latency_method == \"proper\" else 1)\r\n        items.append((\"I5\", \"Cấu hình thời gian thực\", pts, 2,\r\n                      f\"p95 = {args.latency_p95_ms:.1f} ms (ngân sách {I5_P95_BUDGET_MS:.0f} ms), \"\r\n                      f\"đo {'đúng cách' if args.latency_method == 'proper' else 'chưa đủ warmup/synchronize'}\"))\r\n\r\n    scored = [i for i in items if i[2] is not None]\r\n    got = sum(i[2] for i in scored)\r\n    max_scored = sum(i[3] for i in scored)\r\n    lines = [\"## Tự chấm RUBRIC mục I (đề xuất; giảng viên xác nhận)\", \"\",\r\n             \"| Mã | Tiêu chí | Điểm | Tối đa | Chi tiết |\", \"|---|---|---|---|---|\"]\r\n    for code, crit, pts, mx, note in items:\r\n        lines.append(f\"| {code} | {crit} | {'-' if pts is None else pts} | {mx} | {note} |\")\r\n    lines += [\"\", f\"**Tổng các ý đã chấm: {got} / {max_scored}** (phần I tối đa 20).\"]\r\n    pending = [i[0] for i in items if i[2] is None]\r\n    if pending:\r\n        lines.append(f\"Chưa chấm được: {', '.join(pending)}.\")\r\n    if warnings:\r\n        lines += [\"\", \"Cảnh báo:\"] + [f\"- {w}\" for w in warnings]\r\n    lines += [\"\", \"Ngưỡng điểm là TẠM THỜI (xem khối hằng số đầu file eval.py và RUBRIC.md mục I).\"]\r\n    print(\"\\n\".join(lines))\r\n\r\n    if args.out:\r\n        out = Path(args.out)\r\n        out.mkdir(parents=True, exist_ok=True)\r\n        (out / \"grade_I.json\").write_text(json.dumps({\r\n            \"items\": [{\"code\": c, \"criterion\": cr, \"points\": p, \"max\": m, \"note\": n} for c, cr, p, m, n in items],\r\n            \"total\": got, \"max_scored\": max_scored, \"warnings\": warnings}, indent=2, ensure_ascii=False))\r\n    return 0\r\n\r\n\r\n# --------------------------------------------------------------------------- #\r\n# CLI\r\n# --------------------------------------------------------------------------- #\r\ndef build_parser() -> argparse.ArgumentParser:\r\n    ap = argparse.ArgumentParser(description=\"Đánh giá Lab Day 2 (DeepWeeds)\")\r\n    sub = ap.add_subparsers(dest=\"cmd\", required=True)\r\n\r\n    sc = sub.add_parser(\"score\", help=\"tính chỉ số cho một nhóm file dự đoán (nhiều seed)\")\r\n    sc.add_argument(\"--pred\", nargs=\"+\", required=True, help=\"glob của file dự đoán\")\r\n    sc.add_argument(\"--test-csv\", help=\"labels/test_subset0.csv để đối chiếu tên file và nhãn\")\r\n    sc.add_argument(\"--labels\", help=\"labels/labels.csv để lấy tên lớp\")\r\n    sc.add_argument(\"--tag\", default=\"score\")\r\n    sc.add_argument(\"--out\", help=\"thư mục lưu JSON/CSV\")\r\n    sc.set_defaults(func=cmd_score)\r\n\r\n    gr = sub.add_parser(\"grade\", help=\"tự chấm phần I của RUBRIC\")\r\n    gr.add_argument(\"--final\", nargs=\"+\", required=True, help=\"glob dự đoán test của cấu hình chung kết\")\r\n    gr.add_argument(\"--baseline\", nargs=\"+\", required=True, help=\"glob dự đoán test của mốc (T00 + I00)\")\r\n    gr.add_argument(\"--uncal\", nargs=\"+\", help=\"glob dự đoán test của chung kết khi chưa temperature scaling\")\r\n    gr.add_argument(\"--final-val\", nargs=\"+\", help=\"glob dự đoán val của chung kết\")\r\n    gr.add_argument(\"--latency-p95-ms\", type=float, help=\"p95 batch 1 của cấu hình thời gian thực (ms)\")\r\n    gr.add_argument(\"--latency-method\", choices=[\"proper\", \"partial\"], default=\"proper\",\r\n                    help=\"proper = có warmup và synchronize; partial = chưa đủ\")\r\n    gr.add_argument(\"--test-csv\", help=\"labels/test_subset0.csv\")\r\n    gr.add_argument(\"--val-csv\", help=\"labels/val_subset0.csv (đối chiếu --final-val)\")\r\n    gr.add_argument(\"--labels\", help=\"labels/labels.csv\")\r\n    gr.add_argument(\"--out\", help=\"thư mục lưu grade_I.json\")\r\n    gr.set_defaults(func=cmd_grade)\r\n    return ap\r\n\r\n\r\ndef main(argv=None) -> int:\r\n    args = build_parser().parse_args(argv)\r\n    try:\r\n        return args.func(args)\r\n    except (ValueError, FileNotFoundError) as e:\r\n        print(f\"LỖI: {e}\", file=sys.stderr)\r\n        return 2\r\n\r\n\r\nif __name__ == \"__main__\":\r\n    sys.exit(main())\r\n",
  "README.md": "# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds\r\n\r\n> Track 4 · Ngày 2 · *Tích chập, chuỗi, attention · backbone · huấn luyện · suy luận*\r\n> Bài lab này mở rộng **Lab #2** trong slide Day 2. Slide chỉ yêu cầu 1 backbone, 3 cách khởi tạo, có/không CutMix và TTA. Ở đây bạn làm đầy đủ: **≥ 5 backbone**, **nhiều công thức huấn luyện**, **nhiều cách suy luận**, rồi chọn cấu hình tốt nhất và báo cáo.\r\n\r\nRepo gồm **hướng dẫn, tiêu chí chấm, bộ khung code (pseudo-code) và công cụ đánh giá**. Bộ khung `starter/` chỉ có chữ ký hàm, docstring và các bước `TODO`: **bạn tự viết phần ruột** (model, loss, augmentation, vòng huấn luyện, TTA, đo độ trễ). Riêng `eval.py` đã hoàn chỉnh, bạn không sửa. Làm vậy để bạn hiểu từng thành phần trong slide, nhưng vẫn đo bằng cùng một thước.\r\n\r\n| File | Dùng để làm gì |\r\n|---|---|\r\n| `README.md` (file này) | Tổng quan, dataset, quy tắc chia dữ liệu, cách đánh giá, sản phẩm phải nộp, cách nộp bài |\r\n| [`GUIDE.md`](GUIDE.md) | Quy trình từng bước, danh sách thí nghiệm, cấu trúc file xlsx và báo cáo, bẫy thường gặp |\r\n| [`RUBRIC.md`](RUBRIC.md) | Thang điểm 100, tiêu chí đạt, lỗi bị trừ điểm |\r\n| [`eval.py`](eval.py) | **Đã hoàn chỉnh.** Tính chỉ số đúng định nghĩa ở mục 2.2 và tự chấm phần I của RUBRIC (mục 2.4) |\r\n| [`starter/`](starter) | **Pseudo-code** để bạn hoàn thiện: `dataset.py`, `model.py`, `losses.py`, `train.py`, `inference.py`, `benchmark.py`, `lab_day2.ipynb` |\r\n| [`tests/`](tests) | Test của `eval.py` và của bộ khung (chạy được không cần GPU) |\r\n\r\n---\r\n\r\n## 1. Mục tiêu học tập\r\n\r\nSau bài lab, bạn có thể:\r\n\r\n1. So sánh công bằng nhiều backbone (CNN và transformer) trên cùng một bài toán, cùng một công thức huấn luyện.\r\n2. Đo riêng đóng góp của từng yếu tố trong **công thức huấn luyện** (khởi tạo, augmentation, loss, optimizer/LR, EMA). Slide chương 5 nhấn mạnh công thức quan trọng ngang kiến trúc.\r\n3. So sánh các **kỹ thuật suy luận** (TTA, độ phân giải kiểm tra, ensemble, hiệu chuẩn, gộp BatchNorm/FP16) bằng cả độ chính xác lẫn **độ trễ đo đúng cách**.\r\n4. Phân biệt chênh lệch thật với **nhiễu** do hạt giống (seed) bằng mean ± std.\r\n5. Rút ra kết luận có bằng chứng, nêu rõ cấu hình nào tốt nhất và vì sao.\r\n\r\n## 2. Dataset: DeepWeeds\r\n\r\n| Thuộc tính | Giá trị |\r\n|---|---|\r\n| Bài toán | Phân loại ảnh cỏ dại ngoài đồng (robot nông nghiệp, Queensland, Úc) |\r\n| Số ảnh | 17.509 ảnh RGB 256×256 |\r\n| Số lớp | 9: 8 loài cỏ dại + `Negative` (thực vật không phải loài mục tiêu) |\r\n| Đặc điểm | **Mất cân bằng lớp**: `Negative` có 9.106 ảnh (khoảng 52%), mỗi loài cỏ có 1.009–1.125 ảnh (xem bảng dưới). Số liệu theo Table 1 của bài báo; bạn phải tự đếm lại ở bước EDA và đối chiếu |\r\n| Dung lượng | Khoảng 490 MB |\r\n| Giấy phép | CC BY 4.0 |\r\n| Bài báo | Olsen et al., *DeepWeeds: A Multiclass Weed Species Image Dataset for Deep Learning*, Scientific Reports 9, 2058 (2019), [doi:10.1038/s41598-018-38343-3](https://doi.org/10.1038/s41598-018-38343-3) |\r\n| Baseline tham khảo | Bài báo báo cáo ResNet-50 đạt 95,7% và Inception-v3 đạt 95,1% (chi tiết và điều kiện huấn luyện ở mục 2.3) |\r\n\r\nSố ảnh theo lớp (Table 1 của bài báo, tổng 17.509):\r\n\r\n| Lớp | Số ảnh | Lớp | Số ảnh |\r\n|---|---|---|---|\r\n| Chinee apple | 1.125 | Rubber vine | 1.009 |\r\n| Lantana | 1.064 | Siam weed | 1.074 |\r\n| Parkinsonia | 1.031 | Snake weed | 1.016 |\r\n| Parthenium | 1.022 | **Negative** | **9.106** |\r\n| Prickly acacia | 1.062 | | |\r\n\r\n**Nơi tải:**\r\n\r\n- Ảnh: [Zenodo, DOI 10.5281/zenodo.7939060](https://zenodo.org/record/7939059), file `images.zip`.\r\n  - Link trực tiếp: `https://zenodo.org/records/7939060/files/images.zip?download=1`\r\n  - MD5: `b7b30f96d466fba86016aa5a26606e0f`. Hãy kiểm tra checksum sau khi tải.\r\n- Nhãn và các fold chia sẵn: [github.com/AlexOlsen/DeepWeeds](https://github.com/AlexOlsen/DeepWeeds), thư mục `labels/` gồm `labels.csv`, `train_subset{0-4}.csv`, `val_subset{0-4}.csv`, `test_subset{0-4}.csv` (cột `Filename, Label, Species`; chia 60/20/20).\r\n- Ngoài ra có trong [TensorFlow Datasets](https://www.tensorflow.org/datasets/catalog/deep_weeds), nhưng nên dùng file CSV ở trên để mọi người dùng **cùng một cách chia**.\r\n\r\n### 2.1 Quy tắc chia train / val / test (BẮT BUỘC)\r\n\r\nDataset được tác giả chia sẵn thành 5 fold, mỗi fold là bộ ba file CSV 60% train / 20% val / 20% test. Chia ngẫu nhiên có phân tầng theo lớp (riêng lớp `Negative` không phân tầng), không chia theo địa điểm. **Mọi sinh viên dùng cùng một cách chia để kết quả so sánh được.**\r\n\r\n| # | Quy tắc |\r\n|---|---|\r\n| S1 | Dùng **fold 0**: `train_subset0.csv`, `val_subset0.csv`, `test_subset0.csv`. Tải nguyên bản từ GitHub của tác giả, **không sửa, không lọc, không chia lại**. |\r\n| S2 | **Train** chỉ để cập nhật trọng số. **Val** dùng để chọn backbone, siêu tham số, phương pháp suy luận, checkpoint (early stopping) và khớp nhiệt độ T. **Test** chỉ để báo cáo kết quả cuối (xem 2.2). |\r\n| S3 | **Không gộp val vào train**, kể cả ở lần huấn luyện cuối cùng. Không huấn luyện trên test. |\r\n| S4 | Không dùng bất kỳ thông tin nào từ test để quyết định gì: siêu tham số, ngưỡng, nhiệt độ T, thống kê chuẩn hoá, chọn model, chọn phương pháp suy luận. |\r\n| S5 | Seed chỉ thay đổi khởi tạo head, thứ tự batch và augmentation ngẫu nhiên. **Seed không được thay đổi cách chia.** |\r\n| S6 | Fold 1–4 chỉ dùng cho điểm thưởng, và khi dùng thì dùng đủ bộ ba file của cùng một fold. |\r\n\r\n**Kiểm tra bắt buộc trước khi train** (in kết quả ra notebook và ghi vào báo cáo):\r\n\r\n1. Số ảnh mỗi tập và số ảnh mỗi lớp trong từng tập. Tỉ lệ kỳ vọng xấp xỉ 60/20/20, tức khoảng 10.505 / 3.502 / 3.502 ảnh (số suy ra từ tỉ lệ, **bạn ghi số đếm thật**).\r\n2. Giao của từng cặp tập (train∩val, train∩test, val∩test) theo tên file phải **rỗng**; hợp ba tập phải bằng đúng 17.509 ảnh.\r\n3. Mọi file trong CSV đều tồn tại trong thư mục ảnh.\r\n\r\nNếu số đếm lệch rõ rệt khỏi 60/20/20 (hơn khoảng 1 điểm phần trăm) hoặc giao khác rỗng, hãy báo giảng viên trước khi chạy tiếp.\r\n\r\n### 2.2 Cách đánh giá (BẮT BUỘC)\r\n\r\n**Tập đánh giá và thời điểm dùng:**\r\n\r\n| Giai đoạn | Tập dùng | Mục đích |\r\n|---|---|---|\r\n| Sàng backbone, ablation huấn luyện, so sánh suy luận | **val** | Chọn cấu hình |\r\n| Chọn checkpoint trong một lần chạy | **val** | Epoch có macro-F1 val cao nhất (hòa thì lấy epoch sớm hơn) |\r\n| Khớp nhiệt độ T (temperature scaling) | **val** | Một T duy nhất, áp dụng sang test |\r\n| **Chung kết** (GUIDE mục 5) | **test** | Chạy **đúng một lần cho mỗi seed**, trên **toàn bộ** tập test (không lấy mẫu con, không loại ảnh) |\r\n\r\n**Tiền xử lý lúc đánh giá:** không dùng augmentation ngẫu nhiên; chỉ resize hoặc center-crop và chuẩn hoá giống hệt lúc val. `model.eval()`. Ngoại lệ chỉ là các thí nghiệm suy luận cố ý đổi tiền xử lý (TTA, độ phân giải kiểm tra); khi đó khai báo rõ trong `results.xlsx`.\r\n\r\n**Định nghĩa chỉ số** (mọi con số trong bảng và báo cáo phải theo định nghĩa này):\r\n\r\n| Chỉ số | Định nghĩa |\r\n|---|---|\r\n| **Top-1 accuracy** | Số ảnh dự đoán đúng / tổng số ảnh của tập, không trọng số theo lớp |\r\n| **Macro-F1** (chỉ số chính) | Trung bình cộng F1 của **9 lớp**, mỗi lớp trọng số bằng nhau (ví dụ `sklearn.metrics.f1_score(average=\"macro\")`) |\r\n| Balanced accuracy | Trung bình recall của 9 lớp |\r\n| Precision, recall, F1 theo lớp | Tính riêng cho từng lớp; bắt buộc báo cáo cho **Chinee apple** và **Snake weed** (hai lớp khó nhất, xem 2.3) |\r\n| Ma trận nhầm lẫn | Số lượng ảnh, hàng là nhãn thật, cột là nhãn dự đoán |\r\n| **ECE** | 15 bin đều theo độ tin cậy; độ tin cậy = max softmax; `ECE = Σ_m (n_m / n) · abs(acc_m − conf_m)`, với `n_m` là số ảnh trong bin m, `acc_m` và `conf_m` là accuracy và độ tin cậy trung bình của bin đó (slide trang 69) |\r\n| Độ trễ | p50 / p95 / p99 theo GUIDE mục 4.1 |\r\n| mean ± std | Qua **≥ 3 seed**, std mẫu (`ddof=1`). Ghi rõ số seed |\r\n\r\nVì dữ liệu mất cân bằng (`Negative` ≈ 52%), **top-1 accuracy bị lớp `Negative` kéo cao**. Luôn báo cáo kèm macro-F1 và chỉ số từng lớp.\r\n\r\n**Quy trình chung kết (chi tiết ở GUIDE mục 5):** chốt cấu hình trên val, huấn luyện lại với ≥ 3 seed, chạy test một lần cho mỗi seed. Chạy cả **mốc so sánh** (công thức nền `T00` + suy luận 1-view `I00`) với cùng số seed để tính mức cải thiện. Với mỗi lần chạy test, **lưu file dự đoán** `predictions/<exp_id>_seed<k>_test.csv` gồm các cột `Filename, y_true, y_pred, p0, p1, …, p8` (xác suất softmax; thứ tự lớp theo cột `Label` của `labels.csv`). Giảng viên sẽ **tính lại chỉ số từ file này**; số trong báo cáo và xlsx phải khớp.\r\n\r\n### 2.3 Số tham khảo từ bài báo gốc\r\n\r\nCác số dưới đây lấy từ [bài báo gốc (Scientific Reports 2019)](https://pmc.ncbi.nlm.nih.gov/articles/PMC6375952). Rubric dùng chúng làm mốc (xem `RUBRIC.md` mục I). Đây là số **trích dẫn**, không phải kết quả của bạn.\r\n\r\n| Nội dung | Giá trị theo bài báo |\r\n|---|---|\r\n| ResNet-50, độ chính xác trung bình (weighted average, 5 fold) | **95,7%** |\r\n| Inception-v3, độ chính xác trung bình (weighted average, 5 fold) | **95,1%** |\r\n| ResNet-50, lớp tốt nhất | `Negative` 97,6%; Parkinsonia 97,2% |\r\n| ResNet-50, lớp kém nhất | **Chinee apple 88,5%**; **Snake weed 88,8%** |\r\n| Nhầm lẫn chính | 3,4% Chinee apple bị đoán thành Snake weed và 4,1% chiều ngược lại; 1,3% Parkinsonia bị đoán thành Prickly acacia |\r\n| Suy luận ResNet-50 trên Jetson TX2 | 180 ms (TensorFlow); 53,4 ms (TensorRT) |\r\n\r\n**Điều kiện huấn luyện của bài báo (rất khác bài lab này):** Keras, Adam, LR 1e-4 (giảm một nửa khi val loss không giảm sau 16 epoch), khởi tạo ImageNet, khoảng **100 epoch**, augmentation mạnh (xoay ±360°, scale, đổi màu, perspective), mỗi model train trung bình 13 giờ trên GTX 1080Ti.\r\n\r\nLưu ý khi so sánh với kết quả của bạn:\r\n\r\n- Bài lab dùng **10–15 epoch** và công thức đơn giản hơn nhiều, nên có thể thấp hơn các số trên. Đó là bình thường.\r\n- Bài báo ghi \"weighted average accuracy\"; bài lab đo top-1 accuracy không trọng số. Hai định nghĩa có thể không trùng hoàn toàn, nên so sánh chỉ mang tính tham khảo.\r\n- Các số theo lớp của bài báo được coi là tương đương recall theo lớp khi đối chiếu (giả định, bài báo không nói rõ).\r\n- Bài báo chia ngẫu nhiên, không theo địa điểm, nên điểm test có thể hơi lạc quan so với khi gặp địa điểm mới. Hãy nêu điều này trong phần *Hạn chế* của báo cáo.\r\n\r\n### 2.4 Công cụ: `eval.py` và bộ khung `starter/`\r\n\r\n**`eval.py` (đã hoàn chỉnh, chỉ cần numpy và pandas).** Mọi con số của bạn phải khớp kết quả của file này.\r\n\r\n```bash\r\n# Chỉ số của một cấu hình (nhiều seed): top-1, macro-F1, balanced acc, theo lớp, ECE, mean ± std\r\npython eval.py score --pred \"predictions/F01_seed*_test.csv\" \\\r\n    --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag F01 --out eval_out\r\n\r\n# Tự chấm phần I của RUBRIC (chung kết so với mốc)\r\npython eval.py grade --final \"predictions/F01_seed*_test.csv\" --baseline \"predictions/T00_seed*_test.csv\" \\\r\n    --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv\r\n```\r\n\r\n- `score` kiểm tra định dạng file dự đoán (đủ cột `p0..p8`, xác suất cộng bằng 1, `y_pred` đúng argmax), đối chiếu tên ảnh và nhãn với `test_subset0.csv`, rồi in bảng và lưu JSON/CSV.\r\n- `grade` tính điểm đề xuất cho I1–I4 (và I5 nếu bạn truyền `--latency-p95-ms`). Thêm `--uncal` (dự đoán test của cùng cấu hình khi chưa temperature scaling) để chấm I4(a), và `--final-val` (dự đoán val của chung kết, đặt tên `<exp_id>_seed<k>_val.csv`) để chấm I4(b). Ngưỡng điểm là **tạm thời** và nằm ở đầu file `eval.py`.\r\n- Lỗi định dạng làm `eval.py` thoát với mã 2 và in rõ file nào, dòng nào sai.\r\n\r\n**Bộ khung `starter/` (pseudo-code, bạn hoàn thiện):**\r\n\r\n| File | Bạn làm gì |\r\n|---|---|\r\n| `dataset.py` | Đọc CSV, kiểm tra chia dữ liệu (S1–S4), transform và augmentation, `Dataset`, `DataLoader` |\r\n| `model.py` | Backbone qua `timm`, đóng băng, 3 nhóm tham số (slide trang 52), đếm params/GMAC |\r\n| `losses.py` | Label smoothing, focal loss, trọng số lớp, Mixup/CutMix |\r\n| `train.py` | Một hàm `run(cfg)` dùng chung: AMP, warmup + cosine, EMA, chọn checkpoint theo macro-F1 val, vẽ đường cong |\r\n| `inference.py` | TTA, gộp xác suất/logit, ensemble, temperature scaling, gộp BatchNorm |\r\n| `benchmark.py` | Đo độ trễ p50/p95/p99 đúng cách |\r\n| `lab_day2.ipynb` | Notebook Colab/Kaggle: phần cài đặt và tải dữ liệu đã viết sẵn, các ô `TODO` là phần của bạn |\r\n\r\nCách dùng: chép `starter/` thành `code/` trong thư mục bài nộp của bạn, hoàn thiện các `TODO`, giữ nguyên **tên hàm và kiểu dữ liệu vào/ra** ghi trong docstring (bạn được thêm hàm, tham số, file mới). Chạy test của repo bằng `python -m unittest discover -s tests` (cần scikit-learn).\r\n\r\n## 3. Môi trường gợi ý: Google Colab hoặc Kaggle\r\n\r\n| Nền tảng | Ưu điểm | Lưu ý |\r\n|---|---|---|\r\n| **Kaggle Notebooks** (khuyên dùng) | GPU miễn phí theo hạn mức hằng tuần, phiên chạy ổn định, tắt trình duyệt vẫn chạy được bằng *Save & Run All* | Kiểm tra hạn mức GPU hiện hành trong tài khoản của bạn; cần bật Internet để tải dataset |\r\n| **Google Colab** | Khởi động nhanh, gắn Google Drive để lưu checkpoint | GPU miễn phí không được đảm bảo và có thể bị ngắt; phải lưu checkpoint và log ra Drive thường xuyên |\r\n\r\nGợi ý chung:\r\n\r\n- Dùng GPU (T4 trở lên), bật mixed precision (AMP).\r\n- Lưu **mọi log, đường cong, logit dự đoán và checkpoint** ra ổ bền (Drive, hoặc Kaggle output) ngay sau mỗi lần chạy, để phiên bị ngắt không mất kết quả.\r\n- Ước lượng ngân sách tính toán và mẹo tiết kiệm nằm ở [`GUIDE.md`](GUIDE.md), mục 7.\r\n\r\n## 4. Bạn phải nộp những gì\r\n\r\n| # | Sản phẩm | Yêu cầu tối thiểu |\r\n|---|---|---|\r\n| 1 | `results.xlsx` | Bảng so sánh **tất cả** thí nghiệm (backbone, training, inference, kết quả cuối, độ trễ). Cấu trúc sheet và cột ở GUIDE mục 6.1 |\r\n| 2 | `report.md` (hoặc `report.pdf`) | Báo cáo kết luận: thiết lập, kết quả, phân tích, cấu hình tốt nhất, hạn chế. Dàn ý ở GUIDE mục 6.3 |\r\n| 3 | `curves/` | **Ảnh biểu đồ training của từng thí nghiệm** (loss và metric theo epoch, train và val), một ảnh `.png` cho mỗi `exp_id` |\r\n| 4 | `code/` | **Toàn bộ code** của bạn, bắt đầu từ bộ khung `starter/` đã hoàn thiện: model, dataset/augmentation, train loop, các loss, inference/TTA/ensemble, đo độ trễ, tạo bảng và biểu đồ. Dùng `eval.py` gốc, không sửa |\r\n| 5 | `README.md` riêng của bạn | Link notebook Colab/Kaggle chạy lại được, phiên bản thư viện, lệnh/thứ tự chạy, seed đã dùng |\r\n| 6 | `predictions/` | File dự đoán trên **test** của các cấu hình chung kết và mốc, từng seed (định dạng ở mục 2.2; tạo bằng `eval.save_predictions`). Nên kèm file dự đoán **val** của chung kết (`*_val.csv`) và bản chưa temperature scaling (`*uncal*`) để `eval.py grade` chấm được I4. Dùng để giảng viên tính lại chỉ số |\r\n\r\nSố thí nghiệm tối thiểu (chi tiết ở GUIDE):\r\n\r\n- **≥ 5 backbone** (có ít nhất 1 họ transformer và ít nhất 1 mạng nhẹ).\r\n- **≥ 3 trục công thức huấn luyện**, mỗi trục thử ít nhất 2–3 giá trị (ví dụ: khởi tạo, augmentation, loss).\r\n- **≥ 4 phương pháp suy luận** và đo độ trễ p50/p95/p99.\r\n- Cấu hình cuối cùng chạy **≥ 3 seed**, báo cáo mean ± std.\r\n\r\n## 5. Cách nộp bài\r\n\r\n1. `git clone` (hoặc `git pull`) repo này để lấy bài lab.\r\n2. Tạo thư mục bài làm của bạn, đặt đúng cấu trúc sau:\r\n\r\n```\r\nsubmissions/<mssv>_<ho_ten_khong_dau>/\r\n├── README.md          # link Colab/Kaggle, cách chạy lại\r\n├── results.xlsx\r\n├── report.md          # hoặc report.pdf\r\n├── curves/\r\n│   ├── B01_resnet50.png\r\n│   ├── T03_cutmix.png\r\n│   └── ...            # mỗi exp_id một ảnh\r\n├── predictions/\r\n│   ├── F01_seed0_test.csv\r\n│   └── ...            # chung kết + mốc, mỗi seed một file\r\n└── code/\r\n    ├── *.ipynb\r\n    └── *.py           # model, train, inference, benchmark, ...\r\n```\r\n\r\n3. **Không commit** dataset (`images.zip`, ảnh) và checkpoint lớn. Chỉ commit code, `results.xlsx`, báo cáo, ảnh biểu đồ và file `predictions/` (nhỏ). Nếu cần chia sẻ checkpoint, đặt link ở README riêng của bạn.\r\n4. Nộp bài theo cách giảng viên thông báo (ví dụ Pull Request vào repo này, hoặc nén thư mục và nộp lên hệ thống của lớp). Hạn nộp do giảng viên công bố.\r\n\r\n## 6. Quy tắc trung thực học thuật\r\n\r\n- Mọi con số trong `results.xlsx` và báo cáo phải **đến từ log chạy thật** của bạn, truy ngược được tới một `exp_id` và một dòng log.\r\n- Không bịa số, không chép số từ bài báo hay slide rồi ghi như kết quả của mình. Số tham khảo từ nguồn khác phải ghi rõ là *trích dẫn* và có nguồn.\r\n- Được thảo luận ý tưởng với bạn học, nhưng code, bảng và báo cáo phải của riêng bạn.\r\n- Kết quả xấu, thí nghiệm thất bại, hoặc phát hiện \"kỹ thuật X không giúp ích\" **vẫn được điểm** nếu có phân tích tốt. Xem `RUBRIC.md`.\r\n\r\n## 7. Tài liệu đọc trước\r\n\r\n- Slide Day 2: chương 4 (backbone), chương 5 (huấn luyện), chương 6 (suy luận) và trang \"Lab #2\".\r\n- *ResNet strikes back* (Wightman et al., arXiv:2110.00476): công thức huấn luyện quan trọng ngang kiến trúc.\r\n- CutMix (arXiv:1905.04899), Focal Loss (arXiv:1708.02002), Temperature scaling (Guo et al., arXiv:1706.04599), FixRes (arXiv:1906.06423).\r\n- Karpathy, *A Recipe for Training Neural Networks*: checklist gỡ lỗi khi huấn luyện không hội tụ.\r\n",
  "GUIDE.md": "# GUIDE — Quy trình thực hiện Lab Day 2\r\n\r\nHướng dẫn này đi theo thứ tự bạn nên làm. Hãy đọc hết một lượt trước khi bắt đầu. Các số liệu và trích dẫn từ slide Day 2 ghi kèm số trang; mọi mốc thời gian GPU trong tài liệu này là **ước lượng chưa được đo**, hãy tự đo epoch đầu tiên rồi tính lại.\r\n\r\n## Mục lục\r\n\r\n0. [Nguyên tắc thực nghiệm (đọc kỹ)](#0-nguyên-tắc-thực-nghiệm)\r\n1. [Bước 0: Chuẩn bị, EDA, dựng pipeline](#1-bước-0-chuẩn-bị-eda-dựng-pipeline)\r\n2. [Bước 1: So sánh backbone](#2-bước-1-so-sánh-backbone--5)\r\n3. [Bước 2: Công thức huấn luyện](#3-bước-2-công-thức-huấn-luyện)\r\n4. [Bước 3: Phương pháp suy luận](#4-bước-3-phương-pháp-suy-luận)\r\n5. [Bước 4: Cấu hình tốt nhất và chạy test](#5-bước-4-cấu-hình-tốt-nhất-và-chạy-test)\r\n6. [Bước 5: Sản phẩm (xlsx, biểu đồ, báo cáo)](#6-bước-5-sản-phẩm)\r\n7. [Ngân sách tính toán và mẹo Colab/Kaggle](#7-ngân-sách-tính-toán-và-mẹo-colabkaggle)\r\n8. [Bẫy thường gặp](#8-bẫy-thường-gặp)\r\n9. [Câu hỏi gợi ý cho phần phân tích](#9-câu-hỏi-gợi-ý-cho-phần-phân-tích)\r\n\r\n---\r\n\r\n## 0. Nguyên tắc thực nghiệm\r\n\r\nBài lab chấm **độ chặt chẽ của thí nghiệm** ngang với kết quả cuối. Năm nguyên tắc sau áp dụng cho mọi bước.\r\n\r\n**N1. Một thay đổi mỗi lần.** Khi so sánh hai cấu hình, chỉ khác nhau đúng một yếu tố, mọi thứ còn lại giữ nguyên (cùng backbone, cùng số epoch, cùng seed khởi đầu, cùng split). Nếu đổi hai thứ cùng lúc, bạn không kết luận được yếu tố nào có tác dụng.\r\n\r\n**N2. Chia dữ liệu cố định.** Dùng `train_subset0.csv`, `val_subset0.csv`, `test_subset0.csv`. Quy tắc đầy đủ (S1–S6) và các kiểm tra bắt buộc nằm ở [`README.md` mục 2.1](README.md#21-quy-tắc-chia-train--val--test-bắt-buộc); đọc kỹ trước khi làm.\r\n\r\n| Tập | Dùng để |\r\n|---|---|\r\n| train | Cập nhật trọng số |\r\n| val | Chọn backbone, siêu tham số, phương pháp suy luận, early stopping, khớp nhiệt độ T |\r\n| test | **Chỉ dùng một lần ở Bước 4** để báo cáo con số cuối cùng |\r\n\r\nNếu bạn nhìn điểm test rồi quay lại chỉnh cấu hình, con số test không còn đáng tin và sẽ bị trừ điểm (xem `RUBRIC.md`). Bạn được lưu logit test vào file khi chạy, nhưng chưa mở hay tính chỉ số trên chúng cho tới Bước 4.\r\n\r\n**N3. Chỉ số.**\r\n\r\n- **Chỉ số chính: macro-F1** trên val. Dataset mất cân bằng nên accuracy bị lớp `Negative` kéo cao và che lỗi ở các lớp hiếm.\r\n- Chỉ số phụ: top-1 accuracy, balanced accuracy, F1 theo từng lớp, ECE (ở Bước 3).\r\n- Luôn ghi cả tên chỉ số lẫn tập đo (val hay test).\r\n- Định nghĩa chính xác từng chỉ số (macro-F1 trên 9 lớp, ECE 15 bin, mean ± std với `ddof=1`...) và quy trình đánh giá nằm ở [`README.md` mục 2.2](README.md#22-cách-đánh-giá-bắt-buộc). Mọi con số của bạn phải theo định nghĩa đó.\r\n\r\n**N4. Seed và nhiễu.** Slide (trang 59) cho biết ResNet-50 lệch chuẩn khoảng 0,10 điểm qua 100 seed, nên chênh lệch dưới ~0,3 điểm là nhiễu. Trên tập nhỏ như DeepWeeds, độ lệch có thể lớn hơn, hãy tự đo. Quy tắc:\r\n\r\n- Quét sàng (Bước 1 và 2): 1 seed là chấp nhận được, nhưng chỉ kết luận \"tốt hơn\" khi chênh lệch **rõ ràng**, và phải ghi chú là mới 1 seed.\r\n- Các cấu hình lọt vào **vòng chung kết** (Bước 4): **≥ 3 seed**, báo cáo mean ± std.\r\n- Khi chênh lệch giữa hai cấu hình nhỏ hơn std, hãy viết \"không phân biệt được\", không viết \"A tốt hơn B\".\r\n\r\n**N5. Ghi lại tất cả.** Mỗi lần chạy phải lưu: cấu hình đầy đủ (hyperparameter, seed, version thư viện), log theo epoch, đường cong, logit dự đoán trên val và test, thời gian train, checkpoint tốt nhất. Đặt tên theo `exp_id` (xem mục 6.1).\r\n\r\n---\r\n\r\n## 1. Bước 0: Chuẩn bị, EDA, dựng pipeline\r\n\r\n### 1.1 Tải dữ liệu\r\n\r\nTải `images.zip` từ Zenodo, kiểm tra MD5, giải nén; tải các file CSV trong `labels/` từ GitHub (xem README mục 2). Với 17,5k ảnh nhỏ (256×256), bạn có thể giải nén vào đĩa cục bộ của notebook cho nhanh, tránh đọc từng ảnh từ Drive.\r\n\r\n### 1.2 EDA (bắt buộc, đưa vào báo cáo)\r\n\r\n- Đếm số ảnh theo lớp trong train, val, test của fold 0 và chạy các kiểm tra ở [`README.md` mục 2.1](README.md#21-quy-tắc-chia-train--val--test-bắt-buộc) (giao rỗng, hợp đủ 17.509 ảnh, file tồn tại). Vẽ biểu đồ cột. Tỉ lệ lớp lớn nhất so với lớp nhỏ nhất là bao nhiêu? Đối chiếu với Table 1 của bài báo (`Negative` 9.106; các loài 1.009–1.125 ảnh).\r\n- Xem trực quan ít nhất 3 ảnh mỗi lớp. Lớp nào dễ nhầm với nhau bằng mắt thường? `Negative` trông như thế nào?\r\n- Kiểm tra không trùng ảnh giữa train, val, test (theo tên file).\r\n- Nhìn thống kê ảnh (kích thước, kênh) để chọn chuẩn hoá đầu vào.\r\n\r\nSlide (trang 59) khuyên **\"nhìn dữ liệu\"** trước mọi bước khác.\r\n\r\n### 1.3 Dựng pipeline và kiểm tra tính đúng đắn\r\n\r\nLàm đúng các bước kiểm tra trong checklist gỡ lỗi của slide (trang 59) **trước khi chạy thí nghiệm thật**:\r\n\r\n1. Cố định seed (`random`, `numpy`, `torch`, và DataLoader worker).\r\n2. **Mất mát ban đầu:** với 9 lớp, loss CE ban đầu của một mạng khởi tạo hợp lý phải xấp xỉ `-ln(1/9) ≈ 2,197`. Nếu lệch xa, kiểm tra lại (đặc biệt khi dùng head mới).\r\n3. **Quá khớp một batch nhỏ** (vài mẫu) tới loss gần 0. Nếu không làm được thì lỗi nằm ở code hoặc model, chưa nên chạy tiếp.\r\n4. Kiểm tra đầu vào sau tiền xử lý: vẽ vài ảnh sau augmentation (đã giải chuẩn hoá) để chắc chắn ảnh và nhãn khớp nhau.\r\n5. Kiểm tra chế độ `model.train()` và `model.eval()` được dùng đúng lúc (BatchNorm, dropout).\r\n\r\n### 1.4 Công thức nền (baseline recipe)\r\n\r\nTừ Bước 1 trở đi, mọi backbone đều dùng **cùng một công thức nền** để so sánh công bằng. Đề xuất, dựa trên ví dụ nhóm tham số ở slide (trang 52):\r\n\r\n| Thành phần | Giá trị đề xuất |\r\n|---|---|\r\n| Khởi tạo | Trọng số tiền huấn luyện ImageNet, thay head mới 9 lớp, tinh chỉnh toàn bộ |\r\n| Đầu vào | Train: `RandomResizedCrop(224)` + lật ngang. Val/test: resize 256 rồi `CenterCrop(224)` hoặc dùng thẳng 256 (ghi rõ bạn chọn gì) |\r\n| Chuẩn hoá | Mean/std của ImageNet (hoặc đúng cấu hình của trọng số bạn dùng) |\r\n| Optimizer | AdamW |\r\n| LR | Backbone `1e-4`, head mới `1e-3` (gấp 10 lần) |\r\n| Weight decay | `0,05`, **không áp dụng cho norm và bias** (`weight_decay = 0`) |\r\n| Lịch LR | Warmup khoảng 1 epoch, sau đó cosine về gần 0 |\r\n| Loss | Cross-entropy |\r\n| Batch size | 64 (giảm nếu hết bộ nhớ, và ghi lại) |\r\n| Epoch | 10–15, giống nhau cho mọi backbone |\r\n| Mixed precision | Bật (AMP) |\r\n| Chọn checkpoint | Epoch có macro-F1 val cao nhất |\r\n\r\nĐây là điểm xuất phát, không phải chân lý. Với ViT/Swin, LR có thể cần nhỏ hơn. Nếu bạn đổi công thức nền, hãy đổi **cho tất cả** backbone và ghi lại lý do.\r\n\r\n### 1.5 Dùng bộ khung `starter/` và `eval.py`\r\n\r\nChi tiết từng file ở [`README.md` mục 2.4](README.md#24-công-cụ-evalpy-và-bộ-khung-starter). Quy trình gợi ý:\r\n\r\n1. Chép `starter/` thành `code/` trong thư mục bài nộp. Mở `lab_day2.ipynb` trên Colab hoặc Kaggle, chạy ô cài đặt và tải dữ liệu.\r\n2. Hoàn thiện theo thứ tự: `dataset.py` → `model.py` → `losses.py` → `train.py`. Sau mỗi file, tự viết một kiểm tra nhỏ (ví dụ focal loss với `γ = 0` phải bằng CE; một batch phải overfit được). **Dừng lại ở bước kiểm tra pipeline (mục 1.3) trước khi chạy thí nghiệm thật.**\r\n3. Mọi thí nghiệm đi qua **một** hàm `train.run(Config(...))`; đổi thí nghiệm bằng cách đổi `Config`, không copy code.\r\n4. Khi cần ghi dự đoán, dùng nguyên `eval.save_predictions(...)` (`from eval import save_predictions`). Đừng tự viết lại định dạng.\r\n5. Trước khi nộp, chạy `eval.py score` cho từng nhóm file và `eval.py grade` cho chung kết so với mốc. Nếu `eval.py` báo lỗi định dạng, sửa code ghi file của bạn, không sửa `eval.py`.\r\n\r\n---\r\n\r\n## 2. Bước 1: So sánh backbone (≥ 5)\r\n\r\n**Mục tiêu:** xem với cùng công thức nền, backbone nào cho macro-F1 val tốt nhất, và đánh đổi giữa chất lượng, kích thước và tốc độ.\r\n\r\n### 2.1 Danh sách gợi ý\r\n\r\nChọn **ít nhất 5**, thỏa các ràng buộc sau:\r\n\r\n- ít nhất 1 mạng thuộc họ **ResNet** (làm mốc so sánh);\r\n- ít nhất 1 mạng **ResNeXt hoặc ConvNeXt**;\r\n- ít nhất 1 mạng **transformer** (ViT, DeiT hoặc Swin);\r\n- ít nhất 1 mạng **nhẹ** (MobileNetV3, EfficientNet-B0, RegNet nhỏ...).\r\n\r\n| Họ | Gợi ý (tên trong `timm`) | Ghi chú từ slide (trang 41–42) |\r\n|---|---|---|\r\n| ResNet | `resnet50` | 25,6M tham số, 4,1 GMAC; mốc mặc định |\r\n| ResNeXt | `resnext50_32x4d` | 25,0M, 4,2 GMAC; thêm cardinality |\r\n| ConvNeXt | `convnext_tiny` | 28,6M, 4,5 GMAC; \"ResNet hiện đại hoá\" |\r\n| ViT / DeiT | `vit_small_patch16_224` hoặc `deit_small_patch16_224` | DeiT-S 22,1M, 4,6 GMAC; thiên kiến quy nạp yếu hơn CNN |\r\n| Swin | `swin_tiny_patch4_window7_224` | 28,3M, 4,5 GMAC; attention cửa sổ |\r\n| Nhẹ | `efficientnet_b0`, `mobilenetv3_large_100` | EfficientNet-B0 0,39 GMAC, MobileNetV3-L 0,22 GMAC |\r\n| Bonus | DINOv2 ViT-S/B đóng băng + linear probe | Slide (trang 44) gợi ý khi nhãn ít |\r\n\r\n**Lưu ý về `timm`:** mỗi kiến trúc có nhiều bộ trọng số (tag) khác nhau, tạo ra từ các công thức huấn luyện khác nhau. Tên và tag có thể đổi theo phiên bản, hãy dùng `timm.list_pretrained('resnet50*')` để xem và **ghi rõ tag đã dùng** trong `results.xlsx`. Slide nhắc: so sánh top-1 mà không ghi công thức là lỗi hay gặp (trang 45). Bạn cũng có thể dùng `torchvision.models`.\r\n\r\n### 2.2 Việc phải làm\r\n\r\n- Huấn luyện từng backbone bằng **công thức nền ở 1.4**, 1 seed (cùng seed cho mọi backbone).\r\n- Với mỗi backbone ghi: số tham số, GMAC (đếm bằng thư viện hoặc tự đếm), macro-F1 và top-1 trên val, thời gian train mỗi epoch, độ trễ suy luận sơ bộ (batch 1, một lần đo là đủ ở bước này; đo kỹ ở Bước 3).\r\n- Lưu ảnh biểu đồ training của **mỗi** backbone: `curves/B0x_<ten>.png`.\r\n- Chọn **1–2 backbone** để đi tiếp sang Bước 2 và 3. Lý do chọn phải có số liệu (ví dụ: \"tốt nhất về macro-F1\", hoặc \"cân bằng nhất giữa F1 và độ trễ\"). Slide (trang 33) nhắc câu hỏi thực tế không phải \"mạng nào đứng đầu ImageNet\" mà là \"mạng nào đủ tốt trong độ trễ đo được\".\r\n\r\n### 2.3 Gợi ý phân tích\r\n\r\n- Backbone nào hội tụ nhanh nhất? Có backbone nào quá khớp rõ rệt (train tăng, val giảm)?\r\n- Thứ hạng backbone trên DeepWeeds có giống thứ hạng trên ImageNet (slide trang 41–42) không? Vì sao có/không?\r\n- FLOPs có dự đoán được thời gian train hoặc độ trễ không? (Slide trang 43: *FLOPs không phải độ trễ*.)\r\n\r\n---\r\n\r\n## 3. Bước 2: Công thức huấn luyện\r\n\r\n**Mục tiêu:** đo từng yếu tố của công thức huấn luyện đóng góp bao nhiêu, trên **1–2 backbone** đã chọn ở Bước 1. Yêu cầu: **ít nhất 3 trục** trong bảng dưới, mỗi trục ít nhất 2 giá trị khác nhau (một trong hai là giá trị của công thức nền).\r\n\r\n| Trục | Các giá trị nên thử | Liên hệ slide | Câu hỏi cần trả lời |\r\n|---|---|---|---|\r\n| **A. Khởi tạo** | từ đầu · đóng băng backbone, chỉ train head · tinh chỉnh toàn bộ | Trang 51, 53; Lab #2 | Với ~10k ảnh khác ImageNet, từ đầu có kịp không? Đóng băng có đủ không? |\r\n| **B. Augmentation** | cơ bản (crop + flip) · + đổi màu · TrivialAugment hoặc RandAugment · Mixup · CutMix | Trang 47–50 | Phép nào giúp, phép nào hại? Lật dọc có hợp lệ với ảnh cỏ dại không? |\r\n| **C. Hàm loss** | CE · CE + label smoothing (ε ≈ 0,1) · focal loss (γ ≈ 2) · CE có trọng số theo lớp (hoặc class-balanced) | Trang 56–57 | Với `Negative` áp đảo, loss nào cải thiện F1 các lớp hiếm? |\r\n| **D. Cân bằng mẫu** | không · sampler cân bằng lớp (oversample) | Trang 57 | Sampler khác loss có trọng số như thế nào? |\r\n| **E. LR và optimizer** | LR backbone/head (cùng LR vs gấp 10 lần) · AdamW vs SGD+momentum · có/không warmup · LR theo tầng | Trang 51–52, 55 | Mạng tiền huấn luyện nhạy với LR đến mức nào? |\r\n| **F. Chính quy hoá** | có/không EMA trọng số · weight decay · stochastic depth · dropout head | Trang 56 | EMA có giúp \"miễn phí\" không? |\r\n| **G. Độ phân giải và thời gian huấn luyện** | 224 vs 256 · 10 vs 20 epoch | Trang 46, 68 | Công thức dài hơn có đáng không? |\r\n\r\n### 3.1 Cách làm\r\n\r\n1. Gọi cấu hình nền là `T00`. Mỗi lần chạy mới chỉ khác `T00` **một yếu tố** (nguyên tắc N1).\r\n2. Làm theo thứ tự hợp lý: trục có tác dụng lớn (khởi tạo, loss, augmentation) trước, rồi mới tinh chỉnh LR/EMA.\r\n3. Khi một yếu tố **thắng rõ rệt**, bạn được phép đưa nó vào nền cho các thí nghiệm sau (ghi lại nền mới và đánh dấu trong bảng). Cách này gọi là *tham lam theo từng trục*. Nêu rõ trong báo cáo là bạn đã dùng cách này, vì thứ tự các trục có thể ảnh hưởng kết quả.\r\n4. Cố gắng thử **ít nhất một kết hợp** các yếu tố tốt nhất (ví dụ CutMix + label smoothing + EMA) và kiểm tra xem hiệu ứng có **cộng dồn** hay triệt tiêu nhau. Slide (trang 46) cho thấy các bước công thức cộng lại thành khoảng cách lớn, nhưng từng bước chỉ tăng một phần nhỏ.\r\n5. Mỗi thí nghiệm một ảnh biểu đồ: `curves/T0x_<mota>.png`.\r\n\r\n### 3.2 Lưu ý kỹ thuật\r\n\r\n- **Mixup và CutMix trộn cả nhãn:** loss phải dùng nhãn mềm hoặc tổng có trọng số của hai CE; accuracy lúc train khi dùng Mixup/CutMix không còn nghĩa như bình thường, hãy dựa vào val.\r\n- **Focal loss:** `γ = 0` phải cho lại đúng CE. Hãy tự kiểm tra điều này như một unit test nhỏ.\r\n- **Từ đầu với ViT/Swin:** thiên kiến quy nạp yếu, ít dữ liệu nên thường kém. Đó là kết quả hợp lệ để phân tích, không phải lỗi.\r\n- **Đóng băng:** vẫn phải để BatchNorm ở chế độ eval nếu backbone đóng băng. Hãy nghĩ vì sao.\r\n- **EMA:** đánh giá bằng trọng số EMA, và nếu mạng có BatchNorm thì cân nhắc cập nhật thống kê BN (slide trang 56).\r\n- Giữ **nguyên số epoch** giữa các cấu hình trừ khi đang thử chính yếu tố số epoch.\r\n\r\n---\r\n\r\n## 4. Bước 3: Phương pháp suy luận\r\n\r\n**Mục tiêu:** với mô hình đã huấn luyện xong (không huấn luyện lại), so sánh các cách suy luận về **độ chính xác, hiệu chuẩn và độ trễ**. Yêu cầu **ít nhất 4 phương pháp** khác nhau (không tính 1-view mốc).\r\n\r\n**Mẹo quan trọng:** khi chạy ở Bước 1–2, hãy **lưu logit** (tập val và test) ra file `.npy`. Khi đó phần lớn thí nghiệm suy luận chỉ là tính toán trên các file này, không cần GPU. Riêng TTA, đổi độ phân giải và đo độ trễ cần chạy lại model.\r\n\r\n| Mã | Phương pháp | Ghi chú |\r\n|---|---|---|\r\n| I00 | **1 view** (mốc) | Resize + center crop, `model.eval()` |\r\n| I01 | **TTA lật ngang** (K = 2) | Slide trang 62, 75 |\r\n| I02 | **TTA nhiều crop hoặc nhiều tỉ lệ** (ví dụ 5 crop, hoặc 3 tỉ lệ) | K càng lớn, chi phí gần tuyến tính theo K (slide trang 63) |\r\n| I03 | **Gộp xác suất vs gộp logit** | Slide chưa có kết luận cái nào luôn tốt hơn: chọn một, ghi rõ, hoặc so sánh cả hai |\r\n| I04 | **Dò độ phân giải kiểm tra** (ví dụ 224 / 256 / 288 / 320) | Slide trang 68 (FixRes): test ở độ phân giải cao hơn lúc train có thể tốt hơn; không đổi tham số, chỉ tăng FLOPs |\r\n| I05 | **Ensemble** vài mô hình (khác backbone hoặc khác seed) | Trung bình xác suất; chi phí = số mô hình (slide trang 67) |\r\n| I06 | **Trọng số EMA** (nếu đã train có EMA) hoặc **model soup** | Không tốn thêm khi suy luận (slide trang 67) |\r\n| I07 | **Temperature scaling + ECE** | Khớp một T duy nhất trên **val**, đo ECE trước và sau. Accuracy không đổi (slide trang 69) |\r\n| I08 | **Gộp BatchNorm vào conv** hoặc **FP16/AMP** | Hiệu chỉnh/độ chính xác có thay đổi không? Đo độ trễ (slide trang 71) |\r\n\r\n### 4.1 Đo độ trễ đúng cách\r\n\r\nĐây là phần hay làm sai nhất (slide trang 73, 76). Quy tắc:\r\n\r\n- **Warmup:** chạy 10 lần đầu bỏ đi (lần đầu tải thư viện, cuBLAS...).\r\n- **Đồng bộ GPU:** GPU chạy bất đồng bộ. Trước và sau đoạn cần đo phải gọi `torch.cuda.synchronize()` (hoặc dùng CUDA event). Dùng `time.time()` mà không đồng bộ sẽ cho số sai.\r\n- **Nhiều lần đo:** ≥ 50 lần, báo cáo **p50, p95, p99**, không chỉ trung bình.\r\n- **Hai điều kiện:** batch 1 (giống robot) và một batch lớn hơn (ví dụ 32, đo thông lượng ảnh/giây).\r\n- **Ghi rõ điều kiện:** tên GPU, độ phân giải, dtype (FP32/AMP/FP16), có/không gộp BN, phiên bản torch.\r\n- **Tính cả tiền xử lý hay không?** Chọn một cách và ghi rõ. Với TTA, thời gian gần bằng K lần một lượt chạy.\r\n- Nhớ rằng ở batch 1 trên một số GPU, AMP có thể **chậm hơn** FP32 (slide trang 73). Hãy kiểm tra trên máy của bạn, đừng giả định.\r\n\r\n### 4.2 Tổng hợp\r\n\r\nVới mỗi phương pháp ghi: macro-F1 val, top-1 val, ECE (nếu đo), độ trễ p50/p95/p99 ở batch 1, và **chi phí tương đối** so với I00. Từ đó vẽ **đường đánh đổi** độ chính xác và độ trễ (một biểu đồ scatter là đủ). Slide kết luận TTA và ensemble hợp *ngoại tuyến*, còn trên robot nên dùng thứ không tốn thêm (EMA, soup, gộp BN, FP16, độ phân giải đã dò). Dữ liệu của bạn có ủng hộ kết luận này không?\r\n\r\n---\r\n\r\n## 5. Bước 4: Cấu hình tốt nhất và chạy test\r\n\r\n1. **Chốt cấu hình** dựa hoàn toàn trên **val**: backbone, công thức huấn luyện (kết hợp tốt nhất từ Bước 2), phương pháp suy luận (Bước 3).\r\n2. Huấn luyện lại cấu hình đó với **≥ 3 seed khác nhau** (mã `F01`, `F02`, `F03`...). Báo cáo macro-F1 và top-1 val ở dạng mean ± std.\r\n3. Chạy **test đúng một lần cho mỗi seed** trên toàn bộ tập test, cho các cấu hình cuối cùng **và** mốc `T00`/`I00` (cùng số seed, để tính mức cải thiện). Báo cáo mean ± std của macro-F1, top-1, F1 từng lớp (đặc biệt Chinee apple và Snake weed), ECE. **Lưu file dự đoán** `predictions/<exp_id>_seed<k>_test.csv` theo định dạng ở [`README.md` mục 2.2](README.md#22-cách-đánh-giá-bắt-buộc); giảng viên sẽ tính lại chỉ số từ các file này. Chạy `python eval.py score ...` và `python eval.py grade ...` ([`README.md` mục 2.4](README.md#24-công-cụ-evalpy-và-bộ-khung-starter)) để lấy số liệu đưa vào xlsx và báo cáo. Đối chiếu kết quả với số tham khảo ở [`README.md` mục 2.3](README.md#23-số-tham-khảo-từ-bài-báo-gốc), nhớ rằng điều kiện huấn luyện của bài báo (100 epoch, augmentation mạnh) khác bài lab.\r\n4. So sánh với mốc: **cấu hình tốt nhất cải thiện bao nhiêu so với công thức nền + 1-view?** Chênh lệch có lớn hơn std không?\r\n5. Vẽ **ma trận nhầm lẫn** trên test, nêu lớp nào còn nhầm nhiều nhất và đưa ra giả thuyết nguyên nhân (xem vài ảnh bị dự đoán sai).\r\n6. Nếu giảng viên yêu cầu thêm tiêu chí triển khai (ví dụ độ trễ ≤ X ms), hãy nêu **hai** cấu hình: tốt nhất về độ chính xác (ngoại tuyến) và tốt nhất khi có ràng buộc thời gian thực.\r\n\r\n> Sau khi mở kết quả test, **không** được quay lại sửa cấu hình rồi chạy test lần nữa. Nếu bạn phát hiện lỗi sau đó, hãy ghi lỗi vào báo cáo và nêu rõ con số test nào bị ảnh hưởng.\r\n\r\n---\r\n\r\n## 6. Bước 5: Sản phẩm\r\n\r\n### 6.1 `results.xlsx`\r\n\r\nMột file Excel gồm các sheet sau. Mỗi dòng là một lần chạy hoặc một cấu hình suy luận.\r\n\r\n**Quy ước `exp_id`:** `B01…` backbone · `T01…` công thức huấn luyện · `I01…` suy luận · `F01…` chung kết (nhiều seed). `exp_id` phải trùng tên file ảnh trong `curves/` (ví dụ `B01_resnet50.png`).\r\n\r\n| Sheet | Cột bắt buộc |\r\n|---|---|\r\n| `Backbones` | exp_id · backbone · tag trọng số · #tham số (M) · GMAC · độ phân giải · epoch · seed · macro-F1 val · top-1 val · thời gian train/epoch · độ trễ batch-1 (ms) · ghi chú |\r\n| `Training` | exp_id · backbone · trục thay đổi (A–G) · khác `T00` ở điểm nào · seed · macro-F1 val · top-1 val · Δ so với `T00` · F1 các lớp hiếm (nếu có) · ghi chú |\r\n| `Inference` | exp_id · phương pháp · mô hình/checkpoint dùng · K (số view hoặc số mô hình) · macro-F1 val · top-1 val · ECE val · độ trễ p50/p95/p99 (ms) batch-1 · thông lượng (ảnh/s) · chi phí tương đối so với `I00` |\r\n| `Final` | exp_id · cấu hình (backbone + công thức + suy luận) · seed · macro-F1 val · macro-F1 test · top-1 test · ECE test · mean ± std qua seed (dòng tổng hợp) |\r\n| `PerClass` | lớp · số ảnh test · precision · recall · F1 (cho cấu hình cuối cùng, dòng mốc và dòng tốt nhất) |\r\n| `Latency` | cấu hình · GPU · dtype · batch · gộp BN (có/không) · p50 · p95 · p99 · ảnh/s |\r\n| `Summary` | **Bảng tổng hợp một trang**: top 10 cấu hình theo macro-F1 val, cùng chi phí/độ trễ. Đây là bảng so sánh chính để đưa vào báo cáo |\r\n\r\nGợi ý trình bày: freeze hàng tiêu đề, định dạng số thập phân thống nhất (ví dụ 4 chữ số), tô nổi bật dòng tốt nhất của mỗi sheet, ghi **đơn vị** trong tên cột. Con số mean ± std có thể để trong hai cột riêng hoặc trong một ô dạng `0.9412 ± 0.0031`, miễn nhất quán.\r\n\r\n### 6.2 Ảnh biểu đồ training\r\n\r\nVới **mỗi** thí nghiệm huấn luyện (B, T, F) lưu một ảnh `curves/<exp_id>_<mota>.png` gồm tối thiểu:\r\n\r\n- loss train và loss val theo epoch,\r\n- macro-F1 (hoặc accuracy) val theo epoch (có thể thêm train),\r\n- (khuyến khích) LR theo bước, để thấy warmup và cosine.\r\n\r\nBiểu đồ phải đọc được: có tiêu đề (`exp_id`, backbone), nhãn trục, chú thích. Được dùng matplotlib, TensorBoard hoặc Weights & Biases rồi chụp màn hình, miễn là mỗi thí nghiệm có ảnh riêng và nhìn rõ số liệu.\r\n\r\n### 6.3 Dàn ý báo cáo (`report.md` hoặc `report.pdf`, khoảng 4–8 trang)\r\n\r\n1. **Tóm tắt (≤ 10 dòng):** bài toán, các thí nghiệm đã làm, cấu hình tốt nhất và con số test cuối cùng (kèm std), kết luận chính.\r\n2. **Dữ liệu và thiết lập:** dataset, fold, phân bố lớp (biểu đồ EDA), chỉ số, công thức nền, phần cứng, phiên bản thư viện, seed.\r\n3. **Kết quả so sánh backbone:** bảng (trích từ `Backbones`), biểu đồ F1 theo độ trễ hoặc tham số, nhận xét.\r\n4. **Kết quả công thức huấn luyện:** bảng ablation theo từng trục, kèm Δ và so sánh với std. Nêu rõ yếu tố nào giúp, yếu tố nào không, và giải thích vì sao (liên hệ slide).\r\n5. **Kết quả suy luận:** bảng, đường đánh đổi độ chính xác–độ trễ, kết quả hiệu chuẩn (ECE trước/sau).\r\n6. **Cấu hình tốt nhất:** mô tả đầy đủ để người khác tái lập, bảng chung kết (mean ± std, test), ma trận nhầm lẫn, phân tích lỗi bằng ảnh.\r\n7. **Kết luận và khuyến nghị:** trả lời trực tiếp các câu hỏi:\r\n   - Cấu hình nào tốt nhất? Tốt hơn mốc bao nhiêu, có vượt nhiễu không?\r\n   - Yếu tố nào đóng góp nhiều nhất: backbone, công thức huấn luyện hay suy luận?\r\n   - Nếu triển khai trên robot với ngân sách 30–100 ms/khung, bạn chọn gì?\r\n8. **Hạn chế và việc tiếp theo:** nêu số seed, một fold, thí nghiệm chưa kiểm chứng, rủi ro lệch phân phối (miền khác, mùa khác...).\r\n9. **Phụ lục:** danh sách `exp_id` và cấu hình đầy đủ, link notebook.\r\n\r\nMọi khẳng định phải dựa vào số trong `results.xlsx`. Dùng ngôn ngữ thận trọng: chênh lệch nhỏ hơn nhiễu thì nói là \"không phân biệt được\".\r\n\r\n---\r\n\r\n## 7. Ngân sách tính toán và mẹo Colab/Kaggle\r\n\r\nSố lượng lần chạy tối thiểu xấp xỉ: Bước 1 khoảng 5–6 lần, Bước 2 khoảng 10–15 lần, Bước 4 khoảng 3 lần (nhiều seed), chưa kể thử nghiệm hỏng. Tổng khoảng **20–25 lần huấn luyện**. Thời gian mỗi lần phụ thuộc rất nhiều vào backbone (ViT/Swin chậm hơn MobileNet nhiều), độ phân giải và GPU.\r\n\r\n**Cách tự ước lượng (đừng tin con số của ai, kể cả tài liệu này):**\r\n\r\n1. Chạy 1 epoch của mỗi backbone, ghi thời gian.\r\n2. Nhân với số epoch và số lần chạy dự kiến, rồi so với hạn mức GPU của bạn.\r\n3. Nếu vượt ngân sách, giảm theo thứ tự ưu tiên: (a) epoch 15 → 10; (b) chạy các ablation ở Bước 2 chỉ trên **1 backbone**; (c) chỉ ablation 1 seed, và dành 3 seed cho Bước 4; (d) hạ độ phân giải huấn luyện. **Ghi lại mọi giảm bớt trong báo cáo.**\r\n\r\nMẹo:\r\n\r\n- Dùng **AMP**, `pin_memory=True`, `num_workers` hợp lý, và `channels_last` nếu GPU hỗ trợ.\r\n- Ảnh chỉ 256×256, 17,5k ảnh khoảng vài GB nếu giải nén vào RAM: có thể nạp trước để tránh nghẽn đọc đĩa. Hoặc giải nén vào đĩa cục bộ của notebook, không đọc từng ảnh từ Drive.\r\n- **Lưu sau mỗi epoch** (checkpoint và log) để tiếp tục được khi phiên bị ngắt.\r\n- Viết **một hàm train** nhận cấu hình (dict hoặc YAML) rồi chạy hàng loạt, thay vì copy notebook cho từng thí nghiệm.\r\n- Trên Kaggle, dùng *Save & Run All (Commit)* cho các lần chạy dài. Trên Colab, mở tab giữ kết nối và lưu ra Drive.\r\n- Ghi kết quả thẳng vào Drive hoặc `/kaggle/working`, và chỉ commit file nhỏ (xlsx, png, code) vào git.\r\n\r\n---\r\n\r\n## 8. Bẫy thường gặp\r\n\r\nPhần lớn lấy từ chính slide Day 2:\r\n\r\n| Bẫy | Hệ quả | Cách tránh |\r\n|---|---|---|\r\n| Quên `model.eval()` khi suy luận | BatchNorm dùng thống kê batch, kết quả phụ thuộc ảnh khác trong batch | Luôn gọi `eval()` và `torch.inference_mode()`/`no_grad()` trước mọi đánh giá (slide trang 14, 61) |\r\n| Chọn mô hình hoặc siêu tham số bằng **test** | Con số test lạc quan, không đại diện | Chỉ val để chọn; test một lần (mục 5) |\r\n| So sánh số của hai lần chạy khác công thức | Kết luận sai về kiến trúc | Cùng công thức, cùng split; ghi rõ khác biệt (slide trang 45) |\r\n| Kết luận từ chênh lệch nhỏ hơn nhiễu | \"Phát hiện\" không lặp lại được | ≥ 3 seed cho vòng cuối, so sánh với std (slide trang 59) |\r\n| Chỉ nhìn accuracy trên dữ liệu mất cân bằng | Mô hình luôn đoán `Negative` vẫn điểm cao | Dùng macro-F1 và F1 từng lớp |\r\n| Dùng `time.time()` trên GPU không đồng bộ | Độ trễ sai | `torch.cuda.synchronize()` hoặc CUDA event, có warmup (slide trang 73, 76) |\r\n| Coi FP16/INT8 luôn nhanh hơn và không mất chính xác | Chọn sai cấu hình triển khai | Đo thực tế trên máy của bạn (slide trang 73, 76) |\r\n| Coi TTA là \"độ chính xác miễn phí\" | Bỏ qua chi phí K lần và việc TTA có thể đổi đúng thành sai | Báo cả độ trễ; xem vài ca bị TTA đổi nhãn (slide trang 66) |\r\n| Rò rỉ dữ liệu giữa train, val, test (ví dụ cùng ảnh, hoặc augment trước khi chia) | Điểm cao giả | Chia theo file CSV, không tự chia lại |\r\n| Mixup/CutMix mà vẫn đo accuracy train như bình thường | Hiểu sai đường cong training | Dùng val để đánh giá |\r\n| Đóng băng backbone mà BN vẫn ở train mode | Thống kê BN bị cập nhật, kết quả khó hiểu | Đặt phần đóng băng ở eval mode |\r\n| Đổi nhiều thứ cùng lúc | Không biết yếu tố nào có tác dụng | Nguyên tắc N1 |\r\n| Quên ghi tag trọng số `timm`, version thư viện, seed | Không tái lập được | Ghi vào cấu hình của mỗi lần chạy |\r\n\r\n---\r\n\r\n## 9. Câu hỏi gợi ý cho phần phân tích\r\n\r\nKhông bắt buộc trả lời hết, nhưng các câu này giúp báo cáo sâu hơn:\r\n\r\n1. Giữa ResNet-50 và ConvNeXt-T (kiến trúc hiện đại hoá), mức chênh lệch thật sự đến từ kiến trúc hay công thức huấn luyện của trọng số tiền huấn luyện? (Slide trang 37.)\r\n2. Với ~10k ảnh, vì sao ViT từ đầu thường kém CNN từ đầu? Dữ liệu của bạn có xác nhận không? (Slide trang 32, 53.)\r\n3. Loss nào cải thiện F1 của lớp hiếm nhất? Cái giá phải trả ở lớp `Negative` là gì?\r\n4. Mixup/CutMix có hợp với ảnh cỏ dại không? Khi nhãn là \"loài cỏ\" mà ảnh chứa vật thể nhỏ, cắt dán có làm mất vật thể không?\r\n5. Hiệu ứng nào **cộng dồn** khi kết hợp, hiệu ứng nào triệt tiêu?\r\n6. TTA tăng bao nhiêu điểm và tốn bao nhiêu lần độ trễ? Đáng giá hay không, với ràng buộc nào?\r\n7. Tăng độ phân giải kiểm tra có giúp không? Có liên hệ với việc `RandomResizedCrop` khi train không? (FixRes.)\r\n8. Temperature scaling giảm ECE bao nhiêu? Nếu ảnh lúc triển khai khác miền (mùa khác, ánh sáng khác), T khớp trên val còn đáng tin không?\r\n9. Cấu hình nào bạn **sẽ triển khai trên robot** và vì sao?\r\n10. Bạn sẽ làm gì tiếp theo nếu có thêm một ngày: nhiều fold, nhiều dữ liệu, chưng cất (slide trang 58), hay thích ứng lúc kiểm tra?\r\n",
  "RUBRIC.md": "# RUBRIC — Tiêu chí chấm Lab Day 2\r\n\r\n**Tổng điểm: 100** (cộng tối đa 10 điểm thưởng, tổng không vượt quá 110).\r\nĐiểm thưởng không bù được điều kiện tiên quyết bị vi phạm.\r\n\r\nBài lab chấm **quy trình thực nghiệm** (80 điểm, phần A–H) và **chất lượng model đạt được** (20 điểm, phần I). Một bài có điểm test thấp nhưng thiết kế chặt chẽ, phân tích trung thực vẫn được nhiều điểm ở A–H và G; một bài có điểm test cao nhưng chọn cấu hình bằng test hoặc không có bằng chứng sẽ bị trừ nặng (mục 3) và có thể mất điểm phần I.\r\n\r\n---\r\n\r\n## 0. Điều kiện tiên quyết\r\n\r\nThiếu một trong các mục sau thì bài **chưa được chấm** (hoặc bị giới hạn điểm, ghi trong ngoặc):\r\n\r\n| # | Điều kiện | Hậu quả nếu thiếu |\r\n|---|---|---|\r\n| P1 | Có đủ 6 sản phẩm theo `README.md` mục 4: `results.xlsx`, báo cáo, `curves/`, `code/`, README riêng với link chạy lại, `predictions/` | Trả bài, yêu cầu bổ sung |\r\n| P2 | Số liệu đến từ lần chạy thật; truy ngược được `exp_id` → log/ảnh biểu đồ | Phần có số liệu bịa hoặc không truy ngược được **bị 0 điểm** và báo cáo vi phạm học thuật |\r\n| P3 | Dùng đúng dataset DeepWeeds và **fold 0 chia sẵn** theo quy tắc S1–S6 (`README.md` mục 2.1) | Giới hạn tối đa 60 điểm nếu tự chia lại, gộp val vào train hoặc sửa file CSV mà không được giảng viên đồng ý |\r\n| P4 | `predictions/` đủ cho cấu hình chung kết và mốc, mọi seed, đúng định dạng `README.md` mục 2.2 | **Phần I bị 0 điểm** (giảng viên không tính lại được chỉ số) |\r\n\r\n---\r\n\r\n## 1. Bảng điểm\r\n\r\n### A. Thiết lập và tính chặt chẽ của pipeline (12 điểm)\r\n\r\n| Tiêu chí | Điểm |\r\n|---|---|\r\n| Chia dữ liệu đúng (fold 0), có chạy và báo cáo các kiểm tra ở `README.md` mục 2.1 (số ảnh từng tập/lớp, giao rỗng, hợp đủ 17.509 ảnh); chọn mô hình/siêu tham số **chỉ trên val**; test chạy **một lần mỗi seed** ở cuối, có nêu rõ trong báo cáo | 4 |\r\n| EDA: phân bố lớp (biểu đồ), ảnh mẫu, đối chiếu với Table 1 của bài báo, nhận xét về mất cân bằng | 2 |\r\n| Kiểm tra pipeline trước khi chạy thật: loss ban đầu ≈ ln 9, overfit 1 batch nhỏ, kiểm tra ảnh sau augmentation (có bằng chứng trong notebook hoặc báo cáo) | 3 |\r\n| Cố định seed, ghi cấu hình đầy đủ (tag trọng số, hyperparameter, version thư viện); dùng `eval()` đúng lúc | 3 |\r\n\r\n### B. So sánh backbone (12 điểm)\r\n\r\n| Tiêu chí | Điểm |\r\n|---|---|\r\n| **≥ 5 backbone**, đủ ràng buộc: có ResNet, có ResNeXt hoặc ConvNeXt, có ít nhất 1 transformer, có ít nhất 1 mạng nhẹ (thiếu mỗi ràng buộc trừ 1 điểm; dưới 5 backbone: tối đa 5/12 cho mục này) | 4 |\r\n| **Công bằng:** cùng công thức nền, cùng split, cùng seed; ghi rõ tag trọng số | 3 |\r\n| Báo cáo đủ: params, GMAC, macro-F1/top-1 val, thời gian train, độ trễ sơ bộ | 3 |\r\n| Chọn backbone đi tiếp **có lý do dựa trên số liệu** (không chỉ \"vì F1 cao nhất\" nếu có đánh đổi khác) | 2 |\r\n\r\n### C. Công thức huấn luyện (16 điểm)\r\n\r\n| Tiêu chí | Điểm |\r\n|---|---|\r\n| **≥ 3 trục** (khởi tạo, augmentation, loss, sampler, LR/optimizer, chính quy hoá, độ phân giải...), mỗi trục ≥ 2 giá trị; **ít nhất một trục là loss hoặc augmentation** | 5 |\r\n| **Thiết kế có kiểm soát:** mỗi lần chạy chỉ khác nền một yếu tố, bảng ghi rõ khác ở điểm nào; nếu dùng cách tham lam theo trục thì nêu rõ | 4 |\r\n| Cài đặt đúng các kỹ thuật (ví dụ focal `γ=0` ≡ CE; Mixup/CutMix trộn cả nhãn; weight decay không áp dụng cho norm/bias; EMA đánh giá đúng trọng số) | 3 |\r\n| Thử **ít nhất một kết hợp** các yếu tố tốt và nhận xét cộng dồn hay triệt tiêu | 1 |\r\n| Phân tích so sánh Δ với nhiễu (std); không kết luận từ chênh lệch nhỏ hơn nhiễu | 3 |\r\n\r\n### D. Suy luận (12 điểm)\r\n\r\n| Tiêu chí | Điểm |\r\n|---|---|\r\n| **≥ 4 phương pháp suy luận** ngoài mốc 1-view (ví dụ: TTA lật, multi-crop/scale, dò độ phân giải, gộp xác suất vs logit, ensemble, EMA/soup, temperature scaling, gộp BN/FP16) | 4 |\r\n| Đo độ trễ **đúng cách**: warmup, `cuda.synchronize` (hoặc CUDA event), ≥ 50 lần, báo cáo p50/p95/p99, ghi rõ GPU/dtype/batch | 3 |\r\n| Có **hiệu chuẩn:** ECE trước và sau temperature scaling, T khớp trên val | 2 |\r\n| Biểu đồ/bảng **đánh đổi độ chính xác và độ trễ**, nhận xét phương pháp nào hợp ngoại tuyến, phương pháp nào hợp thời gian thực | 3 |\r\n\r\n### E. Bảng so sánh `results.xlsx` (8 điểm)\r\n\r\n| Tiêu chí | Điểm |\r\n|---|---|\r\n| Đủ các sheet và cột bắt buộc (`GUIDE.md` mục 6.1); mọi thí nghiệm có mặt | 3 |\r\n| Nhất quán: `exp_id` khớp ảnh biểu đồ, file dự đoán và báo cáo; đơn vị rõ; mean ± std đúng cho cấu hình chung kết | 3 |\r\n| Sheet `Summary` rõ ràng, dễ đọc, làm nổi bật cấu hình tốt nhất và so sánh với mốc | 2 |\r\n\r\n### F. Biểu đồ training (4 điểm)\r\n\r\n| Tiêu chí | Điểm |\r\n|---|---|\r\n| Mỗi thí nghiệm huấn luyện (B, T, F) có một ảnh riêng, đủ loss train/val và metric val theo epoch | 2 |\r\n| Đọc được: tiêu đề, nhãn trục, chú thích, tên khớp `exp_id`; có nhận xét về hiện tượng đáng chú ý (quá khớp, nhiễu, hội tụ chậm...) trong báo cáo | 2 |\r\n\r\n### G. Báo cáo kết luận (12 điểm)\r\n\r\n| Tiêu chí | Điểm |\r\n|---|---|\r\n| Đủ cấu trúc theo `GUIDE.md` mục 6.3; tóm tắt nêu rõ cấu hình tốt nhất và con số test cuối cùng kèm std | 3 |\r\n| **Kết luận có bằng chứng:** trả lời được yếu tố nào đóng góp nhiều nhất (backbone, huấn luyện hay suy luận) và cấu hình nào tốt nhất, mỗi khẳng định gắn với số trong xlsx | 4 |\r\n| **Phân tích lỗi:** ma trận nhầm lẫn, F1 từng lớp, xem ảnh bị đoán sai (đặc biệt cặp Chinee apple ↔ Snake weed) và đưa ra giả thuyết | 3 |\r\n| **Trung thực và hạn chế:** nêu số seed, một fold, chia ngẫu nhiên (không theo địa điểm) có thể làm điểm test lạc quan, giảm bớt do ngân sách GPU, thí nghiệm thất bại, rủi ro lệch phân phối; khuyến nghị triển khai thực tế | 2 |\r\n\r\n### H. Code và khả năng tái lập (4 điểm)\r\n\r\n| Tiêu chí | Điểm |\r\n|---|---|\r\n| Hoàn thiện đúng và đầy đủ bộ khung `starter/` (không còn `NotImplementedError`; có kiểm tra tự viết cho các phần dễ sai như focal `γ=0`, CutMix, gộp BN); một hàm train dùng chung cho mọi cấu hình; code đọc được. Không sửa `eval.py` | 2 |\r\n| README riêng có link notebook Colab/Kaggle, version thư viện, thứ tự chạy; người khác chạy lại được và ra kết quả cùng mức | 2 |\r\n\r\n### I. Chất lượng model đạt được (20 điểm)\r\n\r\n**Cách chấm:** giảng viên **tính lại chỉ số từ `predictions/`** (định nghĩa ở `README.md` mục 2.2), trên **toàn bộ tập test fold 0**, cho **cấu hình chung kết**, **trung bình qua các seed** (≥ 3 seed). Số trong báo cáo hoặc xlsx lệch khỏi số tính lại thì lấy số tính lại, và áp dụng mục 3 nếu lệch không giải thích được.\r\n\r\nCác ngưỡng neo vào số công bố trong bài báo gốc (`README.md` mục 2.3). Bài báo huấn luyện khoảng 100 epoch với augmentation mạnh, còn bài lab chỉ 10–15 epoch, nên các bậc dưới mốc 95,1% vẫn có điểm. **Các ngưỡng này là ngưỡng tạm thời; giảng viên có thể điều chỉnh và sẽ thông báo trước hạn nộp.**\r\n\r\n| # | Tiêu chí | Cách tính | Điểm |\r\n|---|---|---|---|\r\n| I1 | **Top-1 accuracy trên test** (mean qua seed) | ≥ 95,7%: **7** · ≥ 95,1%: **6** · ≥ 94,0%: **5** · ≥ 92,0%: **3** · ≥ 90,0%: **1** · < 90,0%: **0**. (Mốc 95,7% và 95,1% là ResNet-50 và Inception-v3 của bài báo) | 7 |\r\n| I2 | **Macro-F1 test cải thiện so với mốc của chính bạn** (`T00` + `I00`, cùng số seed). Gọi Δ = macro-F1 trung bình của chung kết trừ của mốc, s = std lớn hơn trong hai nhóm seed | Δ ≤ 0: **0** · 0 < Δ ≤ s: **2** · Δ > s: **4** · Δ > s và Δ ≥ 0,01: **5** | 5 |\r\n| I3 | **Hai lớp khó: Chinee apple và Snake weed** (recall trên test, mean qua seed). Mốc bài báo: 88,5% và 88,8% | Cả hai ≥ mốc: **4** · cả hai ≥ 85%: **3** · cả hai ≥ 80%: **2** · có báo cáo đầy đủ nhưng dưới 80%: **1** | 4 |\r\n| I4 | **Ổn định và hiệu chuẩn:** (a) ECE test sau temperature scaling (T khớp trên val) nhỏ hơn ECE trước; (b) chênh lệch macro-F1 giữa val và test của chung kết không quá 0,02, hoặc có giải thích | Mỗi ý đạt được **1** | 2 |\r\n| I5 | **Cấu hình dùng được cho thời gian thực:** nêu một cấu hình có độ trễ **p95 ≤ 100 ms ở batch 1** trên phần cứng bạn khai báo (ngân sách một chu kỳ cảm biến, slide trang 61), kèm macro-F1 test của cấu hình đó | Có và đo đúng cách: **2** · có nhưng đo thiếu (không warmup/synchronize): **1** | 2 |\r\n\r\nGhi chú:\r\n\r\n- Bài thất bại ở I1–I3 vẫn được điểm cho phân tích ở phần G nếu trung thực và có bằng chứng.\r\n- Không chấm I1–I3 theo kết quả của một seed đơn lẻ; dùng mean qua seed. Báo cáo kèm std.\r\n- Số theo lớp của bài báo được coi là tương đương recall theo lớp. Bài báo ghi \"weighted average accuracy\"; I1 dùng top-1 accuracy không trọng số, nên so sánh chỉ mang tính tham chiếu.\r\n- Điểm I chỉ tính trên test. Một lần chạy test lặp lại để chọn kết quả tốt hơn bị xử lý theo mục 3, không chỉ mất điểm I.\r\n\r\n---\r\n\r\n## 2. Điểm thưởng (tối đa +10)\r\n\r\n| Việc làm thêm | Điểm |\r\n|---|---|\r\n| Linear probe với DINOv2 (hoặc mô hình nền tảng khác) đóng băng, so sánh với CNN tinh chỉnh | +2 |\r\n| Chạy nhiều fold (ví dụ 3–5 fold, mỗi fold dùng đủ bộ ba file của fold đó) cho cấu hình cuối, báo cáo mean ± std qua fold | +3 |\r\n| Chưng cất tri thức (giáo viên lớn → học sinh nhỏ) và so sánh với huấn luyện thường | +2 |\r\n| Grad-CAM hoặc bản đồ attention để giải thích lỗi | +1 |\r\n| Test-time adaptation đơn giản (chuẩn hoá lại thống kê BN, hoặc Tent) trên tập lệch miền tự tạo (ví dụ ảnh làm tối/nhiễu), **không** dùng test fold 0 để thích ứng | +2 |\r\n| Xuất ONNX và so sánh độ trễ với PyTorch | +1 |\r\n| Phân tích riêng về lệch phân phối: đánh giá trên ảnh bị nhiễu/làm mờ/thiếu sáng, ECE trước/sau | +2 |\r\n\r\nĐiểm thưởng chỉ tính khi phần làm thêm có số liệu thật và nằm trong báo cáo.\r\n\r\n---\r\n\r\n## 3. Lỗi bị trừ điểm\r\n\r\n| Lỗi | Trừ |\r\n|---|---|\r\n| Chọn cấu hình, siêu tham số hoặc phương pháp suy luận **dựa vào điểm test**, hoặc chạy test nhiều lần rồi chọn lần tốt hơn | −10 đến −20 (nghiêm trọng, tuỳ mức) |\r\n| Gộp val vào train, huấn luyện trên test, hoặc dùng thông tin từ test (thống kê, ngưỡng, nhiệt độ T) | −10 đến −20 (nghiêm trọng, tuỳ mức), đồng thời áp dụng P3 |\r\n| Thiếu `model.eval()` khi đánh giá, hoặc đo độ trễ không `synchronize` / không warmup (số liệu sai) | −3 mỗi loại, tối đa −6 |\r\n| Kết luận \"A tốt hơn B\" khi chênh lệch nhỏ hơn std hoặc chỉ có 1 seed mà không nêu hạn chế | −1 mỗi chỗ, tối đa −5 |\r\n| Chỉ báo accuracy, không có macro-F1 hoặc F1 từng lớp | −3 |\r\n| Thí nghiệm không có ảnh biểu đồ training | −0,5 mỗi thí nghiệm, tối đa −5 |\r\n| Số liệu trong báo cáo/xlsx mâu thuẫn với số tính lại từ `predictions/` hoặc với nhau | −1 mỗi chỗ, tối đa −5 |\r\n| Commit dataset hoặc checkpoint lớn vào git | −2 |\r\n| Nộp trễ | Theo quy định của giảng viên |\r\n| **Bịa số liệu, chép bài, hoặc chép số từ bài báo/slide như kết quả của mình** | **0 điểm phần liên quan; báo cáo vi phạm học thuật** |\r\n\r\n---\r\n\r\n## 4. Mức điểm tham khảo\r\n\r\n| Điểm | Mô tả |\r\n|---|---|\r\n| **90–100+** | Đủ ≥ 5 backbone, ≥ 3 trục huấn luyện có kiểm soát, ≥ 4 phương pháp suy luận với độ trễ đo đúng; cấu hình cuối ≥ 3 seed; test một lần mỗi seed; kết quả test đạt các bậc cao ở phần I; báo cáo kết luận có bằng chứng và phân tích lỗi; code tái lập được |\r\n| **75–89** | Đạt các ngưỡng tối thiểu, nhưng thiếu một số điểm chặt chẽ (ví dụ ít seed ở vòng cuối, phân tích nông, bảng thiếu cột) hoặc kết quả test ở bậc trung bình |\r\n| **60–74** | Đủ sản phẩm nhưng một phần thí nghiệm thiếu kiểm soát, kết luận không đủ bằng chứng, hoặc kết quả test thấp |\r\n| **< 60** | Thiếu ngưỡng tối thiểu (số backbone, số trục, số phương pháp), hoặc vi phạm quy tắc val/test, hoặc không tái lập được |\r\n\r\n---\r\n\r\n## 5. Danh sách tự kiểm trước khi nộp\r\n\r\n- [ ] Dùng đúng fold 0, không sửa CSV, không gộp val vào train; đã chạy kiểm tra giao rỗng và hợp đủ 17.509 ảnh.\r\n- [ ] ≥ 5 backbone, cùng công thức nền, ghi tag trọng số.\r\n- [ ] ≥ 3 trục công thức huấn luyện, mỗi lần chạy khác nền một yếu tố.\r\n- [ ] ≥ 4 phương pháp suy luận, độ trễ p50/p95/p99 đo đúng cách.\r\n- [ ] Cấu hình cuối và mốc (`T00` + `I00`) chạy ≥ 3 seed, báo cáo mean ± std; test chạy **một lần mỗi seed**.\r\n- [ ] `predictions/<exp_id>_seed<k>_test.csv` đủ cho chung kết và mốc, mọi seed, đúng cột.\r\n- [ ] `python eval.py score` chạy không lỗi trên mọi nhóm file dự đoán; macro-F1, top-1, recall Chinee apple và Snake weed trong báo cáo khớp kết quả của `eval.py`.\r\n- [ ] Đã chạy `python eval.py grade` và xem kết quả phần I (đề xuất).\r\n- [ ] Không còn `NotImplementedError` trong `code/`; không sửa `eval.py`.\r\n- [ ] Đã nêu một cấu hình có p95 ≤ 100 ms ở batch 1 (hoặc giải thích vì sao không có).\r\n- [ ] `results.xlsx` đủ sheet, `exp_id` khớp ảnh trong `curves/`.\r\n- [ ] Mỗi thí nghiệm huấn luyện có ảnh biểu đồ riêng.\r\n- [ ] Báo cáo có tóm tắt, bảng so sánh, ma trận nhầm lẫn, kết luận, hạn chế (nêu chia ngẫu nhiên có thể lạc quan).\r\n- [ ] Code đầy đủ, README riêng có link notebook chạy lại được.\r\n- [ ] Không commit dataset hoặc checkpoint lớn.\r\n- [ ] Mọi số liệu đến từ lần chạy thật của bạn.\r\n",
  "code/runtime.py": "\"\"\"Small compatibility setup shared by notebook and command-line entry points.\"\"\"\nimport os\n\n\ndef configure_runtime():\n    # pandas 3 + PyArrow can raise a native access violation after Torch loads\n    # on this Windows environment. Use Python string storage, leaving eval.py intact.\n    if os.name == \"nt\":\n        import pandas as pd\n        pd.options.mode.string_storage = \"python\"\n\n\nconfigure_runtime()\n",
  "code/dataset.py": "\"\"\"dataset.py - đọc DeepWeeds, kiểm tra chia dữ liệu, transform, DataLoader.\n\nHoàn thiện từ bộ khung `starter/`. Giữ nguyên tên hàm và kiểu dữ liệu vào/ra như docstring gốc.\n\nQuy tắc chia dữ liệu bắt buộc (S1-S6) nằm ở README.md, mục 2.1. Không sửa, không lọc, không chia lại.\n\n    load_split(labels_dir, fold=0)                        -> (train_df, val_df, test_df)\n    check_split(train_df, val_df, test_df, images_dir)   -> dict  (số liệu để ghi báo cáo)\n    build_transforms(train, img_size, aug)                -> torchvision transform\n    DeepWeedsDataset[i]                                   -> (image_tensor, label:int, filename:str)\n    make_loader(df, images_dir, transform, batch_size, train, sampler, num_workers)\n\nBa lựa chọn thiết kế ghi rõ ở đây vì ảnh hưởng tới kết luận của bài:\n  1. Tiền xử lý val/test: `Resize(img_size)` + `CenterCrop(img_size)`. Ảnh DeepWeeds đã là\n     hình vuông 256x256, nên với ảnh vuông CenterCrop là thao tác thừa; đường này tương đương\n     resize toàn bộ ảnh về img_size. Không dùng augmentation ngẫu nhiên khi đánh giá (S2).\n  2. `RandomResizedCrop` dùng `scale=(0.7, 1.0)` thay vì mặc định (0.08, 1.0): đây là bài\n     phân loại 9 lớp ở mức loài, nhiều lớp chỉ ~1.100 ảnh; crop quá aggressive làm mất\n     chi tiết đặc trưng của lá/cụm hoa và thường hại hơn giúp. Giá trị này là biến của\n     trục augmentation (trục B, GUIDE.md mục 3).\n  3. `cache_in_ram`: Kaggle T4 chỉ có 2 vCPU, giải mã JPEG thường là nút thắt. Toàn bộ 17.509\n     ảnh uint8 256x256x3 chiếm ~3,2 GiB, nạp vào RAM một lần để DataLoader không phải\n     giải mã lại mỗi epoch. Trên Linux, tiến trình con được `fork` nên mảng được chia sẻ\n     bản sao-on-write, không nhân đôi bộ nhớ.\n\"\"\"\nfrom __future__ import annotations\n\nimport runtime  # Windows pandas compatibility, before constructing DataFrames.\n\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import DataLoader, Dataset, WeightedRandomSampler\n\nNUM_CLASSES = 9\n# Thứ tự lớp theo cột `Label` của labels.csv (0 = Chinee Apple ... 7 = Snake Weed, 8 = Negatives).\nCLASS_NAMES = [\n    \"Chinee Apple\", \"Lantana\", \"Parkinsonia\", \"Parthenium\", \"Prickly Acacia\",\n    \"Rubber Vine\", \"Siam Weed\", \"Snake Weed\", \"Negatives\",\n]\nIMAGENET_MEAN = (0.485, 0.456, 0.406)  # đổi nếu trọng số timm bạn dùng yêu cầu mean/std khác\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\n# --- hằng số kiểm tra chia dữ liệu (README.md mục 2.1) ---\nEXPECTED_TOTAL = 17509           # hợp ba tập phải bằng đúng số này\nNOMINAL_SHARE = {\"train\": 0.60, \"val\": 0.20, \"test\": 0.20}\nSHARE_TOL = 0.01                 # lệch quá 1 điểm phần trăm thì cảnh báo (không raise)\nIMG_EXTS = (\".jpg\", \".jpeg\", \".png\", \".bmp\")\n\n# --- các mức augmentation của trục B ---\n# trivial     : không augmentation ngẫu nhiên (chỉ resize) -> \"không có augmentation\"\n# basic       : RandomResizedCrop + lật ngang\n# color       : basic + ColorJitter\n# randaug     : basic + RandAugment\n# basic_vflip : basic + lật dọc -> dùng để trả lời câu hỏi \"lật dọc có hợp lệ với ảnh cỏ dại không?\"\nAUG_LEVELS = (\"trivial\", \"basic\", \"color\", \"randaug\", \"basic_vflip\")\n\n\n# --------------------------------------------------------------------------- #\n# 1. Đọc split\n# --------------------------------------------------------------------------- #\ndef load_split(labels_dir: str | Path, fold: int = 0):\n    \"\"\"Đọc train_subset{fold}.csv, val_subset{fold}.csv, test_subset{fold}.csv (S1).\n\n    Mỗi file có cột `Filename, Label, Species`. Trả về ba DataFrame.\n    KHÔNG sửa, lọc hay chia lại dữ liệu: hàm này chỉ đọc và ép kiểu `Label` về int.\n\n    TODO đã làm:\n      - đọc ba file CSV bằng pandas\n      - trả về (train_df, val_df, test_df)\n    \"\"\"\n    labels_dir = Path(labels_dir)\n    frames = []\n    for split in (\"train\", \"val\", \"test\"):\n        path = labels_dir / f\"{split}_subset{fold}.csv\"\n        if not path.exists():\n            raise FileNotFoundError(\n                f\"Không tìm thấy {path}. Phải dùng fold chia sẵn của tác giả (S1); \"\n                f\"tải từ github.com/AlexOlsen/DeepWeeds, thư mục labels/.\"\n            )\n        df = pd.read_csv(path)\n        # File chia sẵn của tác giả có 2 cột (Filename, Label); labels.csv có thêm Species.\n        # Vì vậy chỉ bắt buộc Filename và Label.\n        missing = {\"Filename\", \"Label\"} - set(df.columns)\n        if missing:\n            raise ValueError(f\"{path}: thiếu cột {sorted(missing)}\")\n        df = df.copy()\n        df[\"Label\"] = df[\"Label\"].astype(np.int64)\n        if df[\"Label\"].min() < 0 or df[\"Label\"].max() >= NUM_CLASSES:\n            raise ValueError(f\"{path}: Label phải nằm trong 0..{NUM_CLASSES - 1}\")\n        if df[\"Filename\"].duplicated().any():\n            raise ValueError(f\"{path}: có tên file trùng nhau\")\n        frames.append(df)\n    return tuple(frames)\n\n\n# --------------------------------------------------------------------------- #\n# 2. Kiểm tra chia dữ liệu\n# --------------------------------------------------------------------------- #\ndef resolve_images_dir(images_dir: str | Path) -> Path:\n    \"\"\"Tìm thư mục thực sự chứa file ảnh.\n\n    `images.zip` giải nén ra `data/images/*.jpg` (phẳng). Nếu bạn giải nén lệch một cấp,\n    thư mục con duy nhất chứa ảnh cũng được chấp nhận. Trả về đường dẫn dạng Path.\n    \"\"\"\n    d = Path(images_dir)\n    if not d.is_dir():\n        raise FileNotFoundError(f\"{d} không phải thư mục\")\n    if any(d.glob(\"*\")) and any(d.glob(f\"*{ext}\") for ext in IMG_EXTS):\n        return d\n    subdirs = sorted(p for p in d.iterdir() if p.is_dir())\n    for sub in subdirs:\n        if any(sub.glob(f\"*{ext}\") for ext in IMG_EXTS):\n            print(f\"[dataset] dùng thư mục ảnh con: {sub}\")\n            return sub\n    raise FileNotFoundError(f\"Không tìm thấy file ảnh (*.jpg) nào trong {d} hay thư mục con của nó\")\n\n\ndef check_split(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame,\n                images_dir: str | Path, strict_ratio: bool = False) -> dict:\n    \"\"\"Kiểm tra bắt buộc trước khi train (README.md, mục 2.1). In ra và trả về dict số liệu.\n\n    Kiểm tra (mỗi ý lỗi thì raise để dừng ngay):\n      1. số ảnh mỗi tập và số ảnh mừng lớp trong từng tập (kỳ vọng xấp xỉ 60/20/20)\n      2. giao của từng cặp tập theo Filename phải RỖNG (train∩val, train∩test, val∩test)\n      3. hợp ba tập phải bằng đúng 17.509 ảnh\n      4. mọi Filename đều tồn tại trong `images_dir`\n\n    Ý (1) chỉ *cảnh báo* nếu lệch quá 1 điểm phần trăm so với 60/20/20, vì lớp `Negative`\n    không được phân tầng khi chia nên tỉ lệ có thể lệch tự nhiên. Đặt `strict_ratio=True`\n    để biến cảnh báo thành lỗi.\n    \"\"\"\n    splits = {\"train\": train_df, \"val\": val_df, \"test\": test_df}\n    img_dir = resolve_images_dir(images_dir)\n\n    n = {k: int(len(v)) for k, v in splits.items()}\n    per_class = {k: v[\"Label\"].value_counts().reindex(range(NUM_CLASSES), fill_value=0)\n                    .sort_index().astype(int).tolist()\n                 for k, v in splits.items()}\n    total = sum(n.values())\n\n    sets = {k: set(v[\"Filename\"]) for k, v in splits.items()}\n    overlap = {\n        \"train_val\": sorted(sets[\"train\"] & sets[\"val\"]),\n        \"train_test\": sorted(sets[\"train\"] & sets[\"test\"]),\n        \"val_test\": sorted(sets[\"val\"] & sets[\"test\"]),\n    }\n\n    missing = {k: sorted(s - {p.name for p in img_dir.glob(\"*\") if p.is_file()})\n               for k, s in sets.items()}\n\n    report = {\n        \"images_dir\": str(img_dir),\n        \"n\": n,\n        \"per_class\": per_class,\n        \"class_names\": CLASS_NAMES,\n        \"overlap\": {k: len(v) for k, v in overlap.items()},\n        \"union\": len(sets[\"train\"] | sets[\"val\"] | sets[\"test\"]),\n        \"expected_total\": EXPECTED_TOTAL,\n        \"missing_files\": {k: len(v) for k, v in missing.items()},\n    }\n\n    # --- in báo cáo ---\n    print(f\"[check_split] thư mục ảnh: {img_dir}\")\n    print(f\"{'tập':<6}{'số ảnh':>8}{'tỉ lệ':>9}   (kỳ vọng {', '.join(f'{k} {v:.0%}' for k, v in NOMINAL_SHARE.items())})\")\n    for k, v in n.items():\n        share = v / total if total else 0.0\n        dev = share - NOMINAL_SHARE[k]\n        flag = \"\"\n        if abs(dev) > SHARE_TOL:\n            flag = f\"  <-- LỆCH {dev:+.1%} so với {NOMINAL_SHARE[k]:.0%}\"\n            if strict_ratio:\n                raise ValueError(f\"[check_split] tỉ lệ tập {k} lệch {dev:+.1%} (strict_ratio=True)\")\n        print(f\"{k:<6}{v:>8}{share:>8.1%}{flag}\")\n    print(f\"{'TỔNG':<6}{total:>8}\")\n    print(\"\\n[check_split] số ảnh từng lớp (đối chiếu Table 1 của bài báo):\")\n    hdr = \"\".join(f\"{name[:11]:>13}\" for name in CLASS_NAMES)\n    print(f\"{'lớp':<6}\" + hdr)\n    for k in (\"train\", \"val\", \"test\"):\n        print(f\"{k:<6}\" + \"\".join(f\"{c:>13}\" for c in per_class[k]))\n    allc = [sum(per_class[k][i] for k in splits) for i in range(NUM_CLASSES)]\n    print(f\"{'tất cả':<6}\" + \"\".join(f\"{c:>13}\" for c in allc))\n\n    # --- các kiểm tra cứng ---\n    for pair, files in overlap.items():\n        if files:\n            raise ValueError(f\"[check_split] GIAO KHÔNG RỖNG {pair}: {len(files)} file, ví dụ {files[:5]}\")\n    if total != EXPECTED_TOTAL:\n        raise ValueError(f\"[check_split] hợp ba tập có {total} ảnh, phải bằng {EXPECTED_TOTAL}\")\n    if len(sets[\"train\"] | sets[\"val\"] | sets[\"test\"]) != EXPECTED_TOTAL:\n        raise ValueError(\"[check_split] hợp ba tập theo tên file khác 17.509 (trùng tên giữa các tập)\")\n    for k, files in missing.items():\n        if files:\n            raise ValueError(f\"[check_split] {len(files)} file của tập {k} không tồn tại trong {img_dir}, \"\n                             f\"ví dụ {files[:5]}\")\n    print(f\"\\n[check_split] OK: giao rỗng, hợp đúng {EXPECTED_TOTAL} ảnh, mọi file đều tồn tại.\")\n    return report\n\n\n# --------------------------------------------------------------------------- #\n# 3. Transform\n# --------------------------------------------------------------------------- #\ndef build_transforms(train: bool, img_size: int = 224, aug: str = \"basic\"):\n    \"\"\"Tạo transform. `aug` chọn mức augmentation (xem `AUG_LEVELS`).\n\n    Train (basic): RandomResizedCrop(img_size) + lật ngang + ToTensor + Normalize.\n    Val/test: Resize(img_size) + CenterCrop(img_size) + ToTensor + Normalize.\n               KHÔNG augmentation ngẫu nhiên khi đánh giá (S2).\n\n    Lựa chọn: dùng `bilinear` cho mọi phép resize để khớp với hầu hết tag pretrained của\n    timm; nếu bạn đổi sang tag nào yêu cầu bicubic thì đổi cả hai nhánh cho khớp.\n    Lật dọc (basic_vflip) được tách riêng để kiểm chứng, không đưa vào `basic`.\n    \"\"\"\n    from torchvision import transforms as T\n\n    norm = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)\n    eval_tf = T.Compose([\n        T.Resize(img_size, interpolation=T.InterpolationMode.BILINEAR),\n        T.CenterCrop(img_size),\n        T.ToTensor(),\n        norm,\n    ])\n    if not train:\n        return eval_tf\n\n    if aug not in AUG_LEVELS:\n        raise ValueError(f\"aug={aug!r} không hợp lệ, chọn một trong {AUG_LEVELS}\")\n\n    ops = [\n        T.RandomResizedCrop(img_size, scale=(0.7, 1.0), ratio=(3 / 4, 4 / 3),\n                            interpolation=T.InterpolationMode.BILINEAR),\n        T.RandomHorizontalFlip(),\n    ]\n    if aug == \"color\":\n        ops += [T.ColorJitter(0.4, 0.4, 0.4, 0.1)]\n    elif aug == \"randaug\":\n        ops += [T.RandAugment(num_ops=2, magnitude=9)]\n    elif aug == \"basic_vflip\":\n        ops += [T.RandomVerticalFlip()]\n    # aug == \"trivial\": không thêm phép ngẫu nhiên nào -> thay RandomResizedCrop bằng Resize\n    if aug == \"trivial\":\n        ops[0] = T.Resize(img_size, interpolation=T.InterpolationMode.BILINEAR)\n\n    return T.Compose(ops + [T.ToTensor(), norm])\n\n\n# --------------------------------------------------------------------------- #\n# 4. Dataset\n# --------------------------------------------------------------------------- #\nclass DeepWeedsDataset(Dataset):\n    \"\"\"Dataset đọc ảnh từ `images_dir` theo DataFrame (Filename, Label).\n\n    `__getitem__(i)` trả về (ảnh đã transform, nhãn int, tên file str).\n    Tên file cần có để ghi `predictions/*.csv` đúng định dạng của eval.py.\n\n    `cache_in_ram=True` nạp trước toàn bộ ảnh dưới dạng mảng uint8 [N, H, W, 3].\n    Toàn bộ ảnh DeepWeeds (17.509, 256x256) chiếm ~3,2 GiB. `max_cache_gb` để tự phòng vệ:\n    nếu ảnh lớn hơn ngân sách thì tắt cache và in cảnh báo thay vì làm treo máy.\n    \"\"\"\n\n    def __init__(self, df: pd.DataFrame, images_dir: str | Path, transform=None,\n                 cache_in_ram: bool = False, max_cache_gb: float = 6.0,\n                 img_size: int = 256):\n        self.df = df.reset_index(drop=True)\n        self.images_dir = resolve_images_dir(images_dir)\n        self.transform = transform\n        self.files = self.df[\"Filename\"].astype(str).tolist()\n        self.labels = self.df[\"Label\"].astype(np.int64).to_numpy()\n        self._cache: np.ndarray | None = None\n\n        if cache_in_ram:\n            self._build_cache(img_size, max_cache_gb)\n\n    def _build_cache(self, img_size: int, max_cache_gb: float) -> None:\n        \"\"\"Nạp ảnh vào RAM dưới dạng uint8. Dùng song song 2 luồng cho nhanh.\"\"\"\n        from concurrent.futures import ThreadPoolExecutor\n\n        n = len(self.files)\n        if not n:\n            return\n        # Cache original pixels; img_size is the transform's output size and\n        # must not change the input to RandomResizedCrop or validation resize.\n        with Image.open(self.images_dir / self.files[0]) as first:\n            width, height = first.size\n        est_gb = n * height * width * 3 / 1024 ** 3\n        if est_gb > max_cache_gb:\n            print(f\"[dataset] BỎ cache RAM: ước lượng {est_gb:.1f} GiB > ngân sách {max_cache_gb} GiB\")\n            return\n\n        buf = np.empty((n, height, width, 3), dtype=np.uint8)\n        paths = [self.images_dir / f for f in self.files]\n\n        def _read(i: int) -> None:\n            with Image.open(paths[i]) as im:\n                if im.size != (width, height):\n                    raise ValueError(\"RAM cache requires equal original image dimensions\")\n                buf[i] = np.asarray(im.convert(\"RGB\"), dtype=np.uint8)\n\n        with ThreadPoolExecutor(max_workers=4) as pool:\n            for _ in pool.map(_read, range(n)):\n                pass\n        self._cache = buf\n        print(f\"[dataset] đã cache {n} ảnh vào RAM ({est_gb:.2f} GiB, uint8)\")\n\n    def __len__(self) -> int:\n        return len(self.files)\n\n    def __getitem__(self, i: int):\n        if self._cache is not None:\n            img = Image.fromarray(self._cache[i])       # dùng chung buffer, không copy\n        else:\n            with Image.open(self.images_dir / self.files[i]) as im:\n                img = im.convert(\"RGB\")\n        if self.transform is not None:\n            img = self.transform(img)\n        return img, int(self.labels[i]), self.files[i]\n\n    def class_counts(self) -> np.ndarray:\n        \"\"\"Số ảnh của từng lớp trong tập này (chỉ dùng cho tập TRAIN, S2).\"\"\"\n        return np.bincount(self.labels, minlength=NUM_CLASSES).astype(np.float64)\n\n\n# --------------------------------------------------------------------------- #\n# 5. DataLoader\n# --------------------------------------------------------------------------- #\ndef _worker_init_fn(worker_id: int) -> None:\n    \"\"\"Cố định nguồn ngẫu nhiên của từng worker để chạy lại cho ra kết quả giống nhau.\"\"\"\n    seed = (torch.initial_seed() + worker_id) % (2 ** 32)\n    np.random.seed(seed)\n    random.seed(seed)\n\n\ndef make_loader(df: pd.DataFrame, images_dir: str | Path, transform, batch_size: int,\n                train: bool, sampler: str | None = None, num_workers: int = 2,\n                cache_in_ram: bool = False, seed: int = 0, drop_last: bool | None = None,\n                img_size: int = 256, pin_memory: bool | None = None) -> DataLoader:\n    \"\"\"Tạo DataLoader.\n\n      - train=True: shuffle (hoặc dùng sampler); train=False: KHÔNG shuffle, giữ thứ tự df\n        (thứ tự phải ổn định để ghép logit với Filename)\n      - sampler=None | \"balanced\": \"balanced\" dùng WeightedRandomSampler với trọng số\n        1/(số ảnh của lớp) (trục D của GUIDE.md mục 3)\n      - drop_last=True khi train: batch cuối quá nhỏ làm BatchNorm không ổn định.\n        Với sampler=\"balanced\" thì KHÔNG drop_last, vì mỗi batch đã phản ánh đúng phân phối\n        lớp và batch cuối vẫn đủ lớn nếu batch_size thuộc loại nào đó.\n      - pin_memory: None = tự động bật khi có CUDA (chỉ có lợi khi chuyển tensor sang GPU),\n        tắt khi chạy CPU. Truyền True/False để ép.\n      - num_workers=2 vì Kaggle T4 chỉ có 2 vCPU; seed cho worker để tái lập.\n    \"\"\"\n    if sampler not in (None, \"balanced\"):\n        raise ValueError(f\"sampler={sampler!r} không hợp lệ, chọn None hoặc 'balanced'\")\n    if drop_last is None:\n        drop_last = bool(train) and sampler is None\n    if pin_memory is None:\n        pin_memory = torch.cuda.is_available()\n\n    ds = DeepWeedsDataset(df, images_dir, transform, cache_in_ram=cache_in_ram, img_size=img_size)\n\n    gen = torch.Generator()\n    gen.manual_seed(seed)\n    sampler_obj = None\n    shuffle = bool(train)          # val/test: giữ nguyên thứ tự df để ghép logit với Filename\n    if sampler == \"balanced\":\n        counts = ds.class_counts()\n        w = 1.0 / np.maximum(counts, 1.0)\n        weights = torch.as_tensor(w[ds.labels], dtype=torch.double)\n        sampler_obj = WeightedRandomSampler(weights, num_samples=len(ds), replacement=True, generator=gen)\n        shuffle = False\n\n    return DataLoader(\n        ds,\n        batch_size=batch_size,\n        shuffle=shuffle,\n        sampler=sampler_obj,\n        num_workers=num_workers,\n        pin_memory=pin_memory,\n        drop_last=drop_last,\n        worker_init_fn=_worker_init_fn,\n        generator=gen,\n        persistent_workers=bool(num_workers),   # giữ worker sống để không tải lại ảnh mỗi epoch\n    )\n",
  "code/model.py": "\"\"\"model.py - tạo backbone, đóng băng, nhóm tham số, đếm params/GMAC.\n\nHoàn thiện từ bộ khung `starter/`. Giữ nguyên tên hàm và kiểu dữ liệu vào/ra như docstring gốc.\n\n    build_model(name, pretrained, num_classes, drop_rate, init) -> nn.Module\n    freeze_backbone(model)                                   -> None\n    param_groups(model, lr_backbone, lr_head, weight_decay)  -> list[dict] cho optimizer\n    count_params(model) -> float (triệu)     count_gmacs(model, img_size) -> float\n\nHai điểm cần nêu trong báo cáo:\n  - Tag trọng số: mỗi kiến trúc có nhiều tag pretrained (khác công thức huấn luyện). Hàm này\n    ghi tag thực sự được tải vào `model.pretrained_tag` để đưa vào results.xlsx; không ghi\n    tên kiến trúc cũng là so sánh không công bằng (GUIDE.md mục 2.1).\n  - GMAC: công cụ khác nhau cho kết quả chênh nhau vài phần trăm, nên hàm đếm sẵn và trả về\n    tên công cụ đã dùng (`count_gmacs_with_tool`). Mặc định dùng bộ đếm hook tự viết (chỉ đếm\n    Conv2d và Linear) để không phụ thuộc thư viện ngoài trên Kaggle.\n\"\"\"\nfrom __future__ import annotations\n\n# Gợi ý backbone (GUIDE.md mục 2.1). Tag trọng số của timm có thể đổi theo phiên bản:\n# dùng timm.list_pretrained(\"resnet50*\") để xem, và GHI LẠI tag bạn dùng trong results.xlsx.\nSUGGESTED_BACKBONES = {\n    \"resnet50\": \"resnet50\",\n    \"resnext50\": \"resnext50_32x4d\",\n    \"convnext_tiny\": \"convnext_tiny\",\n    \"deit_small\": \"deit_small_patch16_224\",      # hoặc vit_small_patch16_224\n    \"swin_tiny\": \"swin_tiny_patch4_window7_224\",\n    \"efficientnet_b0\": \"efficientnet_b0\",        # mạng nhẹ\n    \"mobilenetv3\": \"mobilenetv3_large_100\",      # mạng nhẹ\n}\n\nINIT_MODES = (\"scratch\", \"frozen\", \"finetune\")\nBN_TYPES = (\"_BatchNorm\", \"SyncBatchNorm\", \"BatchNorm\")\n\n\n# --------------------------------------------------------------------------- #\n# 1. Tạo model\n# --------------------------------------------------------------------------- #\ndef build_model(name: str, pretrained: bool = True, num_classes: int = 9,\n                drop_rate: float = 0.0, init: str = \"finetune\"):\n    \"\"\"Tạo model phân loại 9 lớp.\n\n    `init` (trục A của GUIDE.md mục 3):\n      - \"scratch\"  : pretrained=False, huấn luyện toàn bộ\n      - \"frozen\"   : pretrained=True, đóng băng backbone, chỉ train head\n      - \"finetune\" : pretrained=True, train toàn bộ\n\n    TODO đã làm:\n      - timm.create_model(name, pretrained=..., num_classes=num_classes, drop_rate=...)\n        (timm tự thay head mới; head khởi tạo ngẫu nhiên)\n      - nếu init == \"frozen\": gọi freeze_backbone(model)\n      - ghi lại tên tag trọng số thực sự được tải (model.pretrained_cfg)\n\n    `init=\"scratch\"` ép `pretrained=False` bất kể tham số truyền vào, để không vô tình dùng\n    trọng số tiền huấn luyện trong thí nghiệm \"từ đầu\".\n    \"\"\"\n    import timm\n    import torch.nn as nn\n\n    if init not in INIT_MODES:\n        raise ValueError(f\"init={init!r} không hợp lệ, chọn một trong {INIT_MODES}\")\n    if init == \"scratch\":\n        pretrained = False\n\n    tname = SUGGESTED_BACKBONES.get(name, name)   # chấp nhận cả tên viết tắt lẫn tên timm đầy đủ\n    model = timm.create_model(tname, pretrained=pretrained, num_classes=num_classes,\n                              drop_rate=drop_rate)\n\n    cfg = getattr(model, \"pretrained_cfg\", None) or {}\n    model.pretrained_tag = (cfg.get(\"tag\") or cfg.get(\"hf_hub_id\") or \"?\") if pretrained else \"random_init\"\n    model.timm_name = tname\n    model.init_mode = init\n\n    if init == \"frozen\":\n        freeze_backbone(model)\n    return model\n\n\ndef get_classifier(model):\n    \"\"\"Lấy module head của model (timm đặt tên là `get_classifier`).\"\"\"\n    if hasattr(model, \"get_classifier\"):\n        try:\n            head = model.get_classifier()\n            if head is not None:\n                return head\n        except (AttributeError, NotImplementedError):\n            pass\n    return list(model.children())[-1]              # dự phòng cho model tự viết\n\n\ndef bn_modules(model) -> list:\n    \"\"\"Danh sách module BatchNorm trong model (dùng cho chế độ đóng băng và cho EMA).\"\"\"\n    import torch.nn as nn\n    return [m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]\n\n\ndef freeze_backbone(model) -> None:\n    \"\"\"Đóng băng mọi tham số trừ head.\n\n    TODO đã làm:\n      - requires_grad = False cho tham số backbone; head (model.get_classifier()) vẫn train\n      - BatchNorm của backbone cũng phải ở chế độ eval (xem set_bn_eval)\n\n    Vì sao phải giữ BN ở eval: BatchNorm ở chế độ train dùng/thống kê theo *batch* hiện tại\n    để chuẩn hoá, nên đặc tính đầu ra thay đổi theo batch và gradient sẽ chảy vào thống kê\n    BN — trong khi backbone đóng băng thì các thống kê đó đã bị \"đóng băng\" theo ý nghĩa là\n    phải cố định. Xem `set_bn_eval`: train loop gọi `model.train()` (để head train) rồi gọi\n    lại `set_bn_eval(model)` để đưa các BN về eval.\n    \"\"\"\n    head = get_classifier(model)\n    head_ids = {id(p) for p in head.parameters()}\n    n_frozen = 0\n    for p in model.parameters():\n        if id(p) not in head_ids:\n            p.requires_grad_(False)\n            n_frozen += 1\n    set_bn_eval(model)\n    print(f\"[model] đóng băng {n_frozen} tensor tham số; head \"\n          f\"({sum(p.numel() for p in head.parameters()) / 1e6:.2f}M tham số) vẫn được huấn luyện\")\n\n\ndef set_bn_eval(model) -> None:\n    \"\"\"Đặt mọi BatchNorm của model về chế độ eval (dùng chung cho backbone đóng băng).\"\"\"\n    for m in bn_modules(model):\n        m.eval()\n\n\ndef set_dropout_eval(model) -> None:\n    \"\"\"Đặt dropout về eval. Không dùng trong huấn luyện, chỉ để tiện khi suy luận.\"\"\"\n    import torch.nn as nn\n    for m in model.modules():\n        if isinstance(m, nn.Dropout):\n            m.eval()\n\n\n# --------------------------------------------------------------------------- #\n# 2. Nhóm tham số\n# --------------------------------------------------------------------------- #\ndef param_groups(model, lr_backbone: float, lr_head: float, weight_decay: float):\n    \"\"\"Chia tham số thành 3 nhóm như slide Day 2, trang 52.\n\n    - backbone có ndim > 1: lr = lr_backbone, weight_decay = weight_decay\n    - norm và bias của backbone (ndim <= 1): lr = lr_backbone, weight_decay = 0\n    - head mới: lr = lr_head (thường gấp 10 lần backbone), weight_decay = weight_decay\n\n    TODO đã làm:\n      - bỏ qua tham số requires_grad == False\n      - trả về list[dict] dạng {\"params\": [...], \"lr\": ..., \"weight_decay\": ...}\n      - (trục E) mở rộng: LR theo tầng nếu bạn muốn thử\n\n    Về số nhóm: slide mô tả 3 nhóm. Vì nguyên tắc \"weight_decay = 0 cho norm và bias\" được\n    áp cho cả backbone lẫn head, mà head hầu hết là Linear (weight ndim=2, bias ndim=1), nên\n    head cũng bị tách làm 2 nhóm -> thực tế 4 nhóm cho kiến trúc điển hình. Đây là chi tiết\n    làm đúng nguyên tắc hơn so với slide, được ghi rõ để không bị hiểu nhầm là lỗi.\n    `names` giữ lại tên tham số để tiện kiểm tra.\n    \"\"\"\n    head = get_classifier(model)\n    head_ids = {id(p) for p in head.parameters()}\n    groups: dict[tuple, dict] = {}\n    for name, p in model.named_parameters():\n        if not p.requires_grad:\n            continue\n        is_head = id(p) in head_ids\n        wd = 0.0 if p.ndim <= 1 else weight_decay\n        key = (\"head\" if is_head else \"backbone\", wd > 0.0)\n        if key not in groups:\n            lr = lr_head if is_head else lr_backbone\n            groups[key] = {\"params\": [], \"lr\": lr, \"weight_decay\": wd,\n                           \"names\": []}\n        groups[key][\"params\"].append(p)\n        groups[key][\"names\"].append(name)\n\n    out = [g for g in groups.values() if g[\"params\"]]\n    desc = \", \".join(f\"{'head' if k[0] == 'head' else 'backbone'}/wd={k[1]} \"\n                     f\"({sum(p.numel() for p in g['params']) / 1e6:.2f}M, lr={g['lr']})\"\n                     for k, g in groups.items())\n    print(f\"[model] nhóm tham số ({len(out)} nhóm): {desc}\")\n    return out\n\n\n# --------------------------------------------------------------------------- #\n# 3. Đếm tham số và GMAC\n# --------------------------------------------------------------------------- #\ndef count_params(model) -> float:\n    \"\"\"Số tham số (triệu), đếm cả tham số bị đóng băng.\"\"\"\n    return sum(p.numel() for p in model.parameters()) / 1e6\n\n\ndef _builtin_macs(model, img_size: int = 224) -> float:\n    \"\"\"Đếm MAC bằng forward hook: chỉ tính Conv2d và Linear (bỏ qua norm/activation/pooling).\n\n    Đây là cùng quy ước với số trên slide (ResNet-50 ~4,1 GMAC). Với transformer, phép\n    matmul trong lớp attention không đi qua Linear nên bị bỏ sót; phần đó nhỏ hơn đáng kể\n    so với tổng (khoảng 5-10% với DeiT-S), và điều này được ghi rõ khi báo cáo.\n    \"\"\"\n    import torch\n    import torch.nn as nn\n\n    total = 0\n    handles = []\n\n    def conv_hook(m, inp, out):\n        nonlocal total\n        oh, ow = out.shape[-2:]\n        total += int(oh * ow * out.shape[1] * (m.in_channels // m.groups) * m.kernel_size[0]\n                     * m.kernel_size[1])\n\n    def linear_hook(m, inp, out):\n        nonlocal total\n        total += int((inp[0].numel() // m.in_features) * m.out_features * m.in_features)\n\n    for m in model.modules():\n        if isinstance(m, nn.Conv2d):\n            handles.append(m.register_forward_hook(conv_hook))\n        elif isinstance(m, nn.Linear):\n            handles.append(m.register_forward_hook(linear_hook))\n\n    was_training = model.training\n    model.eval()\n    try:\n        parameter = next(model.parameters())\n        with torch.no_grad():\n            model(torch.zeros(1, 3, img_size, img_size,\n                              device=parameter.device, dtype=parameter.dtype))\n    finally:\n        for h in handles:\n            h.remove()\n        model.train(was_training)\n    return float(total) / 1e9\n\n\ndef count_gmacs_with_tool(model, img_size: int = 224, tool: str = \"builtin\") -> tuple[float, str]:\n    \"\"\"Đếm GMAC và trả về (gmacs, tên_công_cụ). `tool` ∈ {\"builtin\", \"fvcore\", \"ptflops\"}.\n\n    Ghi rõ công cụ đã dùng; số có thể lệch vài phần trăm giữa các công cụ.\n    \"\"\"\n    import torch\n    import torch.nn as nn\n\n    if tool == \"builtin\":\n        return _builtin_macs(model, img_size), \"builtin(hook: Conv2d+Linear)\"\n\n    if tool == \"fvcore\":\n        try:\n            from fvcore.nn import FlopCountAnalysis\n        except ImportError as e:\n            raise ImportError(\"Cần `pip install fvcore`\") from e\n        was_training = model.training\n        model.eval()\n        parameter = next(model.parameters())\n        try:\n            flops = FlopCountAnalysis(model, torch.zeros(1, 3, img_size, img_size,\n                                                       device=parameter.device, dtype=parameter.dtype)).total()\n        finally:\n            model.train(was_training)\n        return float(flops) / 1e9, \"fvcore\"          # fvcore đếm 1 MAC = 1 flop\n\n    if tool == \"ptflops\":\n        try:\n            from ptflops import get_model_complexity_info\n        except ImportError as e:\n            raise ImportError(\"Cần `pip install ptflops`\") from e\n        macs, _ = get_model_complexity_info(model, (3, img_size, img_size), as_strings=False,\n                                            print_per_layer_stat=False, verbose=False)\n        return float(macs) / 1e9, \"ptflops\"\n\n    raise ValueError(f\"tool={tool!r} không hợp lệ\")\n\n\ndef count_gmacs(model, img_size: int = 224) -> float:\n    \"\"\"GMAC cho một ảnh 3 x img_size x img_size (slide tính MAC, không phải FLOPs 2x).\n\n    TODO đã làm: bộ đếm hook tự viết (không cần cài thêm thư viện); nếu muốn đối chiếu với\n    công cụ khác thì dùng `count_gmacs_with_tool(model, img_size, tool=\"fvcore\")`.\n    Ghi rõ công cụ đã dùng; số có thể lệch vài phần trăm giữa các công cụ.\n    \"\"\"\n    return count_gmacs_with_tool(model, img_size, tool=\"builtin\")[0]\n",
  "code/losses.py": "\"\"\"losses.py - các hàm loss và trộn mẫu (Mixup, CutMix).\n\nHoàn thiện từ bộ khung `starter/`. Giữ nguyên tên hàm và kiểu dữ liệu vào/ra như docstring gốc.\nLiên hệ slide Day 2: label smoothing (trang 56), focal loss (trang 57), Mixup/CutMix (trang 48).\n\n    build_criterion(kind, **kw)                 -> callable(logits, target) -> loss scalar\n    class_weights(counts, beta)                 -> tensor trọng số lớp\n    mix_batch(x, y, alpha, mode)                -> (x_mixed, (y_a, y_b, lam))\n    mixed_loss(criterion, logits, targets)      -> loss scalar\n\nLựa chọn cài đặt (ghi lại vì rubric H chấm phần \"cài đặt đúng kỹ thuật\"):\n  - Label smoothing dùng `F.cross_entropy(..., label_smoothing=eps)` của PyTorch. Công thức\n    q'(k) = (1-eps)*1[k==y] + eps/K là đúng như cách PyTorch hiện thực (trừ chênh lệch ~1/K ở\n    lớp đúng), nên kiểm tra `eps=0` cho đúng CE là đủ.\n  - Focal loss tự viết để kiểm soát chính xác cách rút gọn `(1-p_t)^gamma` và cách chuẩn\n    hoá. Với `gamma=0` hàm trả về đúng `F.cross_entropy` (sai số < 1e-6, xem selftest.py).\n  - CutMix điều chỉnh `lam` theo DIỆN TÍCH THỰC của hộp sau khi cắt ra ngoài biên, và đặt\n    hộp theo tỉ lệ diện tích đích, không phải số pixel tuyệt đối.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n# --------------------------------------------------------------------------- #\n# 1. Các hàm loss\n# --------------------------------------------------------------------------- #\nclass LabelSmoothingCE(nn.Module):\n    \"\"\"Cross-entropy với label smoothing: q'(k) = (1 - eps) * 1[k == y] + eps / K  (slide trang 56).\n\n    TODO đã làm: dùng `torch.nn.functional.cross_entropy(label_smoothing=eps)`.\n    Kiểm tra `eps = 0` cho đúng CE (xem selftest.py: check_label_smoothing_zero_is_ce).\n    \"\"\"\n\n    def __init__(self, smoothing: float = 0.1):\n        super().__init__()\n        if not 0.0 <= smoothing < 1.0:\n            raise ValueError(f\"smoothing phải nằm trong [0, 1), nhận {smoothing}\")\n        self.smoothing = float(smoothing)\n\n    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        return F.cross_entropy(logits, target, label_smoothing=self.smoothing)\n\n    def extra_repr(self) -> str:\n        return f\"smoothing={self.smoothing}\"\n\n\nclass FocalLoss(nn.Module):\n    \"\"\"Focal loss nhiều lớp: FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)  (slide trang 57).\n\n    TODO đã làm:\n      - tính log_softmax, lấy p_t của lớp đúng, nhân (1 - p_t)^gamma, lấy trung bình batch\n      - alpha: None hoặc vector trọng số theo lớp\n    BẮT BUỘC viết một kiểm tra nhỏ: gamma = 0 phải cho đúng cross-entropy (sai số < 1e-6)\n    -- xem selftest.py: check_focal_gamma_zero_is_ce.\n\n    Với `reduction=\"none\"` trả về từng mẫu (cần cho mixup theo mẫu); mặc định \"mean\".\n    `alpha=None` nghĩa là không cân bằng lớp, đây là focal thuần theo slide.\n    \"\"\"\n\n    def __init__(self, gamma: float = 2.0, alpha=None, reduction: str = \"mean\"):\n        super().__init__()\n        if gamma < 0:\n            raise ValueError(f\"gamma phải >= 0, nhận {gamma}\")\n        if reduction not in (\"mean\", \"sum\", \"none\"):\n            raise ValueError(f\"reduction={reduction!r} không hợp lệ\")\n        self.gamma = float(gamma)\n        self.reduction = reduction\n        if alpha is None:\n            self.register_buffer(\"alpha\", None)\n        else:\n            a = torch.as_tensor(alpha, dtype=torch.float32)\n            if a.ndim != 1:\n                raise ValueError(\"alpha phải là vector 1-D theo lớp\")\n            self.register_buffer(\"alpha\", a)\n\n    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        if target.ndim != 1 or logits.ndim != 2:\n            raise ValueError(f\"cần logits (N, K) và target (N,), nhận {tuple(logits.shape)}, {tuple(target.shape)}\")\n        log_p = F.log_softmax(logits, dim=-1)\n        log_pt = log_p.gather(1, target.unsqueeze(1)).squeeze(1)          # log p_t\n        pt = log_pt.exp()\n        loss = -((1.0 - pt) ** self.gamma) * log_pt                       # -alpha_t (1-p_t)^g log p_t\n        if self.alpha is not None:\n            if self.alpha.numel() != logits.shape[1]:\n                raise ValueError(f\"alpha có {self.alpha.numel()} phần tử, logits có {logits.shape[1]} lớp\")\n            loss = loss * self.alpha.to(loss.dtype)[target]\n        if self.reduction == \"mean\":\n            return loss.mean()\n        if self.reduction == \"sum\":\n            return loss.sum()\n        return loss\n\n    def extra_repr(self) -> str:\n        return f\"gamma={self.gamma}, alpha={'None' if self.alpha is None else 'yes'}\"\n\n\ndef class_weights(counts, beta: float = 0.0):\n    \"\"\"Trọng số theo lớp từ số ảnh mỗi lớp trong tập TRAIN.\n\n    - beta = 0: trọng số tỉ lệ nghịch với số ảnh (1 / n_c), chuẩn hoá về trung bình 1\n    - beta > 0: class-balanced theo \"số mẫu hiệu dụng\": w_c = (1 - beta) / (1 - beta ** n_c)\n      (slide trang 57, Cui et al. arXiv:1901.05555); chuẩn hoá tổng trọng số về số lớp\n\n    TODO đã làm: trả về tensor độ dài 9, chuẩn hoá sao cho trung bình = 1 (tức tổng = K).\n    Chỉ dùng số liệu của TRAIN (`DeepWeedsDataset.class_counts()`); không dùng val hay test (S4).\n    \"\"\"\n    n = np.asarray(counts, dtype=np.float64).reshape(-1)\n    if (n <= 0).any():\n        raise ValueError(\"mọi lớp phải có ít nhất 1 ảnh trong tập train\")\n    if beta < 0 or beta >= 1:\n        raise ValueError(f\"beta phải nằm trong [0, 1), nhận {beta}\")\n    if beta == 0:\n        w = 1.0 / n\n    else:\n        w = (1.0 - beta) / (1.0 - np.power(beta, n))\n    w = w / w.mean()                      # trung bình = 1\n    return torch.tensor(w, dtype=torch.float32)\n\n\ndef build_criterion(kind: str = \"ce\", **kw):\n    \"\"\"Trả về hàm loss theo `kind`: \"ce\", \"ls\" (label smoothing), \"focal\", \"ce_weighted\".\n\n    Ví dụ kw: smoothing=0.1, gamma=2.0, alpha=None, weight=tensor.\n\n    TODO đã làm: tạo đúng loss, hoặc gọi các lớp bên dưới.\n\n    Với \"ce_weighted\" có thể truyền sẵn `weight=<tensor>` (từ `class_weights`), hoặc truyền\n    `counts=<array số ảnh train>` + `beta=<float>` để tự tính trọng số.\n    \"\"\"\n    if kind == \"ce\":\n        weight = kw.get(\"weight\")\n        return nn.CrossEntropyLoss(weight=None if weight is None else torch.as_tensor(weight, dtype=torch.float32))\n\n    if kind == \"ls\":\n        return LabelSmoothingCE(smoothing=float(kw.get(\"smoothing\", 0.1)))\n\n    if kind == \"focal\":\n        return FocalLoss(gamma=float(kw.get(\"gamma\", 2.0)), alpha=kw.get(\"alpha\"),\n                         reduction=kw.get(\"reduction\", \"mean\"))\n\n    if kind == \"ce_weighted\":\n        if \"weight\" in kw and kw[\"weight\"] is not None:\n            weight = kw[\"weight\"]\n        else:\n            if \"counts\" not in kw:\n                raise ValueError(\"'ce_weighted' cần weight=<tensor> hoặc counts=<array> và beta=<float>\")\n            weight = class_weights(kw[\"counts\"], beta=float(kw.get(\"beta\", 0.0)))\n        return nn.CrossEntropyLoss(weight=torch.as_tensor(weight, dtype=torch.float32))\n\n    raise ValueError(f\"kind={kind!r} không hợp lệ, chọn 'ce', 'ls', 'focal' hoặc 'ce_weighted'\")\n\n\n# --------------------------------------------------------------------------- #\n# 2. Trộn batch (Mixup / CutMix)\n# --------------------------------------------------------------------------- #\ndef mix_batch(x, y, alpha: float = 1.0, mode: str = \"cutmix\"):\n    \"\"\"Trộn một batch ảnh và nhãn.\n\n    - lam ~ Beta(alpha, alpha)\n    - mode=\"mixup\": x_mix = lam * x + (1 - lam) * x[perm]\n    - mode=\"cutmix\": cắt một hộp chữ nhật từ x[perm] dán vào x, rồi điều chỉnh lam theo\n      DIỆN TÍCH THỰC của hộp sau khi cắt ra ngoài biên (slide trang 48)\n    - trả về (x_mix, (y_a, y_b, lam)) với y_a = y, y_b = y[perm]\n\n    TODO đã làm: tự cài đặt.\n    Diện tích hộp được chọn theo tỉ lệ `1 - lam` của tổng diện tích ảnh, rồi cắt theo biên\n    ảnh; `lam` cuối cùng = 1 - area_thuc_te / (H*W). Kiểm tra bằng mắt: xem selftest.py\n    (check_cutmix_lambda_matches_area) và hàm `show_mixed_batch` trong notebook.\n    \"\"\"\n    if mode not in (\"mixup\", \"cutmix\"):\n        raise ValueError(f\"mode={mode!r} không hợp lệ, chọn 'mixup' hoặc 'cutmix'\")\n    if x.ndim != 4:\n        raise ValueError(f\"x phải có dạng (N, C, H, W), nhận {tuple(x.shape)}\")\n\n    lam = float(np.random.beta(alpha, alpha))\n    perm = torch.randperm(x.size(0), device=x.device)\n    y_a, y_b = y, y[perm]\n\n    if mode == \"mixup\":\n        return lam * x + (1.0 - lam) * x[perm], (y_a, y_b, lam)\n\n    _, _, h, w = x.shape\n    cut_ratio = float(np.sqrt(1.0 - lam))\n    cut_h, cut_w = int(h * cut_ratio), int(w * cut_ratio)\n    if cut_h == 0 or cut_w == 0:\n        return x.clone(), (y_a, y_b, 1.0)\n\n    # tâm hộp ngẫu nhiên, cho phép hộp tràn ra ngoài biên như CutMix gốc\n    cy = int(np.random.randint(h))\n    cx = int(np.random.randint(w))\n    y0, y1 = np.clip(cy - cut_h // 2, 0, h), np.clip(cy + cut_h // 2, 0, h)\n    x0, x1 = np.clip(cx - cut_w // 2, 0, w), np.clip(cx + cut_w // 2, 0, w)\n\n    x_mix = x.clone()\n    x_mix[:, :, y0:y1, x0:x1] = x[perm][:, :, y0:y1, x0:x1]\n    lam = 1.0 - ((y1 - y0) * (x1 - x0) / (h * w))\n    return x_mix, (y_a, y_b, lam)\n\n\ndef mixed_loss(criterion, logits, targets):\n    \"\"\"Loss cho batch đã trộn: lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b).\n\n    TODO đã làm. Lưu ý: accuracy trên batch đã trộn không còn nghĩa bình thường; đánh giá bằng val.\n    (Train loop chỉ ghi `train_top1` khi không trộn, và luôn chọn checkpoint bằng macro-F1 val.)\n    \"\"\"\n    y_a, y_b, lam = targets\n    return lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)\n\n\n@torch.no_grad()\ndef soft_target_nll(logits: torch.Tensor, targets) -> torch.Tensor:\n    \"\"\"NLL với nhãn mềm: -sum_k q_k * log_softmax_k. Tiện khi muốn dùng nhãn mềm trực tiếp.\"\"\"\n    log_p = F.log_softmax(logits, dim=-1)\n    q = torch.as_tensor(targets, dtype=logits.dtype, device=logits.device)\n    return -(q * log_p).sum(-1).mean()",
  "code/train.py": "\"\"\"train.py - vòng huấn luyện cho mọi thí nghiệm (B, T, F).\n\nHoàn thiện từ bộ khung `starter/`. Giữ nguyên tên hàm và kiểu dữ liệu vào/ra như docstring gốc.\nDùng MỘT hàm `run(cfg)` cho mọi cấu hình (RUBRIC mục H): đổi thí nghiệm chỉ bằng cách đổi `Config`.\n\nChạy một thí nghiệm từ dòng lệnh:\n    python train.py --set exp_id=B01 backbone=resnet50 seed=0\nChỉ số dùng để chọn checkpoint (macro-F1 val) tính bằng eval.compute_metrics của repo gốc:\n    sys.path.insert(0, \"<thư mục chứa eval.py>\");  from eval import compute_metrics\n\nGhi chú thiết kế cho môi trường Kaggle (Tesla T4, 2 vCPU):\n  - AMP dùng **fp16** (T4 là sm_75, không hỗ trợ bf16) nên luôn cần GradScaler.\n  - `cache_images=True` nạp ảnh vào RAM để không bị nghẽn giải mã JPEG trên 2 vCPU.\n  - `resume=True` nối tiếp từ `last.pt` nếu phiên bị đóng giữa chừng.\n  - Mỗi epoch ghi `last.pt` và `best.pt`; sau khi huấn luyện xong thì xoá `last.pt`\n    (chỉ chứa optimizer state, không cần cho suy luận) để tiết kiệm ~200 MB mỗi lần chạy.\n  - Logit val/test được lưu ra `.npy` (GUIDE.md mục 4) để phần lớn thí nghiệm suy luận ở\n    Bước 3 chạy được không cần GPU.\n\"\"\"\nfrom __future__ import annotations\n\nimport runtime  # Windows pandas compatibility shared with the notebook.\n\nimport argparse\nimport dataclasses\nimport json\nimport math\nimport os\nimport random\nimport sys\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\n# Ghi file dự đoán đúng định dạng bằng hàm có sẵn trong eval.py (repo gốc):\n#     from eval import save_predictions, compute_metrics\n# Log theo epoch (history.csv) và config.json bạn tự ghi bằng pandas/json.\n\n_HERE = Path(__file__).resolve().parent\n\n\ndef _setup_console() -> None:\n    \"\"\"Cho phép in tiếng Việt khi chạy trên Windows (console cp1252).\n\n    Trên Kaggle/Colab (Linux) hàm này không có tác dụng gì.\n    \"\"\"\n    for stream in (sys.stdout, sys.stderr):\n        if hasattr(stream, \"reconfigure\"):\n            try:\n                stream.reconfigure(encoding=\"utf-8\", errors=\"replace\")\n            except (ValueError, OSError):\n                pass\n\n\n_setup_console()\n\n\ndef _import_eval():\n    \"\"\"Import `eval.py` của repo gốc mà không cần sửa file đó.\n\n    Thử lần lượt: thư mục cha (code/ nằm trong repo), cùng thư mục, rồi thư mục làm việc.\n    \"\"\"\n    if \"eval\" in sys.modules:\n        import eval as ev\n        return ev\n    for cand in (_HERE.parent / \"eval.py\", _HERE / \"eval.py\", Path.cwd() / \"eval.py\"):\n        if cand.exists():\n            sys.path.insert(0, str(cand.parent))\n            import eval as ev\n            return ev\n    raise ImportError(\"Không tìm thấy eval.py. Hãy đặt code/ cạnh eval.py của repo bài lab.\")\n\n\n@dataclass\nclass Config:\n    # --- định danh ---\n    exp_id: str = \"T00\"\n    seed: int = 0\n    fold: int = 0\n    # --- mô hình ---\n    backbone: str = \"resnet50\"\n    init: str = \"finetune\"            # scratch | frozen | finetune\n    drop_rate: float = 0.0\n    # --- dữ liệu / augmentation ---\n    img_size: int = 224\n    aug: str = \"basic\"                # trivial | color | basic | randaug | basic_vflip\n    sampler: str | None = None        # None | balanced\n    mix: str | None = None            # None | mixup | cutmix\n    mix_alpha: float = 1.0\n    # --- loss ---\n    loss: str = \"ce\"                  # ce | ls | focal | ce_weighted\n    label_smoothing: float = 0.0\n    focal_gamma: float = 2.0\n    class_weight_beta: float | None = None\n    # --- tối ưu (công thức nền, GUIDE.md mục 1.4) ---\n    epochs: int = 12\n    batch_size: int = 64\n    lr_backbone: float = 1e-4\n    lr_head: float = 1e-3\n    weight_decay: float = 0.05\n    warmup_epochs: float = 1.0\n    ema_decay: float | None = None\n    amp: bool = True\n    num_workers: int = 2\n    # --- tuỳ chọn đã thêm (được phép: \"bạn được thêm hàm, tham số, file mới\") ---\n    grad_clip: float = 1.0            # 0 = không clip\n    min_lr_ratio: float = 0.01        # cosine về còn min_lr_ratio * lr ban đầu\n    deterministic: bool = False       # True: cudnn.deterministic, chậm hơn nhưng tái lập tốt hơn\n    cache_images: bool = True         # nạp ảnh vào RAM (xem dataset.py)\n    pin_memory: bool | None = None     # None = tự bật khi có CUDA; tắt nếu máy lỗi pinned memory\n    check_split: bool = True          # chạy kiểm tra S1-S4 mỗi lần (cache kết quả vào json)\n    resume: bool = False              # nối tiếp từ last.pt nếu có (Kaggle bị ngắt phiên)\n    save_val_predictions: bool = True # luôn ghi predictions/<exp>_seed<k>_val.csv (cần cho --final-val)\n    curves_dir: str = \"curves\"\n    logit_dir: str = \"logits\"         # lưu val/test logits ra .npy để Bước 3 không cần GPU\n    # --- đường dẫn ---\n    images_dir: str = \"data/images\"\n    labels_dir: str = \"data/labels\"\n    out_dir: str = \"runs\"             # config.json, history.csv, checkpoint, logit của từng lần chạy\n    pred_dir: str = \"predictions\"     # file dự đoán đúng định dạng eval.py (nộp cùng bài)\n    # --- chỉ bật ở Bước 4 (chung kết): ghi predictions trên TEST. Mặc định TẮT (quy tắc S4). ---\n    save_test_predictions: bool = False\n    # Isolated pipeline verification only; never a reportable experiment.\n    debug_samples_per_class: int | None = None\n\n\ndef run_dir(cfg: Config) -> Path:\n    \"\"\"Thư mục kết quả của một lần chạy: <out_dir>/<exp_id>/seed<k>/ .\"\"\"\n    return Path(cfg.out_dir) / cfg.exp_id / f\"seed{cfg.seed}\"\n\n\ndef pred_path(cfg: Config, split: str) -> Path:\n    \"\"\"Đường dẫn chuẩn của file dự đoán: <pred_dir>/<exp_id>_seed<k>_<split>.csv (split = val | test).\"\"\"\n    return Path(cfg.pred_dir) / f\"{cfg.exp_id}_seed{cfg.seed}_{split}.csv\"\n\n\ndef logit_path(cfg: Config, split: str) -> Path:\n    \"\"\"Đường dẫn file logit: <logit_dir>/<exp_id>/seed<k>_<split>.npz\"\"\"\n    return Path(cfg.logit_dir) / cfg.exp_id / f\"seed{cfg.seed}_{split}.npz\"\n\n\n# --------------------------------------------------------------------------- #\n# 1. Seed, optimizer, scheduler, EMA\n# --------------------------------------------------------------------------- #\ndef set_seed(seed: int, deterministic: bool = False) -> None:\n    \"\"\"Cố định mọi nguồn ngẫu nhiên.\n\n    TODO đã làm: random, numpy, torch (CPU và CUDA), os.environ PYTHONHASHSEED, seed cho\n    worker của DataLoader (xem dataset._worker_init_fn).\n\n    Về mức độ tái lập: seed cố định head, thứ tự batch và augmentation. Với\n    `deterministic=False`, `cudnn.benchmark=True` để T4 chọn thuật toán nhanh nhất; thuật toán\n    đã chọn có thể khác giữa hai lần chạy nên kết quả KHÔNG giống nhau từng bit, nhưng sai\n    số ở mức ~1e-6 và không đổi kết luận. Bật `deterministic=True` (Config) khi cần đối chiếu\n    một cấu hình cụ thể. Điều này được ghi lại trong báo cáo.\n    \"\"\"\n    os.environ[\"PYTHONHASHSEED\"] = str(seed)\n    random.seed(seed)\n    np.random.seed(seed)\n    import torch\n\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = bool(deterministic)\n    torch.backends.cudnn.benchmark = not deterministic\n\n\ndef build_optimizer(model, cfg: Config):\n    \"\"\"AdamW với 3 nhóm tham số (xem model.param_groups).\n\n    TODO đã làm: groups từ `param_groups`, AdamW với betas (0.9, 0.999) mặc định của AdamW.\n    \"\"\"\n    import torch\n\n    from model import param_groups\n\n    groups = param_groups(model, cfg.lr_backbone, cfg.lr_head, cfg.weight_decay)\n    return torch.optim.AdamW(groups, betas=(0.9, 0.999))\n\n\ndef build_scheduler(optimizer, cfg: Config, steps_per_epoch: int):\n    \"\"\"Warmup tuyến tính rồi cosine về ~0 (slide trang 55).\n\n    TODO đã làm: cập nhật theo BƯỚC (iteration) bằng `LambdaLR`; mỗi nhóm tham số giữ đúng\n    tỉ lệ lr của nó vì LambdaLR nhân với lr gốc của từng nhóm.\n    Vẽ đường LR theo bước để thấy đúng hình warmup + cosine: `curves/<exp_id>_lr.png`.\n    \"\"\"\n    import torch\n\n    total = max(1, cfg.epochs * steps_per_epoch)\n    warmup = max(0, int(round(cfg.warmup_epochs * steps_per_epoch)))\n    floor = float(cfg.min_lr_ratio)\n\n    def fn(step: int) -> float:\n        if warmup > 0 and step < warmup:\n            return (step + 1) / warmup\n        p = (step - warmup) / max(1, total - warmup)\n        return floor + (1.0 - floor) * 0.5 * (1.0 + math.cos(math.pi * min(1.0, p)))\n\n    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)\n\n\nclass EMA:\n    \"\"\"Trung bình động trọng số: W_ema <- d * W_ema + (1 - d) * W  (slide trang 56).\n\n    TODO đã làm:\n      - __init__(model, decay): sao chép trọng số (bản sao riêng, ở chế độ eval)\n      - update(model): sau mỗi bước tối ưu\n      - copy_to(model): chép sang model để đánh giá/suy luận bằng trọng số EMA\n      - BatchNorm: mọi tensor float trong state_dict (kể cả running_mean/var) đều được\n        trung bình như tham số -- đây cũng là cách timm làm; tensor nguyên (ví dụ\n        num_batches_tracked) được chép thẳng. Nếu muốn giữ nguyên thống kê BN của model\n        đang huấn luyện thì truyền `copy_buffers=True`.\n    \"\"\"\n\n    def __init__(self, model, decay: float, copy_buffers: bool = False):\n        if not 0.0 <= decay < 1.0:\n            raise ValueError(f\"decay phải nằm trong [0, 1), nhận {decay}\")\n        import copy\n\n        import torch\n\n        self.decay = float(decay)\n        self.copy_buffers = copy_buffers\n        self.module = copy.deepcopy(model).eval()\n        for p in self.module.parameters():\n            p.requires_grad_(False)\n        self._ema_state = self.module.state_dict()\n        self._model_state = model.state_dict()\n        for v in self._ema_state.values():\n            if torch.is_floating_point(v):\n                v.requires_grad_(False)\n\n    @torch.no_grad()\n    def update(self, model) -> None:\n        import torch\n\n        msd = model.state_dict()\n        for k, e in self._ema_state.items():\n            m = msd[k].detach()\n            if torch.is_floating_point(e) and not self.copy_buffers:\n                e.mul_(self.decay).add_(m, alpha=1.0 - self.decay)\n            else:\n                e.copy_(m)\n\n    @torch.no_grad()\n    def copy_to(self, model) -> None:\n        model.load_state_dict(self.module.state_dict(), strict=True)\n\n    def state_dict(self) -> dict:\n        return self.module.state_dict()\n\n\n# --------------------------------------------------------------------------- #\n# 2. Vòng huấn luyện và đánh giá\n# --------------------------------------------------------------------------- #\ndef _amp_on(cfg: Config, device) -> bool:\n    return bool(cfg.amp) and getattr(device, \"type\", \"cpu\") == \"cuda\"\n\n\ndef train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, cfg: Config,\n                    device, ema: EMA | None = None, lr_trace: list | None = None) -> dict:\n    \"\"\"Một epoch huấn luyện. Trả về dict, ví dụ {\"train_loss\": ..., \"lr\": ...}.\n\n    TODO đã làm:\n      - model.train() (nếu init == \"frozen\": giữ phần backbone ở eval, xem model.set_bn_eval)\n      - nếu cfg.mix: mix_batch rồi mixed_loss (losses.py)\n      - AMP (autocast fp16 + GradScaler), clip gradient trước optimizer.step(), scheduler.step()\n      - nếu có EMA: ema.update(model)\n      - `lr_trace` (nếu truyền) ghi (global_step, lr_backbone) để vẽ đường LR theo bước.\n    \"\"\"\n    import torch\n\n    model.train()\n    if cfg.init == \"frozen\":\n        from model import set_bn_eval\n        set_bn_eval(model)\n\n    amp_on = _amp_on(cfg, device)\n    loss_sum = 0.0\n    top1_sum, n_seen = 0.0, 0\n    t0 = time.perf_counter()\n\n    for images, labels, _ in loader:\n        images = images.to(device, non_blocking=True)\n        labels = labels.to(device, non_blocking=True)\n\n        targets = None\n        if cfg.mix:\n            from losses import mix_batch\n            images, targets = mix_batch(images, labels, alpha=cfg.mix_alpha, mode=cfg.mix)\n\n        with torch.autocast(device_type=\"cuda\", dtype=torch.float16, enabled=amp_on):\n            logits = model(images)\n            if targets is not None:\n                from losses import mixed_loss\n                loss = mixed_loss(criterion, logits, targets)\n            else:\n                loss = criterion(logits, labels)\n\n        optimizer.zero_grad(set_to_none=True)\n        scaler.scale(loss).backward()\n        if cfg.grad_clip and cfg.grad_clip > 0:\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(\n                [p for g in optimizer.param_groups for p in g[\"params\"] if p.grad is not None],\n                cfg.grad_clip)\n        scale_before = scaler.get_scale()\n        scaler.step(optimizer)\n        scaler.update()\n        # AMP overflow skips optimizer.step; schedule/EMA must skip the same update.\n        if scaler.get_scale() >= scale_before:\n            scheduler.step()\n            if ema is not None:\n                ema.update(model)\n\n        bs = images.size(0)\n        loss_sum += loss.detach().item() * bs\n        n_seen += bs\n        if targets is None:\n            # accuracy trên batch đã trộn không còn nghĩa bình thưng -> không ghi khi mix\n            top1_sum += (logits.detach().argmax(1) == labels).sum().item()\n        if lr_trace is not None:\n            lr_trace.append((scheduler.last_epoch, optimizer.param_groups[0][\"lr\"]))\n\n    return {\n        \"train_loss\": loss_sum / max(1, n_seen),\n        \"train_top1\": (top1_sum / n_seen) if (cfg.mix is None and n_seen) else None,\n        \"lr\": optimizer.param_groups[0][\"lr\"],\n        \"seconds\": time.perf_counter() - t0,\n    }\n\n\ndef evaluate(model, loader, criterion, device):\n    \"\"\"Chạy model trên một loader ở chế độ eval, KHÔNG tính gradient.\n\n    Trả về (filenames: list[str], y_true: ndarray[N], logits: ndarray[N, 9], loss: float).\n    Giữ đúng thứ tự của loader để ghép logit với tên file.\n\n    TODO đã làm: model.eval(), torch.inference_mode(), gom kết quả.\n    Đánh giá chạy FP32 (không autocast) để xác suất ổn định, vì xác suất này dùng cho ECE,\n    temperature scaling và cả file predictions. Nếu muốn tiết kiệm thời gian thì chấp nhận\n    fp16 nhưng phải ghi rõ trong báo cáo.\n    \"\"\"\n    import torch\n\n    model.eval()\n    filenames: list[str] = []\n    ys, logits_all = [], []\n    loss_sum, n_seen = 0.0, 0\n    with torch.inference_mode():\n        for images, labels, names in loader:\n            images = images.to(device, non_blocking=True)\n            labels = labels.to(device, non_blocking=True)\n            logits = model(images)\n            loss_sum += criterion(logits, labels).item() * images.size(0)\n            n_seen += images.size(0)\n            logits_all.append(logits.float().cpu().numpy())\n            ys.append(labels.cpu().numpy())\n            filenames.extend(names)\n    return (filenames,\n            np.concatenate(ys) if ys else np.zeros(0, dtype=np.int64),\n            np.concatenate(logits_all) if logits_all else np.zeros((0, 9), dtype=np.float32),\n            loss_sum / max(1, n_seen))\n\n\ndef softmax_np(logits: np.ndarray) -> np.ndarray:\n    \"\"\"softmax ổn định trên numpy (trừ max trước để không tràn số).\"\"\"\n    z = np.asarray(logits, dtype=np.float64)\n    z = z - z.max(axis=1, keepdims=True)\n    e = np.exp(z)\n    return e / e.sum(axis=1, keepdims=True)\n\n\ndef metrics_from_logits(y_true: np.ndarray, logits: np.ndarray) -> dict:\n    \"\"\"Chỉ số theo ĐÚNG định nghĩa của eval.py (dùng eval.compute_metrics).\"\"\"\n    ev = _import_eval()\n    probs = softmax_np(logits)\n    return ev.compute_metrics(y_true, probs.argmax(1), probs)\n\n\ndef save_logits(path: str | Path, filenames, logits: np.ndarray, y_true: np.ndarray) -> Path:\n    \"\"\"Lưu logit + tên file + nhãn ra .npz để Bước 3 xử lý TTA/ensemble/T mà không cần GPU.\"\"\"\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(path, filenames=np.asarray(list(filenames), dtype=object),\n                        logits=np.asarray(logits, dtype=np.float32),\n                        y_true=np.asarray(y_true, dtype=np.int64))\n    return path\n\n\ndef load_logits(path: str | Path) -> tuple[list[str], np.ndarray, np.ndarray]:\n    \"\"\"Đọc lại file .npz của save_logits -> (filenames, logits, y_true).\"\"\"\n    d = np.load(path, allow_pickle=True)\n    return [str(x) for x in d[\"filenames\"]], d[\"logits\"], d[\"y_true\"]\n\n\ndef save_checkpoint(path: Path, model, ema: EMA | None, epoch: int,\n                    best_macro_f1: float, best_epoch: int, extra: dict | None = None) -> None:\n    \"\"\"Lưu trọng số (và trọng số EMA nếu có). `extra` dành cho state của optimizer/scheduler.\"\"\"\n    payload = {\"model\": model.state_dict(), \"epoch\": epoch,\n               \"best_macro_f1\": best_macro_f1, \"best_epoch\": best_epoch,\n               \"pretrained_tag\": getattr(model, \"pretrained_tag\", \"unknown\")}\n    if ema is not None:\n        payload[\"ema\"] = ema.state_dict()\n    if extra:\n        payload.update(extra)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    torch_save(payload, path)\n\n\ndef torch_save(payload, path):\n    import torch\n\n    torch.save(payload, path)\n\n\ndef load_state(cfg: Config, device=None, prefer_ema: bool = False):\n    \"\"\"Dựng lại model của một lần chạy đã có và nạp checkpoint tốt nhất (dùng cho Bước 3).\n\n    Trả về (model, best_epoch, best_macro_f1_val).\n    \"\"\"\n    import torch\n\n    from model import build_model\n\n    rdir = run_dir(cfg)\n    ckpt_path = rdir / \"best.pt\"\n    if not ckpt_path.exists():\n        raise FileNotFoundError(f\"Không có checkpoint {ckpt_path}; hãy chạy run(Config(...)) trước\")\n    ckpt = torch.load(ckpt_path, map_location=\"cpu\", weights_only=False)\n    model = build_model(cfg.backbone, pretrained=False, num_classes=9,\n                        drop_rate=cfg.drop_rate, init=cfg.init)\n    key = \"ema\" if (prefer_ema and \"ema\" in ckpt) else \"model\"\n    model.load_state_dict(ckpt[key])\n    model.pretrained_tag = ckpt.get(\"pretrained_tag\", model.pretrained_tag)\n    if device is not None:\n        model.to(device)\n    return model, ckpt.get(\"best_epoch\", -1), ckpt.get(\"best_macro_f1\", float(\"nan\"))\n\n\n# --------------------------------------------------------------------------- #\n# 3. Biểu đồ\n# --------------------------------------------------------------------------- #\ndef plot_curves(history: list[dict], path: str | Path, title: str,\n                lr_trace: list | None = None) -> None:\n    \"\"\"Vẽ đường cong training của một thí nghiệm -> curves/<exp_id>_<mota>.png (GUIDE.md mục 6.2).\n\n    TODO đã làm: 3 khung — (1) loss train/val, (2) macro-F1 val (kèm top-1 val),\n    (3) LR theo bước để thấy warmup + cosine. Có tiêu đề, nhãn trục, chú thích, lưới.\n    \"\"\"\n    import matplotlib\n\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n\n    ep = [h[\"epoch\"] for h in history]\n    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))\n\n    ax = axes[0]\n    ax.plot(ep, [h[\"train_loss\"] for h in history], \"o-\", label=\"train loss\", color=\"tab:blue\")\n    ax.plot(ep, [h[\"val_loss\"] for h in history], \"s-\", label=\"val loss\", color=\"tab:orange\")\n    ax.set_xlabel(\"epoch\")\n    ax.set_ylabel(\"loss\")\n    ax.set_title(\"Loss\")\n    ax.legend()\n    ax.grid(alpha=0.3)\n\n    ax = axes[1]\n    ax.plot(ep, [h[\"val_macro_f1\"] for h in history], \"s-\", label=\"val macro-F1\", color=\"tab:green\")\n    if any(h.get(\"val_top1\") is not None for h in history):\n        ax.plot(ep, [h[\"val_top1\"] for h in history], \"^--\", label=\"val top-1\", color=\"tab:red\")\n    if any(h.get(\"train_top1\") is not None for h in history):\n        tr = [(e, t) for e, t in zip(ep, [h.get(\"train_top1\") for h in history]) if t is not None]\n        if tr:\n            ax.plot([a for a, _ in tr], [b for _, b in tr], \"o--\", label=\"train top-1\",\n                    color=\"tab:purple\", alpha=0.7)\n    best = max(history, key=lambda h: h[\"val_macro_f1\"])\n    ax.axvline(best[\"epoch\"], ls=\":\", c=\"k\", lw=1,\n               label=f\"best epoch = {best['epoch']} (macro-F1 {best['val_macro_f1']:.4f})\")\n    ax.set_xlabel(\"epoch\")\n    ax.set_ylabel(\"score\")\n    ax.set_title(\"Val metric\")\n    ax.legend(fontsize=8)\n    ax.grid(alpha=0.3)\n\n    ax = axes[2]\n    if lr_trace:\n        steps = [s for s, _ in lr_trace]\n        lrs = [v for _, v in lr_trace]\n        ax.plot(steps, lrs, color=\"tab:purple\")\n    ax.set_xlabel(\"step\")\n    ax.set_ylabel(\"lr (backbone group)\")\n    ax.set_title(f\"LR schedule (warmup {history[0].get('warmup_steps', '?')} bước)\")\n    ax.grid(alpha=0.3)\n\n    fig.suptitle(title)\n    fig.tight_layout()\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(path, dpi=150, bbox_inches=\"tight\")\n    plt.close(fig)\n\n\n# --------------------------------------------------------------------------- #\n# 4. run(cfg) - hàm duy nhất cho mọi thí nghiệm\n# --------------------------------------------------------------------------- #\ndef _build_criterion(cfg: Config, train_df):\n    \"\"\"Dựng criterion từ Config. Trọng số lớp chỉ tính từ TRAIN (S2).\"\"\"\n    from losses import build_criterion\n\n    if cfg.loss == \"ls\":\n        return build_criterion(\"ls\", smoothing=cfg.label_smoothing or 0.1)\n    if cfg.loss == \"focal\":\n        return build_criterion(\"focal\", gamma=cfg.focal_gamma)\n    if cfg.loss == \"ce_weighted\":\n        counts = np.bincount(train_df[\"Label\"].to_numpy(), minlength=9)\n        beta = 0.0 if cfg.class_weight_beta is None else float(cfg.class_weight_beta)\n        return build_criterion(\"ce_weighted\", counts=counts, beta=beta)\n    return build_criterion(cfg.loss)\n\n\ndef run(cfg: Config) -> dict:\n    \"\"\"Huấn luyện một cấu hình và lưu mọi thứ cần thiết. Trả về dict kết quả tóm tắt.\n\n    TODO đã làm, theo thứ tự:\n      1. set_seed; tạo thư mục run_dir(cfg); ghi config.json (dataclasses.asdict(cfg))\n      2. dataset.load_split + dataset.check_split (dừng nếu vi phạm S1-S6)\n      3. dựng train/val loader (test loader chỉ tạo khi cfg.save_test_predictions)\n      4. model.build_model, criterion (losses.build_criterion), optimizer, scheduler, scaler, EMA\n      5. với mỗi epoch: train_one_epoch -> evaluate(val) -> ghi history (loss, macro-F1 val, lr...)\n         và lưu checkpoint tốt nhất theo MACRO-F1 VAL (hòa thì lấy epoch sớm hơn)\n      6. cuối: nạp checkpoint tốt nhất, lưu val logits và eval.save_predictions(pred_path(cfg, \"val\"), ...)\n      7. NẾU cfg.save_test_predictions (chỉ ở Bước 4): đánh giá test đúng MỘT lần,\n         lưu logits và eval.save_predictions(pred_path(cfg, \"test\"), ...)\n      8. ghi history.csv, plot_curves(...), trả về dict tóm tắt\n         (best_epoch, macro-F1 val, thời gian train mỗi epoch, số tham số, GMAC)\n    Quy tắc: KHÔNG dùng test để chọn checkpoint hay bất kỳ quyết định nào (README.md, S4).\n    \"\"\"\n    import torch\n\n    import dataset as ds\n    from model import build_model, count_gmacs_with_tool, count_params\n\n    if cfg.debug_samples_per_class is not None:\n        if cfg.debug_samples_per_class < 1:\n            raise ValueError(\"debug_samples_per_class must be positive\")\n        if cfg.save_test_predictions:\n            raise ValueError(\"debug runs cannot evaluate test\")\n\n    ev = _import_eval()\n    rdir = run_dir(cfg)\n    rdir.mkdir(parents=True, exist_ok=True)\n    (rdir / \"config.json\").write_text(json.dumps(dataclasses.asdict(cfg), indent=2, ensure_ascii=False),\n                                      encoding=\"utf-8\")\n\n    set_seed(cfg.seed, cfg.deterministic)\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    print(f\"\\n=== {cfg.exp_id} | seed {cfg.seed} | {cfg.backbone} | init={cfg.init} | \"\n          f\"loss={cfg.loss} | aug={cfg.aug} | mix={cfg.mix} | sampler={cfg.sampler} ===\")\n    print(f\"[run] device={device}\"\n          + (f\" ({torch.cuda.get_device_name(0)})\" if device.type == \"cuda\" else \" (KHÔNG có GPU!)\"))\n\n    # --- 2. dữ liệu ---\n    train_df, val_df, test_df = ds.load_split(cfg.labels_dir, fold=cfg.fold)\n    if cfg.check_split:\n        cache_json = Path(cfg.out_dir) / f\"split_check_fold{cfg.fold}.json\"\n        if cache_json.exists():\n            print(f\"[run] dùng kết quả kiểm tra chia đã lưu: {cache_json}\")\n        else:\n            report = ds.check_split(train_df, val_df, test_df, cfg.images_dir)\n            report.pop(\"class_names\", None)\n            cache_json.parent.mkdir(parents=True, exist_ok=True)\n            cache_json.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding=\"utf-8\")\n\n    train_tf = ds.build_transforms(train=True, img_size=cfg.img_size, aug=cfg.aug)\n    val_tf = ds.build_transforms(train=False, img_size=cfg.img_size)\n\n    if cfg.debug_samples_per_class is not None:\n        n = cfg.debug_samples_per_class\n        train_df = train_df.groupby(\"Label\", group_keys=False).head(n).reset_index(drop=True)\n        val_df = val_df.groupby(\"Label\", group_keys=False).head(n).reset_index(drop=True)\n        print(f\"[DEBUG ONLY] train={len(train_df)}, val={len(val_df)}; excluded from results.xlsx\")\n\n    train_loader = ds.make_loader(train_df, cfg.images_dir, train_tf, cfg.batch_size, train=True,\n                                  sampler=cfg.sampler, num_workers=cfg.num_workers,\n                                  cache_in_ram=cfg.cache_images, seed=cfg.seed, img_size=cfg.img_size,\n                                  pin_memory=cfg.pin_memory)\n    val_loader = ds.make_loader(val_df, cfg.images_dir, val_tf, cfg.batch_size, train=False,\n                                num_workers=cfg.num_workers, cache_in_ram=cfg.cache_images,\n                                seed=cfg.seed, img_size=cfg.img_size, pin_memory=cfg.pin_memory)\n    test_loader = None\n    if cfg.save_test_predictions:\n        # chỉ tạo ở Bước 4; không dùng test_loader cho bất kỳ quyết định nào (S4)\n        test_loader = ds.make_loader(test_df, cfg.images_dir, val_tf, cfg.batch_size, train=False,\n                                     num_workers=cfg.num_workers, cache_in_ram=cfg.cache_images,\n                                     seed=cfg.seed, img_size=cfg.img_size, pin_memory=cfg.pin_memory)\n\n    # --- 3. model và tối ưu ---\n    model = build_model(cfg.backbone, pretrained=True, num_classes=ds.NUM_CLASSES,\n                        drop_rate=cfg.drop_rate, init=cfg.init).to(device)\n    params_m = count_params(model)\n    gmacs, gmac_tool = count_gmacs_with_tool(model, cfg.img_size)\n    print(f\"[run] tag trọng số = {model.pretrained_tag} | params {params_m:.2f}M | \"\n          f\"{gmacs:.2f} GMAC ({gmac_tool})\")\n\n    criterion = _build_criterion(cfg, train_df).to(device)\n    optimizer = build_optimizer(model, cfg)\n    steps_per_epoch = len(train_loader)\n    scheduler = build_scheduler(optimizer, cfg, steps_per_epoch)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=_amp_on(cfg, device))\n    ema = EMA(model, cfg.ema_decay) if cfg.ema_decay else None\n\n    # --- 4. resume ---\n    start_epoch, best_f1, best_epoch = 1, -1.0, -1\n    last_pt = rdir / \"last.pt\"\n    if cfg.resume and last_pt.exists():\n        ck = torch.load(last_pt, map_location=device, weights_only=False)\n        model.load_state_dict(ck[\"model\"])\n        if ema is not None and \"ema\" in ck:\n            ema.module.load_state_dict(ck[\"ema\"])\n        optimizer.load_state_dict(ck[\"optimizer\"])\n        scheduler.load_state_dict(ck[\"scheduler\"])\n        scaler.load_state_dict(ck[\"scaler\"])\n        start_epoch = ck[\"epoch\"] + 1\n        best_f1, best_epoch = ck[\"best_macro_f1\"], ck[\"best_epoch\"]\n        print(f\"[run] resume từ {last_pt}: bắt đầu lại epoch {start_epoch}, best macro-F1 {best_f1:.4f}\")\n\n    history: list[dict] = []\n    lr_trace: list = []\n    if (rdir / \"history.csv\").exists() and cfg.resume:\n        import pandas as pd\n        history = pd.read_csv(rdir / \"history.csv\").to_dict(\"records\")\n        print(f\"[run] đã đọc {len(history)} epoch trong history.csv\")\n\n    warmup_steps = int(round(cfg.warmup_epochs * steps_per_epoch))\n    for epoch in range(start_epoch, cfg.epochs + 1):\n        tr = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler,\n                             cfg, device, ema=ema, lr_trace=lr_trace)\n        names, y_val, logits_val, val_loss = evaluate(model, val_loader, criterion, device)\n        m = metrics_from_logits(y_val, logits_val)\n        row = {\n            \"epoch\": epoch, \"train_loss\": tr[\"train_loss\"], \"train_top1\": tr[\"train_top1\"],\n            \"val_loss\": val_loss, \"val_top1\": m[\"top1\"], \"val_macro_f1\": m[\"macro_f1\"],\n            \"val_balanced_acc\": m[\"balanced_acc\"], \"val_ece\": m[\"ece\"], \"val_nll\": m[\"nll\"],\n            \"lr\": tr[\"lr\"], \"seconds\": tr[\"seconds\"], \"warmup_steps\": warmup_steps,\n        }\n        history.append(row)\n        star = \"\"\n        if m[\"macro_f1\"] > best_f1:          # \">\" nghĩa là hòa thì lấy epoch sớm hơn\n            best_f1, best_epoch = m[\"macro_f1\"], epoch\n            save_checkpoint(rdir / \"best.pt\", model, ema, epoch, best_f1, best_epoch)\n            star = \"  <- best\"\n        print(f\"[{cfg.exp_id} s{cfg.seed}] ep {epoch:>2}/{cfg.epochs} \"\n              f\"train_loss {tr['train_loss']:.4f} val_loss {val_loss:.4f} \"\n              f\"val_macroF1 {m['macro_f1']:.4f} val_top1 {m['top1']:.4f} \"\n              f\"ece {m['ece']:.4f} lr {tr['lr']:.2e} {tr['seconds']:.1f}s{star}\")\n\n        save_checkpoint(rdir / \"last.pt\", model, ema, epoch, best_f1, best_epoch,\n                        extra={\"optimizer\": optimizer.state_dict(),\n                               \"scheduler\": scheduler.state_dict(), \"scaler\": scaler.state_dict()})\n        _write_history(rdir / \"history.csv\", history)\n\n    # --- 5. nạp lại checkpoint tốt nhất ---\n    model, best_epoch, best_f1 = load_state(cfg, device=device, prefer_ema=False)\n    names, y_val, logits_val, _ = evaluate(model, val_loader, criterion, device)\n    save_logits(logit_path(cfg, \"val\"), names, logits_val, y_val)\n    val_m = metrics_from_logits(y_val, logits_val)\n    if cfg.save_val_predictions:\n        ev.save_predictions(pred_path(cfg, \"val\"), names, y_val, softmax_np(logits_val))\n\n    summary = {\n        \"exp_id\": cfg.exp_id, \"seed\": cfg.seed, \"backbone\": cfg.backbone,\n        \"debug_run\": cfg.debug_samples_per_class is not None,\n        \"init\": cfg.init, \"aug\": cfg.aug, \"img_size\": cfg.img_size, \"epochs\": cfg.epochs,\n        \"batch_size\": cfg.batch_size, \"loss\": cfg.loss, \"mix\": cfg.mix, \"sampler\": cfg.sampler,\n        \"lr_backbone\": cfg.lr_backbone, \"lr_head\": cfg.lr_head, \"weight_decay\": cfg.weight_decay,\n        \"ema_decay\": cfg.ema_decay, \"pretrained_tag\": model.pretrained_tag,\n        \"params_m\": round(params_m, 3), \"gmacs\": round(gmacs, 3), \"gmac_tool\": gmac_tool,\n        \"best_epoch\": best_epoch,\n        \"val_macro_f1\": float(val_m[\"macro_f1\"]), \"val_top1\": float(val_m[\"top1\"]),\n        \"val_balanced_acc\": float(val_m[\"balanced_acc\"]), \"val_ece\": float(val_m[\"ece\"]),\n        \"val_recall\": [float(x) for x in val_m[\"recall\"]],\n        \"sec_per_epoch\": float(np.mean([h[\"seconds\"] for h in history])) if history else None,\n        \"total_sec\": float(sum(h[\"seconds\"] for h in history)) if history else None,\n        \"gpu\": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n        \"torch\": torch.__version__,\n    }\n\n    # --- 6. test: đúng MỘT lần, chỉ khi bật cờ ở Bước 4 ---\n    if cfg.save_test_predictions and test_loader is not None:\n        names_t, y_test, logits_test, _ = evaluate(model, test_loader, criterion, device)\n        save_logits(logit_path(cfg, \"test\"), names_t, logits_test, y_test)\n        ev.save_predictions(pred_path(cfg, \"test\"), names_t, y_test, softmax_np(logits_test))\n        mt = metrics_from_logits(y_test, logits_test)\n        summary.update({\"test_macro_f1\": float(mt[\"macro_f1\"]), \"test_top1\": float(mt[\"top1\"]),\n                        \"test_balanced_acc\": float(mt[\"balanced_acc\"]), \"test_ece\": float(mt[\"ece\"]),\n                        \"test_recall\": [float(x) for x in mt[\"recall\"]]})\n        print(f\"[{cfg.exp_id} s{cfg.seed}] TEST (một lần): top1 {mt['top1']:.4f} \"\n              f\"macroF1 {mt['macro_f1']:.4f} ece {mt['ece']:.4f}\")\n\n    _write_history(rdir / \"history.csv\", history)\n    plot_curves(history, Path(cfg.curves_dir) / f\"{cfg.exp_id}_seed{cfg.seed}.png\",\n                title=f\"{cfg.exp_id} | {cfg.backbone} (tag={model.pretrained_tag}) | seed={cfg.seed} \"\n                      f\"| aug={cfg.aug} loss={cfg.loss} mix={cfg.mix}\",\n                lr_trace=lr_trace)\n    (rdir / \"summary.json\").write_text(json.dumps(summary, indent=2, ensure_ascii=False),\n                                       encoding=\"utf-8\")\n    if last_pt.exists() and not cfg.resume:\n        last_pt.unlink()                      # optimizer state không cần cho suy luận\n    return summary\n\n\ndef _write_history(path: Path, history: list[dict]) -> None:\n    \"\"\"Ghi history.csv (mỗi epoch một dòng, không kèm lr_trace dài).\"\"\"\n    import pandas as pd\n\n    pd.DataFrame(history).to_csv(path, index=False)\n\n\n# --------------------------------------------------------------------------- #\n# 7. CLI\n# --------------------------------------------------------------------------- #\ndef _coerce(value: str, field_type: str):\n    \"\"\"Ép kiểu giá trị string theo kiểu của field trong Config.\"\"\"\n    if value.lower() in (\"none\", \"null\", \"\"):\n        return None\n    t = field_type\n    try:\n        if \"bool\" in t:\n            return value.lower() in (\"1\", \"true\", \"yes\", \"y\", \"on\")\n        if t.startswith(\"int\") or t == \"int\":\n            return int(float(value))\n        if t.startswith(\"float\") or t == \"float\":\n            return float(value)\n        return value\n    except ValueError as e:\n        raise ValueError(f\"không ép được '{value}' sang {t}\") from e\n\n\ndef parse_overrides(pairs: list[str]) -> dict:\n    \"\"\"Biến ['seed=1', 'loss=focal', 'ema_decay=none'] thành dict, ép kiểu theo field của Config.\n\n    TODO đã làm: tách key/value (theo dấu `=` đầu tiên), báo lỗi rõ nếu key không có trong\n    Config, ép int/float/bool/None theo kiểu field. `none` -> None (dùng cho `sampler`,\n    `mix`, `ema_decay`, `class_weight_beta`).\n    \"\"\"\n    types = {f.name: str(f.type) for f in dataclasses.fields(Config)}\n    out: dict = {}\n    for pair in pairs:\n        if \"=\" not in pair:\n            raise ValueError(f\"override '{pair}' phải có dạng KEY=VALUE\")\n        key, value = pair.split(\"=\", 1)\n        key, value = key.strip(), value.strip()\n        if key not in types:\n            raise ValueError(f\"'{key}' không phải field của Config. \"\n                             f\"Các field hợp lệ: {', '.join(sorted(types))}\")\n        out[key] = _coerce(value, types[key])\n    return out\n\n\ndef main() -> None:\n    \"\"\"Điểm vào dòng lệnh: `python train.py --set exp_id=B01 backbone=resnet50 seed=0`.\n\n    TODO đã làm: argparse nhận `--set KEY=VALUE ...`, dựng Config qua parse_overrides,\n    gọi run(cfg), in kết quả tóm tắt dạng JSON.\n    \"\"\"\n    ap = argparse.ArgumentParser(description=\"Huấn luyện một cấu hình của lab Day 2\")\n    ap.add_argument(\"--set\", nargs=\"*\", default=[], metavar=\"KEY=VALUE\",\n                    help=\"ghi đè field của Config, ví dụ: --set exp_id=B01 backbone=resnet50 seed=0\")\n    args = ap.parse_args()\n\n    cfg = Config(**parse_overrides(args.set))\n    summary = run(cfg)\n    print(json.dumps(summary, indent=2, ensure_ascii=False))\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "code/inference.py": "\"\"\"inference.py - các phương pháp suy luận (Bước 3 của GUIDE.md).\n\nHoàn thiện từ bộ khung `starter/`. Giữ nguyên tên hàm và kiểu dữ liệu vào/ra như docstring gốc.\nLiên hệ slide Day 2: TTA (trang 62-66, 75), ensemble/EMA/soup (trang 67), độ phân giải kiểm tra\n(trang 68), temperature scaling (trang 69), gộp BatchNorm (trang 71).\n\nMọi hàm chạy ở chế độ eval, không gradient. Chọn phương pháp CHỈ dựa trên val; nhiệt độ T khớp\ntrên VAL rồi áp dụng sang test (README.md, S2 và S4).\n\n    predict_logits(model, loader, device, view=None) -> (filenames, y_true, logits[N, 9])\n    aggregate_views(list_of_logits, space)           -> probs[N, 9]\n    fit_temperature(val_logits, val_labels)          -> float T\n    apply_temperature(logits, T)                     -> probs\n    ensemble_probs(list_of_probs)                    -> probs\n    fuse_conv_bn(model)                              -> model (BN đã gộp vào conv)\n\nLựa chọn đáng ghi lại:\n  - `predict_logits` chạy FP32. Xác suất đầu ra dùng cho ECE, temperature scaling và file\n    predictions nên cần ổn định; độ trễ thì đo riêng trong benchmark.py (có dtype riêng).\n  - `aggregate_views` hỗ trợ cả \"prob\" và \"logit\"; bài lab cần so sánh cả hai (I03) chứ không\n    mặc định chọn một.\n  - `fit_temperature` dùng tìm kiếm trên `log T` (lưới thô + golden-section) thay vì LBFGS:\n    tất định, không phụ thuộc optimizer, và cho T tối ưu trong khoảng [0.05, 20].\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\n\n\n# --------------------------------------------------------------------------- #\n# 1. Lấy logit từ model\n# --------------------------------------------------------------------------- #\ndef predict_logits(model, loader, device, view=None):\n    \"\"\"Chạy model trên loader và gom logit theo đúng thứ tự file.\n\n    `view` là hàm biến đổi batch ảnh trước khi đưa vào model (ví dụ lật ngang), hoặc None.\n    TODO đã làm: model.eval(), torch.inference_mode(), trả về numpy.\n    \"\"\"\n    import torch\n\n    model.eval()\n    filenames: list[str] = []\n    ys, out = [], []\n    with torch.inference_mode():\n        for images, labels, names in loader:\n            images = images.to(device, non_blocking=True)\n            if view is not None:\n                images = view(images)\n            logits = model(images)\n            out.append(logits.float().cpu().numpy())\n            ys.append(labels.cpu().numpy())\n            filenames.extend(names)\n    if not out:\n        return [], np.zeros(0, dtype=np.int64), np.zeros((0, 9), dtype=np.float32)\n    return filenames, np.concatenate(ys), np.concatenate(out)\n\n\ndef view_identity(x):\n    return x\n\n\ndef view_hflip(x):\n    \"\"\"Lật ngang batch (N, C, H, W). TODO đã làm: torch.flip trên chiều rộng (slide trang 75).\"\"\"\n    import torch\n\n    return torch.flip(x, dims=[-1])\n\n\ndef view_vflip(x):\n    \"\"\"Lật dọc. Chỉ dùng để kiểm tra: ảnh cỏ dại có đối xứng dọc hay không.\"\"\"\n    import torch\n\n    return torch.flip(x, dims=[-2])\n\n\ndef views_multicrop(x, crop: int):\n    \"\"\"5 crop (4 góc + giữa) kích thước `crop`, và tuỳ chọn thêm bản lật. Trả về list các batch.\n\n    TODO đã làm: cắt từ ảnh đã tensor (N, C, H, W) với H, W >= crop; trả về [c1..c5].\n    Hàm KHÔNG tự thêm bản lật: ghép TTA do `tta_logits` điều khiển nên tách bạch \"view nào\"\n    với \"lật hay không\" (I01 = chỉ lật).\n    \"\"\"\n    import torch\n    import torch.nn.functional as F\n\n    n, c, h, w = x.shape\n    if h < crop or w < crop:\n        raise ValueError(f\"ảnh {h}x{w} nhỏ hơn crop={crop}\")\n    out = []\n    for top, left in [(0, 0), (0, w - crop), (h - crop, 0), (h - crop, w - crop)]:\n        out.append(x[:, :, top:top + crop, left:left + crop])\n    top, left = (h - crop) // 2, (w - crop) // 2\n    out.append(x[:, :, top:top + crop, left:left + crop])\n    return out\n\n\ndef views_multiscale(x, sizes):\n    \"\"\"Resize batch về từng kích thước trong `sizes`, trả về list các batch. TODO đã làm.\n\n    Lưu ý: model phải chấp nhận ảnh khác kích thước lúc train (CNN có global pooling thì được;\n    ViT/Swin cần xử lý riêng vị trí/cửa sổ). Ghi rõ hạn chế gặp:\n      - CNN của timm (ResNet, ConvNeXt, EfficientNet, MobileNetV3) nhận mọi kích thước chia hết\n        cho 32 (global average pool), nên dò độ phân giải 224/256/288/320 chạy được.\n      - DeiT/ViT có `pos_embed` cố định 224x224: truyền ảnh khác kích thước sẽ ra lỗi hình dạng,\n        phải nội suy `pos_embed` trước. Hàm này KHÔNG tự xử lý; nếu backbone không chịu được\n        kích thước lạ thì hãy bỏ nhánh đó khỏi thí nghiệm I04 và ghi rõ trong báo cáo.\n      - Swin yêu cầu kích thước cố định theo lưới cửa sổ; xem `dynamic_img_size` của timm.\n    \"\"\"\n    import torch.nn.functional as F\n\n    return [F.interpolate(x, size=(s, s), mode=\"bilinear\", align_corners=False) for s in sizes]\n\n\n# --------------------------------------------------------------------------- #\n# 2. Gộp nhiều lượt chạy\n# --------------------------------------------------------------------------- #\ndef _softmax_np(logits: np.ndarray) -> np.ndarray:\n    z = np.asarray(logits, dtype=np.float64)\n    z = z - z.max(axis=1, keepdims=True)\n    e = np.exp(z)\n    return e / e.sum(axis=1, keepdims=True)\n\n\ndef aggregate_views(logits_per_view, space: str = \"prob\"):\n    \"\"\"Gộp K lượt chạy của TTA thành một dự đoán (slide trang 62).\n\n      - space=\"prob\":  trung bình softmax của từng view\n      - space=\"logit\": trung bình logit rồi softmax\n    TODO đã làm: trả về xác suất (N, 9) đã chuẩn hoá.\n\n    Cả hai cách đều cho xác suất cộng lại bằng 1 theo từng dòng (đây là điều eval.py kiểm tra).\n    \"\"\"\n    if space not in (\"prob\", \"logit\"):\n        raise ValueError(f\"space={space!r} không hợp lệ, chọn 'prob' hoặc 'logit'\")\n    views = [np.asarray(v, dtype=np.float64) for v in logits_per_view]\n    if not views:\n        raise ValueError(\"cần ít nhất một view\")\n    shapes = {v.shape for v in views}\n    if len(shapes) != 1:\n        raise ValueError(f\"các view phải cùng dạng, nhận {shapes}\")\n    if space == \"prob\":\n        probs = np.mean([_softmax_np(v) for v in views], axis=0)\n        return probs / probs.sum(axis=1, keepdims=True)\n    return _softmax_np(np.mean(views, axis=0))\n\n\ndef ensemble_probs(list_of_probs):\n    \"\"\"Trung bình xác suất của nhiều mô hình (khác backbone hoặc khác seed). TODO đã làm.\n\n    Chi phí suy luận = số mô hình. Chỉ ghép các mô hình trên CÙNG tập ảnh và cùng thứ tự file.\n    \"\"\"\n    probs = [np.asarray(p, dtype=np.float64) for p in list_of_probs]\n    if not probs:\n        raise ValueError(\"cần ít nhất một mô hình\")\n    shapes = {p.shape for p in probs}\n    if len(shapes) != 1:\n        raise ValueError(f\"các mô hình phải cùng dạng, nhận {shapes}\")\n    avg = np.mean(probs, axis=0)\n    return avg / avg.sum(axis=1, keepdims=True)\n\n\ndef tta_logits(model, loader, device, view_list=None, space: str = \"prob\"):\n    \"\"\"Chạy nhiều view rồi gộp. `view_list` là list hàm biến đổi batch (mặc định: chỉ 1 view).\n\n    Trả về (filenames, y_true, probs). Với mỗi view, thứ tự tên file phải khớp; hàm này kiểm tra.\n    \"\"\"\n    if view_list is None:\n        view_list = [view_identity]\n    per_view, ref_names, y_true = [], None, None\n    for v in view_list:\n        names, y, logits = predict_logits(model, loader, device, view=v)\n        if ref_names is None:\n            ref_names, y_true = names, y\n        elif names != ref_names:\n            raise ValueError(\"các view phải cho cùng thứ tự tên file\")\n        per_view.append(logits)\n    return ref_names, y_true, aggregate_views(per_view, space=space)\n\n\n# --------------------------------------------------------------------------- #\n# 3. Hiệu chuẩn\n# --------------------------------------------------------------------------- #\ndef fit_temperature(val_logits, val_labels) -> float:\n    \"\"\"Tìm nhiệt độ T > 0 cực tiểu NLL trên VAL: p = softmax(logit / T)  (slide trang 69).\n\n    TODO đã làm: tìm lưới thô trên `log T` (61 điểm trong [0.05, 20]) rồi TINH bằng\n    golden-section search trên `log T` quanh điểm tốt nhất. Cách này không cần optimizer,\n    cho kết quả tất định (chạy lại ra cùng T) và tránh phụ thuộc hành vi của LBFGS.\n    Accuracy không đổi vì thứ tự lớp không đổi. KHÔNG khớp T trên test.\n    \"\"\"\n    import torch\n    import torch.nn.functional as F\n\n    z = torch.as_tensor(np.asarray(val_logits), dtype=torch.float32)\n    y = torch.as_tensor(np.asarray(val_labels), dtype=torch.int64)\n    if z.shape[0] != y.shape[0]:\n        raise ValueError(f\"số mẫu lệch: {z.shape[0]} logit vs {y.shape[0]} nhãn\")\n    if z.shape[0] == 0:\n        raise ValueError(\"không có mẫu nào để khớp T\")\n\n    def nll(log_t: float) -> float:\n        return float(F.cross_entropy(z / float(np.exp(log_t)), y))\n\n    lo, hi = float(np.log(0.05)), float(np.log(20.0))\n    grid = np.linspace(lo, hi, 61)\n    vals = [nll(g) for g in grid]\n    i = int(np.argmin(vals))\n    a = float(grid[max(0, i - 1)])\n    b = float(grid[min(len(grid) - 1, i + 1)])\n\n    phi = (np.sqrt(5.0) - 1.0) / 2.0\n    c, d = b - phi * (b - a), a + phi * (b - a)\n    fc, fd = nll(c), nll(d)\n    for _ in range(60):                       # hội tụ trên log T -> ổn định về cả hai phía\n        if fc < fd:\n            b, d, fd = d, c, fc\n            c = b - phi * (b - a)\n            fc = nll(c)\n        else:\n            a, c, fc = c, d, fd\n            d = a + phi * (b - a)\n            fd = nll(d)\n    return float(np.exp((a + b) / 2.0))\n\n\ndef apply_temperature(logits, T: float):\n    \"\"\"Trả về softmax(logits / T). TODO đã làm.\"\"\"\n    if T <= 0:\n        raise ValueError(f\"T phải > 0, nhận {T}\")\n    return _softmax_np(np.asarray(logits, dtype=np.float64) / float(T))\n\n\ndef ece_of(probs, y_true, bins: int = 15) -> float:\n    \"\"\"ECE dùng đúng định nghĩa của eval.py (tránh tự viết lại khác đi một chút).\"\"\"\n    import sys\n    from pathlib import Path\n\n    here = Path(__file__).resolve().parent\n    for cand in (here.parent / \"eval.py\", here / \"eval.py\", Path.cwd() / \"eval.py\"):\n        if cand.exists() and str(cand.parent) not in sys.path:\n            sys.path.insert(0, str(cand.parent))\n            break\n    import eval as ev\n\n    return ev.ece_score(np.asarray(probs), np.asarray(y_true), bins)\n\n\n# --------------------------------------------------------------------------- #\n# 4. Gộp BatchNorm\n# --------------------------------------------------------------------------- #\ndef fuse_conv_bn(model):\n    \"\"\"Gộp BatchNorm vào tích chập liền trước, chính xác lúc suy luận (slide trang 71, 75):\n\n        w' = gamma * w / sqrt(var + eps)        b' = beta + gamma * (b - mean) / sqrt(var + eps)\n\n    TODO đã làm:\n      - model.eval() trước; với mỗi cặp (Conv2d, BatchNorm2d) LIỀN KỀ trong cùng một module cha:\n        tạo conv mới (có bias) và thay BN bằng Identity\n      - in ra sai số lớn nhất giữa đầu ra trước/sau gộp (xem `fuse_conv_bn_checked`)\n\n    Quét `children()` của từng module nên chỉ gộp đúng cặp liền kề, không gộp nhầm ở nhánh\n    residual (BasicBlock của ResNet không có conv+bn liền kề qua nhánh cộng).\n    Với kiến trúc không có BN (ViT, Swin, ConvNeXt dùng LayerNorm) hàm không đổi gì;\n    `n_fused` trả về 0 là tín hiệu \"không áp dụng\".\n    \"\"\"\n    import torch.nn as nn\n\n    model.eval()\n    n_fused = 0\n    for parent in model.modules():\n        names = list(dict(parent.named_children()).keys())\n        for i in range(len(names) - 1):\n            conv, bn = getattr(parent, names[i]), getattr(parent, names[i + 1])\n            if not (isinstance(conv, nn.Conv2d) and isinstance(bn, nn.modules.batchnorm._BatchNorm)):\n                continue\n            if not bn.track_running_stats:      # không có running stats thì không gộp được\n                continue\n            fused = fuse_conv_bn_pair(conv, bn)\n            setattr(parent, names[i], fused)\n            setattr(parent, names[i + 1], nn.Identity())\n            n_fused += 1\n    print(f\"[inference] đã gộp {n_fused} cặp Conv+BN\")\n    return model\n\n\ndef fuse_conv_bn_pair(conv, bn):\n    \"\"\"Tạo Conv2d mới đã hấp thụ BN (theo đúng công thức trên).\"\"\"\n    import torch\n    import torch.nn as nn\n\n    fused = nn.Conv2d(conv.in_channels, conv.out_channels, conv.kernel_size,\n                      stride=conv.stride, padding=conv.padding, dilation=conv.dilation,\n                      groups=conv.groups, bias=True,\n                      padding_mode=conv.padding_mode, device=conv.weight.device,\n                      dtype=conv.weight.dtype)\n    std = (bn.running_var + bn.eps).sqrt()\n    t = (bn.weight / std).view(-1, 1, 1, 1)\n    with torch.no_grad():\n        fused.weight.copy_(conv.weight * t)\n        b = conv.bias if conv.bias is not None else torch.zeros_like(bn.running_mean)\n        fused.bias.copy_(bn.bias + (b - bn.running_mean) * bn.weight / std)\n    return fused\n\n\ndef fuse_conv_bn_checked(model, img_size: int = 224, device=None):\n    \"\"\"Gộp BN và trả về (model_đã_gộp, sai_số_lớn_nhất) so với model gốc.\n\n    Kiểm tra bắt buộc của rubric H: đầu ra trước/sau gộp phải lệch nhau cỡ 1e-5 trở xuống.\n    Kiểm tra trên đầu vào ngẫu nhiên đúng kích thước lúc đánh giá, không dùng dữ liệu test.\n    \"\"\"\n    import copy\n\n    import torch\n\n    from inference import fuse_conv_bn\n\n    device = device or (\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    ref = copy.deepcopy(model).eval().to(device)\n    fused = fuse_conv_bn(copy.deepcopy(model).eval().to(device))\n    x = torch.randn(2, 3, img_size, img_size, device=device)\n    a, b = ref(x).float(), fused(x).float()\n    err = float((a - b).abs().max().item())\n    rel = err / max(1e-12, float(a.abs().max().item()))\n    print(f\"[inference] kiểm tra gộp BN: sai số tuyệt đối lớn nhất = {err:.3e} \"\n          f\"(tương đối {rel:.3e})\")\n    return fused, err\n",
  "code/benchmark.py": "\"\"\"benchmark.py - đo độ trễ suy luận đúng cách (slide Day 2, trang 73 và 75; GUIDE.md mục 4.1).\n\nHoàn thiện từ bộ khung `starter/`. Giữ nguyên tên hàm và kiểu dữ liệu vào/ra như docstring gốc.\n\nQuy tắc đo (vi phạm bị trừ điểm, RUBRIC mục 3):\n  - warmup: bỏ >= 10 lần chạy đầu\n  - đồng bộ GPU: torch.cuda.synchronize() (hoặc CUDA event) TRƯỚC và SAU đoạn cần đo\n  - >= 50 lần đo, báo cáo p50, p95, p99 (không chỉ trung bình)\n  - ghi rõ GPU, dtype (FP32/AMP/FP16), batch, độ phân giải, có/không gộp BN, phiên bản torch\n  - chọn và ghi rõ có tính tiền xử lý hay không\n\nLựa chọn đo: `latency_report` đo **chỉ phần forward** của model trên tensor đầu vào đã nằm\nsẵn trên GPU (`includes_preprocess=False`), vì tiền xử lý (resize/normalize) giống nhau ở mọi\nphương pháp nên tách ra sẽ dễ so sánh hơn. Nếu muốn tính cả tiền xử lý, truyền\n`preprocess=<callable>` và báo cáo sẽ ghi `includes_preprocess=True`.\n\"\"\"\nfrom __future__ import annotations\n\nimport time\n\nimport numpy as np\n\n\ndef bench(fn, warmup: int = 10, iters: int = 100, sync=None) -> dict:\n    \"\"\"Đo thời gian một hàm `fn()` (không tham số), trả về mili-giây.\n\n    `sync` là hàm đồng bộ (ví dụ torch.cuda.synchronize) hoặc None trên CPU.\n\n    TODO đã làm:\n      - chạy warmup lần đầu rồi bỏ\n      - với mỗi lần đo: sync(); t0 = time.perf_counter(); fn(); sync(); lấy hiệu * 1000\n      - trả về {\"p50\": ..., \"p95\": ..., \"p99\": ..., \"mean\": ..., \"n\": iters}\n    Gợi ý: dùng numpy.percentile. Ngoài ra trả kèm `min` và `max` để thấy đuôi phân phối.\n    \"\"\"\n    if warmup < 10:\n        print(f\"[bench] CẢNH BÁO: chỉ warmup {warmup} lần; GUIDE.md yêu cầu >= 10\")\n    if iters < 50:\n        print(f\"[bench] CẢNH BÁO: chỉ {iters} lần đo; GUIDE.md yêu cầu >= 50\")\n\n    for _ in range(warmup):\n        fn()\n    if sync is not None:\n        sync()\n\n    times = np.empty(iters, dtype=np.float64)\n    for i in range(iters):\n        if sync is not None:\n            sync()\n        t0 = time.perf_counter()\n        fn()\n        if sync is not None:\n            sync()\n        times[i] = (time.perf_counter() - t0) * 1000.0\n    return {\n        \"p50\": float(np.percentile(times, 50)),\n        \"p95\": float(np.percentile(times, 95)),\n        \"p99\": float(np.percentile(times, 99)),\n        \"mean\": float(times.mean()),\n        \"min\": float(times.min()),\n        \"max\": float(times.max()),\n        \"n\": int(iters),\n        \"warmup\": int(warmup),\n    }\n\n\ndef latency_report(model, batch_size: int, img_size: int, dtype: str = \"fp32\", device: str = \"cuda\",\n                   warmup: int = 10, iters: int = 100, preprocess=None,\n                   channels_last: bool = False) -> dict:\n    \"\"\"Đo độ trễ forward của `model` với đầu vào ngẫu nhiên (batch_size, 3, img_size, img_size).\n\n    Trả về dict có thể ghi thẳng vào sheet `Latency` của results.xlsx:\n        {\"gpu\": ..., \"dtype\": ..., \"batch\": ..., \"img_size\": ..., \"p50\": ..., \"p95\": ..., \"p99\": ...,\n         \"images_per_s\": batch_size / (p50 / 1000), \"torch\": torch.__version__}\n\n    TODO đã làm:\n      - model.eval(), torch.inference_mode()\n      - dtype: \"fp32\" | \"amp\" (autocast fp16) | \"fp16\" (model.half())\n      - gọi bench(...) với sync phù hợp; lấy tên GPU bằng torch.cuda.get_device_name\n      - Nhớ: ở batch 1, AMP có thể CHẬM hơn FP32 (slide trang 73): đo thật, đừng giả định\n\n    `channels_last=True` đặt tensor theo layout NHWC, thường nhanh hơn với tensor core fp16.\n    \"\"\"\n    import torch\n\n    if device.startswith(\"cuda\") and not torch.cuda.is_available():\n        raise RuntimeError(\"Yêu cầu đo trên CUDA nhưng máy không có GPU\")\n    dev = torch.device(device)\n    sync = torch.cuda.synchronize if dev.type == \"cuda\" else None\n\n    model = model.to(dev).eval()\n    x = torch.randn(batch_size, 3, img_size, img_size, device=dev)\n\n    autocast_on = False\n    if dtype == \"fp16\":\n        model = model.half()\n        x = x.half()\n    elif dtype == \"amp\":\n        autocast_on = True\n    elif dtype != \"fp32\":\n        raise ValueError(f\"dtype={dtype!r} không hợp lệ, chọn 'fp32', 'amp' hoặc 'fp16'\")\n\n    if channels_last:\n        model = model.to(memory_format=torch.channels_last)\n        x = x.to(memory_format=torch.channels_last)\n\n    def step():\n        with torch.inference_mode(), torch.autocast(\"cuda\", dtype=torch.float16, enabled=autocast_on):\n            model(preprocess(x) if preprocess is not None else x)\n\n    r = bench(step, warmup=warmup, iters=iters, sync=sync)\n    r.update({\n        \"gpu\": torch.cuda.get_device_name(0) if dev.type == \"cuda\" else \"CPU\",\n        \"dtype\": dtype,\n        \"batch\": int(batch_size),\n        \"img_size\": int(img_size),\n        \"channels_last\": bool(channels_last),\n        \"includes_preprocess\": preprocess is not None,\n        \"images_per_s\": batch_size / (r[\"p50\"] / 1000.0),\n        \"torch\": torch.__version__,\n    })\n    print(f\"[bench] {r['gpu']} | {dtype} | batch {batch_size} @ {img_size} | \"\n          f\"p50 {r['p50']:.2f} ms | p95 {r['p95']:.2f} ms | p99 {r['p99']:.2f} ms | \"\n          f\"{r['images_per_s']:.1f} img/s\")\n    return r\n\n\ndef tta_latency(model, k_views: int, **kw) -> dict:\n    \"\"\"Độ trễ của TTA K view: xấp xỉ K lần một lượt chạy (slide trang 63). TODO đã làm.\n\n    Đo thật phần \"chạy K view rồi gộp\" (dùng lật ngang + trung bình xác suất) và so với\n    `k_views * p50` của một lượt chạy đơn, để kiểm chứng nhận định \"chi phí gần tuyến tính\".\n    \"\"\"\n    import torch\n\n    from inference import aggregate_views, view_hflip, view_identity\n\n    device = kw.pop(\"device\", \"cuda\")\n    img_size = kw.pop(\"img_size\", 224)\n    batch_size = kw.pop(\"batch_size\", 1)\n    warmup = kw.pop(\"warmup\", 10)\n    iters = kw.pop(\"iters\", 100)\n    dev = torch.device(device)\n    sync = torch.cuda.synchronize if dev.type == \"cuda\" else None\n\n    model = model.to(dev).eval()\n    x = torch.randn(batch_size, 3, img_size, img_size, device=dev)\n    # K view: view đầu là ảnh gốc, các view sau xen kẽ lật ngang (đủ để đo chi phí K lượt chạy)\n    views = [view_identity if i % 2 == 0 else view_hflip for i in range(k_views)]\n\n    def step():\n        with torch.inference_mode():\n            logits = [model(v(x)) for v in views]\n        aggregate_views([t.detach().cpu().numpy() for t in logits], space=\"prob\")\n\n    r = bench(step, warmup=warmup, iters=iters, sync=sync)\n    single = latency_report(model, batch_size, img_size, dtype=kw.get(\"dtype\", \"fp32\"),\n                            device=device, warmup=warmup, iters=iters)\n    r.update({\n        \"k_views\": int(k_views),\n        \"single_view_p50\": single[\"p50\"],\n        \"k_times_single_p50\": k_views * single[\"p50\"],\n        \"overhead_vs_linear\": r[\"p50\"] - k_views * single[\"p50\"],\n        \"gpu\": single[\"gpu\"], \"dtype\": single[\"dtype\"], \"batch\": batch_size,\n        \"img_size\": img_size, \"torch\": single[\"torch\"],\n    })\n    print(f\"[bench] TTA K={k_views}: p50 {r['p50']:.2f} ms (K x p50 = \"\n          f\"{r['k_times_single_p50']:.2f} ms, chênh {r['overhead_vs_linear']:+.2f} ms)\")\n    return r\n\n\ndef sweep_latency(model, batch_sizes=(1, 32), img_size: int = 224, dtypes=(\"fp32\", \"amp\"),\n                  device: str = \"cuda\", iters: int = 100) -> list[dict]:\n    \"\"\"Quét nhiều tổ hợp batch x dtype để lấp sheet `Latency` của results.xlsx.\n\n    Chạy ở cả hai điều kiện mà GUIDE.md mục 4.1 yêu cầu: batch 1 (giống robot) và một batch\n    lớn hơn để đo thông lượng ảnh/giây.\n    \"\"\"\n    rows = []\n    for bs in batch_sizes:\n        for dt in dtypes:\n            rows.append(latency_report(model, bs, img_size, dtype=dt, device=device, iters=iters))\n    return rows",
  "code/lab_workflow.py": "\"\"\"Runnable notebook workflow built on the original lab modules.\n\nAll selection uses validation. Smoke runs and official runs use separate folders.\nThe exporter reads saved results; missing experiments remain missing.\n\"\"\"\nfrom __future__ import annotations\n\nimport runtime\nimport argparse\nimport dataclasses\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\n\nimport dataset as ds\nimport inference as inf\nfrom benchmark import bench\nfrom train import Config, _import_eval, load_state, run, run_dir, set_seed\n\nMETHODS = (\"single\", \"hflip_prob\", \"fivecrop_prob\", \"hflip_logit\", \"temperature\")\n\n\ndef base_config(root, profile=\"local\"):\n    root = Path(root).resolve()\n    candidates = (root / \"data/images\", root / \"images\")\n    images = next((p for p in candidates if p.is_dir()), candidates[0])\n    if profile not in (\"local\", \"kaggle\"):\n        raise ValueError(\"profile must be local or kaggle\")\n    return Config(images_dir=str(images), labels_dir=str(root / \"data/labels\"),\n                  out_dir=str(root / \"runs\"), pred_dir=str(root / \"predictions\"),\n                  curves_dir=str(root / \"curves\"), logit_dir=str(root / \"logits\"),\n                  batch_size=4 if profile == \"local\" else 32,\n                  num_workers=0 if profile == \"local\" else 2,\n                  cache_images=False, pin_memory=False if profile == \"local\" else None)\n\n\ndef experiment_plan(base, stage):\n    if stage == \"backbones\":\n        names = (\"resnet50\", \"convnext_tiny\", \"deit_small\", \"swin_tiny\", \"mobilenetv3\")\n        return [dataclasses.replace(base, exp_id=f\"B{i:02}\", backbone=name,\n                                    save_test_predictions=False) for i, name in enumerate(names, 1)]\n    if stage == \"training\":\n        variants = [({}, \"T00\"), ({\"init\": \"scratch\"}, \"T01\"),\n                    ({\"init\": \"frozen\"}, \"T02\"), ({\"aug\": \"trivial\"}, \"T03\"),\n                    ({\"aug\": \"color\"}, \"T04\"),\n                    ({\"loss\": \"ls\", \"label_smoothing\": .1}, \"T05\"),\n                    ({\"loss\": \"focal\"}, \"T06\"), ({\"mix\": \"cutmix\"}, \"T07\"),\n                    ({\"mix\": \"mixup\"}, \"T08\")]\n        return [dataclasses.replace(base, exp_id=tag, save_test_predictions=False, **kw)\n                for kw, tag in variants]\n    raise ValueError(\"stage must be backbones or training\")\n\n\ndef run_experiments(configs):\n    rows = []\n    for cfg in configs:\n        if cfg.save_test_predictions or cfg.debug_samples_per_class is not None:\n            raise ValueError(\"selection experiments must use full train/val, without test\")\n        summary = run_dir(cfg) / \"summary.json\"\n        if summary.exists():\n            validate_saved_config(cfg)\n            rows.append(json.loads(summary.read_text(encoding=\"utf-8\")))\n            print(f\"Reuse completed run: {cfg.exp_id}, seed {cfg.seed}\")\n        else:\n            rows.append(run(cfg))\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n    return pd.DataFrame(rows)\n\n\ndef validate_saved_config(cfg):\n    old = json.loads((run_dir(cfg) / \"config.json\").read_text(encoding=\"utf-8\"))\n    current = dataclasses.asdict(cfg)\n    # These only control writing/test access and resuming, not the scientific recipe.\n    for key in (\"save_test_predictions\", \"resume\"):\n        old.pop(key, None)\n        current.pop(key, None)\n    if old != current:\n        raise ValueError(f\"Config changed for {cfg.exp_id}; choose a new exp_id\")\n\n\ndef best_config(configs):\n    scores = []\n    for cfg in configs:\n        p = run_dir(cfg) / \"summary.json\"\n        if not p.exists():\n            raise FileNotFoundError(f\"Complete all comparisons first: {p}\")\n        validate_saved_config(cfg)\n        result = json.loads(p.read_text(encoding=\"utf-8\"))\n        if result.get(\"debug_run\"):\n            raise ValueError(\"Cannot select a debug run\")\n        scores.append(result[\"val_macro_f1\"])\n    return configs[int(np.argmax(scores))]\n\n\ndef _view_logits(model, x, method):\n    if method in (\"single\", \"temperature\"):\n        return [model(x)]\n    if method.startswith(\"hflip\"):\n        return [model(x), model(inf.view_hflip(x))]\n    if method == \"fivecrop_prob\":\n        size = x.shape[-1]\n        crops = inf.views_multicrop(x, max(1, size * 7 // 8))\n        return [model(F.interpolate(c, size=(size, size), mode=\"bilinear\", align_corners=False))\n                for c in crops]\n    raise ValueError(f\"Unknown inference method: {method}\")\n\n\ndef _aggregate(logits, method, temperature):\n    if method == \"hflip_logit\":\n        return torch.stack(logits).mean(0).softmax(-1)\n    return torch.stack([(z / temperature if method == \"temperature\" else z).softmax(-1)\n                        for z in logits]).mean(0)\n\n\ndef predict_method(model, loader, device, method=\"single\", temperature=1., return_logits=False):\n    if method not in METHODS:\n        raise ValueError(f\"Unknown inference method: {method}\")\n    model.eval()\n    if return_logits and method not in (\"single\", \"temperature\"):\n        raise ValueError(\"Raw logits are only available for single-view methods\")\n    names, targets, outputs, raw = [], [], [], []\n    with torch.inference_mode():\n        for images, labels, files in loader:\n            logits = _view_logits(model, images.to(device), method)\n            probs = _aggregate(logits, method, temperature).float().cpu().numpy()\n            if return_logits:\n                raw.append(logits[0].float().cpu().numpy())\n            names.extend(files)\n            targets.append(labels.numpy())\n            outputs.append(probs)\n    if not outputs:\n        raise ValueError(\"Cannot evaluate an empty split\")\n    result = names, np.concatenate(targets), np.concatenate(outputs)\n    return (*result, np.concatenate(raw)) if return_logits else result\n\n\ndef _loader(cfg, frame):\n    return ds.make_loader(frame, cfg.images_dir, ds.build_transforms(False, cfg.img_size),\n                          cfg.batch_size, train=False, num_workers=cfg.num_workers,\n                          pin_memory=cfg.pin_memory, cache_in_ram=False)\n\n\ndef measure_method(model, cfg, device, method, temperature=1., iters=50):\n    \"\"\"Time actual views + aggregation on device, excluding disk I/O and decoding.\"\"\"\n    model.eval()\n    x = torch.randn(1, 3, cfg.img_size, cfg.img_size, device=device)\n    def forward():\n        with torch.inference_mode():\n            logits = _view_logits(model, x, method)\n            return _aggregate(logits, method, temperature)\n    timing = bench(forward, warmup=10, iters=iters,\n                   sync=torch.cuda.synchronize if device.type == \"cuda\" else None)\n    timing.update(gpu=torch.cuda.get_device_name(0) if device.type == \"cuda\" else \"CPU\",\n                  dtype=\"fp32\", batch=1, img_size=cfg.img_size,\n                  images_per_s=1000 / timing[\"p50\"], includes_preprocess=True,\n                  timing_scope=\"tensor views + forward + aggregation; excludes image decoding\")\n    return timing\n\n\ndef compare_inference(cfg):\n    if cfg.debug_samples_per_class is not None:\n        raise ValueError(\"Inference selection cannot use debug data\")\n    validate_saved_config(cfg)\n    ev = _import_eval()\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    model, _, _ = load_state(cfg, device)\n    _, val, _ = ds.load_split(cfg.labels_dir, cfg.fold)\n    loader = _loader(cfg, val)\n    _, y, logits = inf.predict_logits(model, loader, device)\n    temperature = inf.fit_temperature(logits, y)\n    rows = []\n    out = Path(cfg.out_dir) / \"inference\" / cfg.exp_id / f\"seed{cfg.seed}\"\n    out.mkdir(parents=True, exist_ok=True)\n    for index, method in enumerate(METHODS):\n        names, y, probs = predict_method(model, loader, device, method, temperature)\n        m = ev.compute_metrics(y, probs.argmax(1), probs)\n        row = {\"exp_id\": f\"I{index:02}\", \"source_exp_id\": cfg.exp_id,\n               \"seed\": cfg.seed, \"backbone\": cfg.backbone, \"method\": method,\n               \"k_views\": 5 if method == \"fivecrop_prob\" else 2 if method.startswith(\"hflip\") else 1,\n               \"temperature\": temperature if method == \"temperature\" else 1.,\n               \"val_macro_f1\": m[\"macro_f1\"], \"val_top1\": m[\"top1\"], \"val_ece\": m[\"ece\"],\n               \"checkpoint\": str(run_dir(cfg) / \"best.pt\"),\n               \"latency\": measure_method(model, cfg, device, method, temperature)}\n        row[\"relative_cost\"] = row[\"latency\"][\"p50\"] / (rows[0][\"latency\"][\"p50\"] if rows else row[\"latency\"][\"p50\"])\n        ev.save_predictions(out / f\"{row['exp_id']}_seed{cfg.seed}_val.csv\", names, y, probs)\n        (out / f\"{row['exp_id']}.json\").write_text(json.dumps(row, indent=2), encoding=\"utf-8\")\n        rows.append(row)\n        print(method, f\"val macro-F1={m['macro_f1']:.4f}\", f\"p95={row['latency']['p95']:.2f} ms\")\n    return pd.DataFrame([{k: v for k, v in r.items() if k != \"latency\"} for r in rows])\n\n\ndef lock_selection(cfg, method):\n    if method not in METHODS or cfg.debug_samples_per_class is not None:\n        raise ValueError(\"Select an official configuration and supported inference method\")\n    validate_saved_config(cfg)\n    cfg = dataclasses.replace(cfg, save_test_predictions=False, resume=False)\n    selected = {\"config\": dataclasses.asdict(cfg), \"method\": method,\n                \"selection_split\": \"val\"}\n    path = Path(cfg.out_dir) / \"selection.json\"\n    if path.exists() and json.loads(path.read_text(encoding=\"utf-8\")) != selected:\n        raise ValueError(\"Final configuration already locked. Use a separate output directory for a new study.\")\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(selected, indent=2), encoding=\"utf-8\")\n    return path\n\n\ndef recover_final(cfg):\n    \"\"\"Repair interrupted final artifacts using saved predictions, never test forward.\"\"\"\n    ev = _import_eval()\n    d = run_dir(cfg)\n    metadata = json.loads((d / \"final_inference.json\").read_text(encoding=\"utf-8\"))\n    fp = Path(cfg.pred_dir) / f\"F01_seed{cfg.seed}_test.csv\"\n    vp = Path(cfg.pred_dir) / f\"F01_seed{cfg.seed}_val.csv\"\n    pending = d / \"final_test_arrays.npz\"\n    if not fp.exists():\n        # Saved immediately after the only test forward; recover a missing CSV.\n        with np.load(pending, allow_pickle=True) as data:\n            ev.save_predictions(fp, data['names'].tolist(), data['y'], data['probs'])\n    if metadata['method'] == 'temperature':\n        uncal = Path(cfg.pred_dir) / f\"F01uncal_seed{cfg.seed}_test.csv\"\n        if not uncal.exists():\n            with np.load(pending, allow_pickle=True) as data:\n                ev.save_predictions(uncal, data['names'].tolist(), data['y'],\n                                    inf.apply_temperature(data['logits'], 1.))\n    summary_path = d / \"summary.json\"\n    summary = json.loads(summary_path.read_text(encoding=\"utf-8\"))\n    summary.update(metadata)\n    for split, path in ((\"val\", vp), (\"test\", fp)):\n        pred = ev.read_pred(str(path))\n        metric = ev.compute_metrics(pred.y_true, pred.y_pred, pred.probs)\n        summary.update({f\"{split}_{key}\": metric[key] for key in (\"macro_f1\", \"top1\", \"ece\", \"balanced_acc\")})\n    summary_path.write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n\n\ndef recover_baseline(cfg):\n    ev = _import_eval()\n    summary_path = run_dir(cfg) / \"summary.json\"\n    summary = (json.loads(summary_path.read_text(encoding=\"utf-8\")) if summary_path.exists()\n               else dataclasses.asdict(cfg))\n    summary['debug_run'] = False\n    for split in ('val', 'test'):\n        path = Path(cfg.pred_dir) / f\"T00_seed{cfg.seed}_{split}.csv\"\n        pred = ev.read_pred(str(path))\n        metric = ev.compute_metrics(pred.y_true, pred.y_pred, pred.probs)\n        summary.update({f\"{split}_{k}\": metric[k] for k in ('macro_f1', 'top1', 'ece', 'balanced_acc')})\n    summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')\n\n\ndef run_final(selection_path, baseline, seeds=(0, 1, 2)):\n    \"\"\"Test once per seed for baseline and final; never overwrite test predictions.\"\"\"\n    if len(seeds) < 3 or len(set(seeds)) != len(seeds):\n        raise ValueError(\"Final evaluation requires at least three distinct seeds\")\n    ev = _import_eval()\n    locked = json.loads(Path(selection_path).read_text(encoding=\"utf-8\"))\n    recipe, method = Config(**locked[\"config\"]), locked[\"method\"]\n    if recipe.debug_samples_per_class is not None or baseline.debug_samples_per_class is not None:\n        raise ValueError(\"Final evaluation cannot use debug data\")\n    for seed in seeds:\n        b = dataclasses.replace(baseline, exp_id=\"T00\", seed=seed, save_test_predictions=True)\n        bp = Path(b.pred_dir) / f\"T00_seed{seed}_test.csv\"\n        if not bp.exists():\n            # A completed selection run without test output can supply its checkpoint.\n            summary = run_dir(b) / \"summary.json\"\n            if summary.exists():\n                trained = Config(**json.loads((run_dir(b) / \"config.json\").read_text(encoding=\"utf-8\")))\n                if dataclasses.replace(trained, save_test_predictions=True) != b:\n                    raise ValueError(\"Baseline config differs from its existing run\")\n                device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n                model, _, _ = load_state(b, device)\n                _, _, test = ds.load_split(b.labels_dir, b.fold)\n                names, y, probs = predict_method(model, _loader(b, test), device)\n                ev.save_predictions(bp, names, y, probs)\n                old = json.loads(summary.read_text(encoding=\"utf-8\"))\n                m = ev.compute_metrics(y, probs.argmax(1), probs)\n                old.update({f\"test_{k}\": m[k] for k in (\"macro_f1\", \"top1\", \"ece\", \"balanced_acc\")})\n                summary.write_text(json.dumps(old, indent=2), encoding=\"utf-8\")\n                del model\n            else:\n                run(b)\n        else:\n            validate_saved_config(b)\n            print(f\"Keep existing test predictions: {bp.name}\")\n        recover_baseline(b)\n\n        f = dataclasses.replace(recipe, exp_id=\"F01\", seed=seed, save_test_predictions=False)\n        fp = Path(f.pred_dir) / f\"F01_seed{seed}_test.csv\"\n        if fp.exists() or (run_dir(f) / \"final_test_arrays.npz\").exists():\n            validate_saved_config(f)\n            recover_final(f)\n            print(f\"Keep existing test predictions: {fp.name}\")\n            continue\n        run_experiments([f])\n        device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n        model, _, _ = load_state(f, device)\n        _, val, test = ds.load_split(f.labels_dir, f.fold)\n        vl = _loader(f, val)\n        temperature = 1.\n        if method == \"temperature\":\n            _, yv, zv = inf.predict_logits(model, vl, device)\n            temperature = inf.fit_temperature(zv, yv)\n        nv, yv, pv = predict_method(model, vl, device, method, temperature)\n        ev.save_predictions(Path(f.pred_dir) / f\"F01_seed{seed}_val.csv\", nv, yv, pv)\n        (run_dir(f) / \"final_inference.json\").write_text(\n            json.dumps({\"method\": method, \"temperature\": temperature}), encoding=\"utf-8\")\n        result = predict_method(model, _loader(f, test), device, method, temperature,\n                                return_logits=method == \"temperature\")\n        names, y, probs = result[:3]\n        arrays = {\"names\": np.asarray(names, dtype=object), \"y\": y, \"probs\": probs}\n        if method == \"temperature\":\n            arrays['logits'] = result[3]\n        np.savez_compressed(run_dir(f) / \"final_test_arrays.npz\", **arrays)\n        recover_final(f)\n        del model\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n\ndef export_results(root):\n    \"\"\"Seven rubric sheets with provenance, genuine logs, and ddof=1 seed summaries.\"\"\"\n    from openpyxl import Workbook\n    from openpyxl.styles import Alignment, Font, PatternFill\n    from openpyxl.utils import get_column_letter\n\n    root = Path(root)\n    rows = []\n    for p in sorted((root / \"runs\").glob(\"*/seed*/summary.json\")):\n        r = json.loads(p.read_text(encoding=\"utf-8\"))\n        if r.get(\"debug_run\"):\n            continue\n        r = {k: v for k, v in r.items() if not isinstance(v, (list, dict))}\n        r[\"source\"] = str(p.relative_to(root))\n        cp = p.parent / \"config.json\"\n        if cp.exists():\n            config = json.loads(cp.read_text(encoding=\"utf-8\"))\n            for key in (\"label_smoothing\", \"focal_gamma\", \"mix_alpha\", \"sampler\", \"ema_decay\", \"drop_rate\"):\n                r.setdefault(key, config.get(key))\n        rows.append(r)\n    infer, latency = [], []\n    for p in sorted((root / \"runs/inference\").glob(\"*/seed*/I*.json\")):\n        r = json.loads(p.read_text(encoding=\"utf-8\"))\n        timing = r.pop(\"latency\")\n        r.update({f\"{k}_ms\": timing[k] for k in (\"p50\", \"p95\", \"p99\")})\n        r[\"images_per_s\"] = timing[\"images_per_s\"]\n        r[\"source\"] = str(p.relative_to(root))\n        infer.append(r)\n        latency.append({\"exp_id\": r[\"exp_id\"], \"source_exp_id\": r[\"source_exp_id\"],\n                        \"seed\": r[\"seed\"], \"method\": r[\"method\"], \"fused_bn\": False, **timing})\n    backbones = [r for r in rows if r[\"exp_id\"].startswith(\"B\")]\n    training = [r.copy() for r in rows if r[\"exp_id\"].startswith(\"T\")]\n    baseline_scores = {r[\"seed\"]: r.get(\"val_macro_f1\") for r in training if r[\"exp_id\"] == \"T00\"}\n    for r in training:\n        if r[\"seed\"] in baseline_scores:\n            r[\"delta_vs_T00\"] = r[\"val_macro_f1\"] - baseline_scores[r[\"seed\"]]\n        baseline_config = next((b for b in training if b[\"exp_id\"] == \"T00\" and b[\"seed\"] == r[\"seed\"]), None)\n        if baseline_config:\n            keys = (\"init\", \"aug\", \"loss\", \"label_smoothing\", \"focal_gamma\", \"mix\", \"sampler\",\n                    \"lr_backbone\", \"lr_head\", \"ema_decay\")\n            r[\"changed_fields\"] = \"; \".join(f\"{k}={r.get(k)}\" for k in keys if r.get(k) != baseline_config.get(k)) or \"baseline\"\n    finals = [r.copy() for r in rows if r[\"exp_id\"] in (\"F01\", \"T00\") and \"test_macro_f1\" in r]\n    for exp in (\"F01\", \"T00\"):\n        group = [r for r in finals if r[\"exp_id\"] == exp]\n        if group:\n            agg = {\"exp_id\": exp, \"seed\": \"mean_std\", \"n_seeds\": len(group)}\n            for key in (\"test_macro_f1\", \"test_top1\", \"test_ece\", \"val_macro_f1\"):\n                values = [r[key] for r in group if r.get(key) is not None]\n                if values:\n                    agg[key + \"_mean\"] = float(np.mean(values))\n                    agg[key + \"_std\"] = float(np.std(values, ddof=1)) if len(values) > 1 else None\n            finals.append(agg)\n    perclass = []\n    ev = _import_eval()\n    for exp in (\"F01\", \"T00\"):\n        for p in sorted((root / \"predictions\").glob(f\"{exp}_seed*_test.csv\")):\n            pred = ev.read_pred(str(p))\n            m = ev.compute_metrics(pred.y_true, pred.y_pred, pred.probs)\n            for k, name in enumerate(ds.CLASS_NAMES):\n                perclass.append({\"exp_id\": exp, \"seed\": pred.seed, \"class\": name,\n                                 \"support\": int(m[\"support\"][k]),\n                                 **{key: float(m[key][k]) for key in (\"precision\", \"recall\", \"f1\")},\n                                 \"source\": str(p.relative_to(root))})\n    sheets = {\"Summary\": sorted(rows + infer, key=lambda r: r.get(\"val_macro_f1\", -1), reverse=True)[:10],\n              \"Backbones\": backbones, \"Training\": training, \"Inference\": infer,\n              \"Final\": finals, \"PerClass\": perclass, \"Latency\": latency}\n    wb = Workbook()\n    wb.remove(wb.active)\n    for name, data in sheets.items():\n        ws = wb.create_sheet(name)\n        if not data:\n            ws.append([\"status\"])\n            ws.append([\"Chưa có kết quả chạy thật cho giai đoạn này\"])\n        else:\n            cols = list(dict.fromkeys(k for r in data for k in r))\n            ws.append(cols)\n            for r in data:\n                ws.append([r.get(k) for k in cols])\n        ws.freeze_panes = \"A2\"\n        ws.auto_filter.ref = ws.dimensions\n        for cell in ws[1]:\n            cell.font = Font(bold=True, color=\"FFFFFF\")\n            cell.fill = PatternFill(\"solid\", fgColor=\"17365D\")\n            cell.alignment = Alignment(wrap_text=True, vertical=\"center\")\n        ws.row_dimensions[1].height = 36\n        for column in ws.columns:\n            letter = get_column_letter(column[0].column)\n            ws.column_dimensions[letter].width = min(42, max(15, max(len(str(c.value or \"\")) for c in column) + 2))\n            for cell in column[1:]:\n                if isinstance(cell.value, float):\n                    cell.number_format = \"0.0000\"\n                cell.alignment = Alignment(vertical=\"top\", wrap_text=isinstance(cell.value, str))\n    path = root / \"results.xlsx\"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    wb.save(path)\n    return path\n\n\ndef smoke(root, profile=\"local\", *, images_dir=None, labels_dir=None):\n    base = base_config(root, profile)\n    if images_dir is not None:\n        base.images_dir = str(images_dir)\n    if labels_dir is not None:\n        base.labels_dir = str(labels_dir)\n    cfg = dataclasses.replace(base, exp_id=\"SMOKE\", backbone=\"mobilenetv3\", init=\"scratch\",\n                              epochs=1, batch_size=2, img_size=64, cache_images=True,\n                              num_workers=0, debug_samples_per_class=2,\n                              out_dir=str(Path(root) / \"runs/_smoke\"),\n                              pred_dir=str(Path(root) / \"runs/_smoke/predictions\"),\n                              logit_dir=str(Path(root) / \"runs/_smoke/logits\"),\n                              curves_dir=str(Path(root) / \"curves/_smoke\"))\n    return run(cfg)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"stage\", choices=(\"smoke\", \"export\"))\n    ap.add_argument(\"--root\", default=str(Path(__file__).resolve().parent.parent))\n    ap.add_argument(\"--profile\", choices=(\"local\", \"kaggle\"), default=\"local\")\n    args = ap.parse_args()\n    result = smoke(args.root, args.profile) if args.stage == \"smoke\" else export_results(args.root)\n    print(json.dumps(result, ensure_ascii=False, indent=2) if isinstance(result, dict) else result)\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "code/selftest.py": "\"\"\"selftest.py - các kiểm tra tự viết cho những phần dễ sai (RUBRIC mục A3 và H1).\n\nChạy:  python selftest.py          (không cần GPU, không cần dataset)\n       python selftest.py --only \"gộp BatchNorm\" CutMix    (chỉ chạy một số kiểm tra)\n\nBao gồm:\n  - focal loss với gamma = 0 phải đúng CE (sai số < 1e-6)          [GUIDE.md 3.2]\n  - label smoothing với eps = 0 phải đúng CE\n  - CutMix: lam phải khớp đúng diện tích hộp thực tế bị cắt         [slide trang 48]\n  - Mixup/CutMix trộn cả nhãn, mixed_loss khớp công thức\n  - gộp BatchNorm: sai số đầu ra <= 1e-5                            [slide trang 71]\n  - class_weights: trung bình = 1, beta=0 cho trọng số nghịch đảo\n  - temperature scaling: T > 0, không đổi argmax, NLL/ECE giảm trên val\n  - 3 nhóm tham số có weight_decay = 0 cho norm và bias           [GUIDE.md 1.4]\n  - đếm GMAC cho resnet50 xấp xỉ 4.1 (số trên slide)                [slide trang 41]\n  - hợp đồng định dạng file dự đoán với eval.py (save -> read -> score)\n  - parse_overrides ép đúng kiểu và báo lỗi rõ ràng\n\"\"\"\nfrom __future__ import annotations\n\nimport runtime  # Select safe pandas string storage on Windows before Torch.\n\nimport os\nimport sys\nimport tempfile\nimport traceback\nfrom pathlib import Path\n\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfor _s in (sys.stdout, sys.stderr):\n    if hasattr(_s, \"reconfigure\"):\n        try:\n            _s.reconfigure(encoding=\"utf-8\", errors=\"replace\")\n        except (ValueError, OSError):\n            pass\n\nPASSED, FAILED = [], []\n\n# `python selftest.py --only \"gộp BatchNorm\" temperature` -> chỉ chạy các kiểm tra chứa\n# một trong các chuỗi nêu trong --only. Hữu ích khi cần biết kiểm tra nào gây lỗi.\n_ONLY = [a for a in sys.argv if not a.startswith(\"-\")][1:] if \"--only\" in sys.argv else []\n\n\ndef check(name):\n    def deco(fn):\n        if _ONLY and not any(tok.lower() in name.lower() for tok in _ONLY):\n            return fn\n        print(f\"  --> {name}\", flush=True)\n        try:\n            fn()\n            PASSED.append(name)\n            print(f\"  [OK]   {name}\", flush=True)\n        except Exception as e:                      # noqa: BLE001 - báo lỗi rồi chạy tiếp\n            FAILED.append((name, e))\n            print(f\"  [FAIL] {name}: {e}\", flush=True)\n            traceback.print_exc()\n        return fn\n    return deco\n\n\n# --------------------------------------------------------------------------- #\n@check(\"focal loss gamma=0 cho đúng cross-entropy (sai số < 1e-6)\")\ndef check_focal_gamma_zero_is_ce():\n    from losses import FocalLoss\n\n    torch.manual_seed(0)\n    logits = torch.randn(64, 9, dtype=torch.float64)\n    y = torch.randint(0, 9, (64,))\n    ce = F.cross_entropy(logits, y)\n    for gamma in (0.0,):\n        fl = FocalLoss(gamma=gamma)(logits, y)\n        err = abs(float(fl) - float(ce))\n        assert err < 1e-6, f\"gamma={gamma}: sai số {err:.3e}\"\n    # focal với alpha=None và reduction='none' phải lấy trung bình khớp\n    per_sample = FocalLoss(gamma=2.0, reduction=\"none\")(logits, y)\n    assert abs(float(per_sample.mean()) - float(FocalLoss(gamma=2.0)(logits, y))) < 1e-12\n\n\n@check(\"label smoothing eps=0 cho đúng CE\")\ndef check_label_smoothing_zero_is_ce():\n    from losses import LabelSmoothingCE\n\n    torch.manual_seed(0)\n    logits = torch.randn(32, 9, dtype=torch.float64)\n    y = torch.randint(0, 9, (32,))\n    ce = F.cross_entropy(logits, y)\n    err = abs(float(LabelSmoothingCE(0.0)(logits, y)) - float(ce))\n    assert err < 1e-6, f\"sai số {err:.3e}\"\n    ls = float(LabelSmoothingCE(0.1)(logits, y))\n    assert ls > 0, \"loss phải dương\"\n\n\n@check(\"CutMix: lam khớp diện tích hộp thực tế bị cắt\")\ndef check_cutmix_lambda_matches_area():\n    import losses\n\n    torch.manual_seed(0)\n    np.random.seed(0)\n    x = torch.randn(16, 3, 32, 32)\n    y = torch.randint(0, 9, (16,))\n    # Cố định permutation thành derangement (không có phần tử nào giữ nguyên vị trí), nếu không\n    # có ảnh nào nhận lại chính nó thì mọi pixel trong hộp đều khác ảnh gốc và đếm pixel chính xác.\n    perm = torch.tensor([1, 0, 3, 2, 5, 4, 7, 6, 9, 8, 11, 10, 13, 12, 15, 14])\n    orig_randperm = torch.randperm\n    torch.randperm = lambda n, *a, **k: perm.clone().to(k.get(\"device\", x.device))\n    try:\n        x_mix, (y_a, y_b, lam) = losses.mix_batch(x, y, alpha=1.0, mode=\"cutmix\")\n    finally:\n        torch.randperm = orig_randperm\n\n    changed = (x_mix != x).any(dim=1)\n    per_img = changed.sum(dim=(1, 2))\n    frac = changed.sum().item() / (x.shape[0] * x.shape[2] * x.shape[3])\n    assert abs(frac - (1.0 - lam)) < 1e-6, f\"pixel khác {frac:.6f} nhưng 1-lam = {1.0 - lam:.6f}\"\n    assert bool((per_img == per_img[0]).all()), f\"hộp phải giống nhau mọi ảnh, nhận {per_img.tolist()}\"\n    # nhãn phải được trộn cả hai lớp (slide trang 48)\n    assert torch.equal(y_a, y) and torch.equal(y_b, y[perm]), \"nhãn phải trộn theo cùng permutation\"\n    assert 0.0 <= lam <= 1.0\n\n\n@check(\"Mixup: ảnh trộn là đúng phép convex, nhãn trộn cả hai lớp\")\ndef check_mixup():\n    from losses import mix_batch, mixed_loss\n\n    torch.manual_seed(0)\n    np.random.seed(0)\n    x = torch.randn(8, 3, 16, 16)\n    y = torch.randint(0, 9, (8,))\n    x_mix, (y_a, y_b, lam) = mix_batch(x, y, alpha=1.0, mode=\"mixup\")\n    assert 0.0 <= lam <= 1.0\n    assert x_mix.shape == x.shape\n    # giá trị mỗi pixel nằm giữa hai ảnh gốc tương ứng\n    lo = torch.minimum(x, x[torch.randperm(8)]) - 1e-6\n    assert x_mix.min() >= lo.min() - 1e-5, \"giá trị trộn nằm ngoài khoảng của hai ảnh gốc\"\n    # mixed_loss khớp công thức lam*CE(y_a) + (1-lam)*CE(y_b)\n    logits = torch.randn(8, 9)\n    crit = nn.CrossEntropyLoss()\n    expect = lam * crit(logits, y_a) + (1 - lam) * crit(logits, y_b)\n    assert abs(float(mixed_loss(crit, logits, (y_a, y_b, lam))) - float(expect)) < 1e-6\n\n\n@check(\"gộp BatchNorm: sai số đầu ra <= 1e-5\")\ndef check_fuse_conv_bn():\n    from inference import fuse_conv_bn\n\n    torch.manual_seed(0)\n    net = nn.Sequential(nn.Conv2d(3, 8, 3, stride=2, padding=1, bias=False),\n                        nn.BatchNorm2d(8), nn.ReLU(),\n                        nn.Conv2d(8, 4, 3, padding=1, bias=True),\n                        nn.BatchNorm2d(4), nn.ReLU()).eval()\n    # cố tình làm running stats khác batch stats để kiểm tra đúng phép chuyển đổi\n    with torch.no_grad():\n        for m in net.modules():\n            if isinstance(m, nn.BatchNorm2d):\n                m.running_mean.uniform_(-1, 1)\n                m.running_var.uniform_(0.5, 2.0)\n                m.weight.uniform_(0.5, 1.5)\n                m.bias.uniform_(-0.5, 0.5)\n    x = torch.randn(4, 3, 17, 19)          # kích thước lẻ để bắt lỗi đảo chiều/padding\n    with torch.no_grad():\n        ref = net(x).clone()\n    fused = fuse_conv_bn(net)\n    with torch.no_grad():\n        out = fused(x)\n    err = float((ref - out).abs().max())\n    assert err < 1e-5, f\"sai số gộp BN quá lớn: {err:.3e}\"\n    assert not any(isinstance(m, nn.BatchNorm2d) for m in fused.modules()), \"còn sót BN\"\n    print(f\"         sai số lớn nhất = {err:.3e}\")\n\n\n@check(\"class_weights: trung bình = 1; beta=0 nghịch với số ảnh\")\ndef check_class_weights():\n    from losses import class_weights\n\n    counts = np.array([1125, 1064, 1031, 1022, 1062, 1009, 1074, 1016, 9106], dtype=np.float64)\n    w0 = class_weights(counts, beta=0.0)\n    assert abs(float(w0.mean()) - 1.0) < 1e-6, f\"trung bình = {float(w0.mean())}\"\n    ratio = float(w0[8] / w0[0])                     # Negatives/Chinee Apple\n    expect = counts[0] / counts[8]\n    assert abs(ratio - expect) < 1e-4 * expect, f\"tỉ lệ trọng số sai: {ratio} vs {expect}\"\n    w9 = class_weights(counts, beta=0.999)\n    assert abs(float(w9.mean()) - 1.0) < 1e-6\n    assert float(w9[8]) < float(w9[0]), \"lớp hiếm phải được trọng số lớn hơn\"\n    assert w9.numel() == 9\n\n\n@check(\"temperature scaling: T>0, không đổi argmax, NLL giảm trên val\")\ndef check_temperature():\n    from inference import apply_temperature, ece_of, fit_temperature\n\n    rng = np.random.default_rng(0)\n    n = 2000\n    y = rng.integers(0, 9, n)\n    logits = rng.normal(size=(n, 9)) * 3.0 + np.eye(9)[y] * 2.0   # over-confident\n    t = fit_temperature(logits, y)\n    assert t > 0, f\"T phải > 0, nhận {t}\"\n    p_before = apply_temperature(logits, 1.0)\n    p_after = apply_temperature(logits, t)\n    assert np.array_equal(p_before.argmax(1), p_after.argmax(1)), \"temperature scaling phải giữ nhãn\"\n    nll = lambda p: float(-np.log(np.clip(p[np.arange(n), y], 1e-12, None)).mean())\n    assert nll(p_after) <= nll(p_before) + 1e-6, \"NLL phải giảm trên tập đã khớp\"\n    ece_b, ece_a = ece_of(p_before, y), ece_of(p_after, y)\n    print(f\"         T = {t:.3f}, NLL {nll(p_before):.4f} -> {nll(p_after):.4f}, \"\n          f\"ECE {ece_b:.4f} -> {ece_a:.4f}\")\n    assert abs(p_after.sum(1) - 1).max() < 1e-9, \"xác suất phải chuẩn hoá\"\n\n\n@check(\"nhóm tham số: weight_decay = 0 cho norm và bias, head LR gấp 10 lần\")\ndef check_param_groups():\n    try:\n        import timm\n    except ImportError:\n        print(\"         (bỏ qua: chưa cài timm)\")\n        return\n    from model import param_groups\n\n    model = timm.create_model(\"resnet50\", pretrained=False, num_classes=9)\n    groups = param_groups(model, 1e-4, 1e-3, 0.05)\n    # Slide trang 52 mô tả 3 nhóm (backbone có wd / backbone norm+bias / head). Vì nguyên tắc\n    # \"weight_decay = 0 cho norm và bias\" được áp cho cả backbone lẫn head, head bị tách\n    # thêm khi có bias (ndim<=1) -> 4 nhóm. Kiểm tra theo TÍNH CHẤT, không theo số nhóm.\n    by_wd = {g[\"weight_decay\"] for g in groups}\n    assert by_wd == {0.0, 0.05}, f\"chỉ được có wd 0 và 0.05, nhận {by_wd}\"\n    for g in groups:\n        for p in g[\"params\"]:\n            expected = 0.0 if p.ndim <= 1 else 0.05\n            assert g[\"weight_decay\"] == expected, (\n                f\"tensor ndim={p.ndim} đang có weight_decay={g['weight_decay']}, cần {expected}\")\n    backbone = [p for g in groups for p in g[\"params\"] if g[\"lr\"] == 1e-4]\n    head = [p for g in groups for p in g[\"params\"] if g[\"lr\"] == 1e-3]\n    assert backbone and head, \"phải có nhóm backbone (1e-4) và head (1e-3)\"\n    # head phải đúng là tham số của get_classifier\n    head_ids = {id(p) for p in model.get_classifier().parameters()}\n    assert {id(p) for p in head} == head_ids, \"nhóm head phải khớp get_classifier()\"\n    # tham số đóng băng không được xuất hiện trong bất kỳ nhóm nào\n    model2 = timm.create_model(\"resnet50\", pretrained=False, num_classes=9)\n    from model import build_model, freeze_backbone\n\n    m2 = build_model(\"resnet50\", pretrained=False, num_classes=9, init=\"frozen\")\n    g2 = param_groups(m2, 1e-4, 1e-3, 0.05)\n    assert len(g2) == 2, f\"backbone đóng băng chỉ còn nhóm head, nhận {len(g2)} nhóm\"\n\n\n@check(\"GMAC của resnet50 xấp xỉ 4.1 (số trên slide trang 41)\")\ndef check_gmac():\n    try:\n        import timm\n    except ImportError:\n        print(\"         (bỏ qua: chưa cài timm)\")\n        return\n    from model import count_gmacs_with_tool, count_params\n\n    model = timm.create_model(\"resnet50\", pretrained=False, num_classes=9)\n    gmac, tool = count_gmacs_with_tool(model, 224)\n    p = count_params(model)\n    # Slide ghi 25,6M là cho đầu 1000 lớp; với đầu 9 lớp của bài lab số tham số nhỏ hơn\n    # (fc: 2048*9+9 = 18,4K thay cho 2048*1000+1000 = 2,05M) -> kỳ vọng ~23,5M.\n    print(f\"         resnet50 (9 lớp): {p:.1f}M tham số, {gmac:.2f} GMAC ({tool}); \"\n          f\"slide ghi 25.6M (đầu 1000 lớp) / 4.1 GMAC\")\n    assert 3.5 < gmac < 4.7, f\"GMAC {gmac:.2f} lệch quá nhiều so với ~4.1 của slide\"\n    assert 23.0 < p < 24.0, f\"số tham số {p:.1f}M lệch quá nhiều so với ~23.5M của ResNet-50 9 lớp\"\n    # đối chiếu: cùng kiến trúc nhưng 1000 lớp thì ra đúng 25,6M như slide\n    p1000 = count_params(timm.create_model(\"resnet50\", pretrained=False, num_classes=1000))\n    assert 25.3 < p1000 < 25.9, f\"ResNet-50 1000 lớp phải ~25.6M, nhận {p1000:.1f}M\"\n\n\n@check(\"file dự đoán đi qua đúng kiểm tra của eval.py\")\ndef check_prediction_contract():\n    import eval as ev\n    from train import softmax_np\n\n    rng = np.random.default_rng(0)\n    n = 200\n    y = rng.integers(0, 9, n)\n    probs = softmax_np(rng.normal(size=(n, 9)))\n    names = [f\"img{i}.jpg\" for i in range(n)]\n    with tempfile.TemporaryDirectory() as d:\n        p = ev.save_predictions(Path(d) / \"F01_seed0_test.csv\", names, y, probs)\n        pred = ev.read_pred(str(p))\n        assert pred.seed == 0\n        np.testing.assert_array_equal(pred.y_pred, probs.argmax(1))\n        np.testing.assert_allclose(pred.probs, probs, atol=1e-6)\n    # logit (chưa softmax) phải bị eval.py từ chối\n    try:\n        ev.save_predictions(\"x.csv\", names[:5], y[:5], rng.normal(size=(5, 9)))\n        raise AssertionError(\"eval.save_predictions phải từ chối logit chưa chuẩn hoá\")\n    except ValueError:\n        pass\n\n\n@check(\"parse_overrides: ép kiểu đúng, lỗi rõ ràng\")\ndef check_parse_overrides():\n    from train import Config, parse_overrides\n\n    d = parse_overrides([\"exp_id=B01\", \"seed=1\", \"lr_backbone=2e-5\", \"amp=false\",\n                         \"mix=cutmix\", \"ema_decay=0.999\", \"sampler=none\"])\n    cfg = Config(**d)\n    assert cfg.exp_id == \"B01\" and cfg.seed == 1\n    assert abs(cfg.lr_backbone - 2e-5) < 1e-12\n    assert cfg.amp is False\n    assert cfg.mix == \"cutmix\" and cfg.ema_decay == 0.999 and cfg.sampler is None\n    try:\n        parse_overrides([\"khong_ton_tai=1\"])\n        raise AssertionError(\"phải báo lỗi với field không có trong Config\")\n    except ValueError as e:\n        assert \"không phải field\" in str(e)\n    try:\n        parse_overrides([\"seed=abc\"])\n        raise AssertionError(\"phải báo lỗi khi không ép được kiểu\")\n    except ValueError:\n        pass\n\n\n@check(\"sinh ảnh ngẫu nhiên trong loader: đủ số, đúng thứ tự, đúng nhãn\")\ndef check_loader_contract():\n    \"\"\"Lưu ý: ép `pin_memory=False`. Đây là unit test chạy trên CPU; `pin_memory=True` chỉ có\n    lợi khi chuyển tensor sang GPU, và trên một số cấu hình Windows + GPU, việc cấp phát\n    pinned host memory làm sập tiến trình (access violation). Trên Kaggle nên để mặc định\n    (None -> tự bật khi có CUDA).\"\"\"\n    import pandas as pd\n    from PIL import Image\n\n    import dataset as ds\n\n    rng = np.random.default_rng(0)\n    with tempfile.TemporaryDirectory() as d:\n        names, labels = [], []\n        for i in range(12):\n            fn = f\"img{i}.jpg\"\n            Image.fromarray(rng.integers(0, 255, (64, 64, 3), dtype=np.uint8)).save(Path(d) / fn)\n            names.append(fn)\n            labels.append(i % 9)\n        df = pd.DataFrame({\"Filename\": names, \"Label\": labels, \"Species\": [\"x\"] * len(names)})\n        eval_tf = ds.build_transforms(False, img_size=32)      # xác định, không ngẫu nhiên\n        ds_obj = ds.DeepWeedsDataset(df, d, eval_tf, img_size=64)\n        img, label, fn = ds_obj[3]\n        assert img.shape == (3, 32, 32), f\"dạng ảnh sai: {tuple(img.shape)}\"\n        assert label == 3 and fn == \"img3.jpg\"\n        # cache RAM phải cho kết quả giống hệt đọc từ đĩa (dùng CÙNG một transform xác định)\n        ds_cached = ds.DeepWeedsDataset(df, d, eval_tf, cache_in_ram=True, img_size=64)\n        for i in (0, 5, 11):\n            a, b = ds_obj[i][0], ds_cached[i][0]\n            assert torch.allclose(a, b), f\"cache RAM khác đọc đĩa ở mẫu {i}\"\n        loader = ds.make_loader(df, d, ds.build_transforms(False, img_size=32), batch_size=4,\n                                train=False, num_workers=0, img_size=64, pin_memory=False)\n        got = []\n        for _, _, fs in loader:\n            got.extend(fs)\n        assert got == names, \"thứ tự file của loader không ổn định\"\n        # check_split phải bắt được giao không rỗng giữa các tập\n        try:\n            ds.check_split(df, df, df.head(2), d)\n            raise AssertionError(\"phải báo lỗi khi các tập giao nhau\")\n        except ValueError as e:\n            assert \"GIAO KHÔNG RỖNG\" in str(e) or \"17.509\" in str(e)\n\n\n# --------------------------------------------------------------------------- #\nif __name__ == \"__main__\":\n    print(\"Chạy selftest (không cần GPU, không cần dataset)\\n\", flush=True)\n    print(f\"KẾT QUẢ: {len(PASSED)} đạt, {len(FAILED)} lỗi\")\n    if FAILED:\n        print(\"\\nCác kiểm tra lỗi:\")\n        for name, e in FAILED:\n            print(f\"  - {name}: {e}\")\n        code = 1\n    else:\n        print(\"Tất cả kiểm tra đạt.\")\n        code = 0\n    sys.stdout.flush()\n    sys.stderr.flush()\n    # Ghi chú về `os._exit`: trên một số cấu hình WINDOWS, khi cả torch và pyarrow (do pandas 3.x\n    # dùng để tạo string array) cùng nạp, bộ dọn DLL của interpreter có thể báo access violation\n    # (-1073741819). Đây là lỗi môi trường, KHÔNG phải lỗi của các kiểm tra: mọi kiểm tra đều\n    # đã chạy xong và in kết quả trước khi sập. `os._exit` bảo đảm mã thoại phản ánh đúng kết\n    # quả kiểm tra. Trên Kaggle (Linux) không cần, và không ảnh hưởng gì.\n    os._exit(code)\n",
  "code/README.md": "# Bài làm DeepWeeds — cách chạy\n\n## Kaggle: chỉ upload notebook và dataset\n\nImport **`lab_day2_kaggle.ipynb`**, thêm dataset vào Input, bật GPU và Internet rồi Run All.\nNotebook này chứa sẵn toàn bộ module và `eval.py`; không cần upload các file `.py` hoặc clone repo.\nTự tìm ảnh và CSV trong `/kaggle/input/`, kể cả hai folder nằm ở hai Kaggle dataset riêng.\nMặc định chạy `setup`; đổi STAGE ở cell đầu để chạy các bước tiếp theo.\nOutput ở `/kaggle/working/deepweeds-lab/`. Giữ checkpoint/log khi chuyển sang phiên Kaggle mới.\n\n`build_kaggle_notebook.py` tạo lại bản độc lập sau khi code nguồn hoặc notebook gốc thay đổi.\n\n## Notebook local\n\nMở `lab_day2.ipynb` trong `code/` hoặc từ thư mục gốc repo. Notebook dùng code đã triển\nkhai, không import các stub trong `starter/`.\n\n## Môi trường\n\nCài PyTorch và torchvision phù hợp GPU trước, rồi chạy từ thư mục gốc:\n\n```powershell\npython -m pip install -r code/requirements.txt\n```\n\nNotebook ghi phiên bản thực tế vào `runs/environment.json`; mỗi lần train ghi Config\nvà phiên bản Torch trong thư mục riêng. Dùng `PROFILE='local'` cho RTX 2050 4 GB:\nbatch 4, worker 0, AMP, không cache toàn bộ dataset. `kaggle` dùng batch 32, worker 2.\nĐổi profile phải áp dụng cho toàn bộ nhóm so sánh và ghi rõ trong báo cáo.\n\nẢnh đặt ở `images/` hoặc `data/images/`; nhãn và fold 0 nguyên bản đặt ở `data/labels/`.\nNotebook tự tìm ảnh và kiểm tra giao rỗng, số ảnh, file thiếu trước khi chạy.\n\n## Thứ tự chạy notebook\n\n1. `STAGE='setup'`: EDA, kiểm tra split, self-test. Mặc định không huấn luyện.\n2. `STAGE='smoke'`: 1 epoch MobileNetV3 scratch, 2 ảnh/lớp train và val, ảnh 64×64.\n   Output riêng ở `runs/_smoke/` và `curves/_smoke/`; không vào bảng chính thức.\n3. `STAGE='backbones'`: 5 backbone, cùng recipe, 12 epoch, seed 0.\n4. `STAGE='training'`: backbone thắng trên val; T00 và ablation khởi tạo, augmentation,\n   loss, CutMix/Mixup. Mỗi lần chỉ đổi một yếu tố; so sánh với T00.\n5. `STAGE='inference'`: 1 view, hflip/prob, five-crop/prob, hflip/logit, temperature.\n   5 phương pháp, FP32, độ trễ batch 1: warmup 10 và 50 lần đo có đồng bộ CUDA.\n6. `STAGE='final'`, `ALLOW_FINAL_TEST=True`: khóa cấu hình bằng val, chạy T00 và F01\n   với seed 0/1/2, test một lần mỗi seed. File test có sẵn được giữ nguyên.\n7. `STAGE='export'`: đọc log thật để xuất `results.xlsx` và chạy `eval.py` khi đã có\n   dự đoán chung kết. Sheet thiếu kết quả ghi trạng thái thiếu; không tự điền số giả.\n\nSau mỗi bước, đổi STAGE rồi restart kernel và Run All. Các nhóm đã chạy xong được tái\nsử dụng khi Config không đổi. Với cấu hình mới, dùng exp_id mới; với nghiên cứu mới,\ndùng out_dir/pred_dir riêng. Không lấy kết quả test để điều chỉnh lựa chọn.\n\nLần đầu dùng pretrained cần Internet tải trọng số ImageNet. Notebook không tự tải\nảnh vì workspace hiện đã có dataset. Colab/Kaggle có thể tải theo README gốc rồi mở notebook.\n\n## Chạy CLI và test\n\n```powershell\npython -X utf8 code/selftest.py\npython -X utf8 -m unittest discover -s tests\npython -X utf8 -m unittest discover -s code/tests -v\npython -X utf8 code/lab_workflow.py smoke\npython -X utf8 code/lab_workflow.py export\n```\n\nMột thí nghiệm riêng trên máy local, đầy đủ train/val:\n\n```powershell\npython -X utf8 code/train.py --set exp_id=B01 backbone=resnet50 images_dir=images batch_size=4 num_workers=0 cache_images=false pin_memory=false\n```\n\nTrên Windows, `runtime.py` chọn pandas Python string storage để tránh native crash\nPyArrow sau khi Torch đã nạp. `-X utf8` xử lý tiếng Việt khi `eval.py` ghi JSON; giữ nguyên\n`eval.py` theo yêu cầu bài lab.\n\n## Output và giới hạn\n\n- `runs/<exp_id>/seed<k>/`: config.json, history.csv, best.pt, summary.json.\n- `logits/<exp_id>/`: logit val và test (chỉ khi được bật).\n- `curves/`: đường cong loss/metric/LR; `_eda/` cho EDA, `_smoke/` cho kiểm tra.\n- `predictions/`: CSV chuẩn cho T00, F01 và val của F01; bản uncal khi chọn temperature.\n- `results.xlsx`: Summary, Backbones, Training, Inference, Final, PerClass, Latency.\n  Có cột source truy về log/dự đoán; std dùng ddof=1, không điền 0 khi chỉ có một seed.\n\nĐộ trễ của workflow bao gồm transform tensor, forward và gộp view trên device; loại trừ\nđọc/giải mã ảnh. Đây chưa phải độ trễ end-to-end của robot. Viết hạn chế này trong báo cáo.\nChạy thử chỉ xác nhận pipeline, không chứng minh accuracy. Bộ test gốc kiểm tra starter\nvà eval; bộ test bổ sung kiểm tra phần triển khai. Kết quả nghiên cứu chỉ có sau khi chạy\nđủ các stage chính thức. Điền link notebook Colab/Kaggle của bạn khi chuẩn bị nộp bài.\n\n`build_notebook.py` tái tạo notebook sạch output khi cần cập nhật cấu trúc cell.\nXem `VALIDATION.md` để biết các kiểm tra đã chạy và giới hạn của kết quả hiện tại.\n",
  "code/requirements.txt": "# Install the GPU-compatible torch/torchvision pair for your environment first.\nnumpy\npandas\nPillow\ntorch\ntorchvision\ntimm\nmatplotlib\nscikit-learn\nopenpyxl\nnbformat\nnbclient\nipykernel\n"
}
for relative, source in BUNDLED_FILES.items():
    destination = LAB_WORK_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(source.encode('utf-8'))

required_csv = ('labels.csv', 'train_subset0.csv', 'val_subset0.csv', 'test_subset0.csv')
manual_labels = globals().get('MANUAL_LABELS_DIR')
if manual_labels:
    candidates = [Path(manual_labels)]
else:
    candidates = sorted({p.parent for p in KAGGLE_INPUT_ROOT.rglob('labels.csv')
                         if all((p.parent / name).is_file() for name in required_csv)})
if not candidates:
    raise FileNotFoundError('Add Input: cần đủ 4 CSV nhãn/fold 0 trong cùng folder labels')
if len(candidates) != 1:
    raise ValueError(f'Nhiều folder labels: {candidates}. Đặt MANUAL_LABELS_DIR ở cell cấu hình')
KAGGLE_LABELS_DIR = candidates[0]
if not all((KAGGLE_LABELS_DIR / name).is_file() for name in required_csv):
    raise FileNotFoundError('Folder labels thiếu CSV bắt buộc: ' + str(KAGGLE_LABELS_DIR))
with (KAGGLE_LABELS_DIR / 'labels.csv').open(encoding='utf-8-sig', newline='') as stream:
    reference = [row['Filename'] for _, row in zip(range(3), csv.DictReader(stream))]
if not reference:
    raise ValueError('labels.csv không có dữ liệu')
manual_images = globals().get('MANUAL_IMAGES_DIR')
if manual_images:
    image_candidates = [Path(manual_images)]
else:
    image_candidates = sorted({p.parent for p in KAGGLE_INPUT_ROOT.rglob(reference[0])
                               if all((p.parent / name).is_file() for name in reference)})
if not image_candidates:
    raise FileNotFoundError('Add Input: chưa tìm thấy folder images chứa các tên ảnh trong labels.csv')
if len(image_candidates) != 1:
    raise ValueError(f'Nhiều folder ảnh: {image_candidates}. Đặt MANUAL_IMAGES_DIR ở cell cấu hình')
KAGGLE_IMAGES_DIR = image_candidates[0]
if not all((KAGGLE_IMAGES_DIR / name).is_file() for name in reference):
    raise FileNotFoundError('Đường dẫn ảnh không khớp CSV: ' + str(KAGGLE_IMAGES_DIR))

# Preserve Kaggle's installed torch/torchvision CUDA pair. Install other packages only if absent.
packages = {'timm': 'timm', 'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow',
            'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn', 'openpyxl': 'openpyxl'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    if not globals().get('INSTALL_MISSING_DEPENDENCIES', True):
        raise ImportError('Missing notebook dependencies: ' + ', '.join(missing))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
for module in ('torch', 'torchvision'):
    if importlib.util.find_spec(module) is None:
        raise ImportError('Kaggle cần có torch/torchvision; chọn môi trường notebook GPU')
print('Ảnh:', KAGGLE_IMAGES_DIR)
print('CSV:', KAGGLE_LABELS_DIR)
print('Code và output:', LAB_WORK_ROOT)

In [ ]:
import os, sys, json, subprocess, dataclasses, platform
from pathlib import Path

ROOT = LAB_WORK_ROOT
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'code'))
import runtime
import numpy as np, pandas as pd, torch, timm, matplotlib.pyplot as plt
import dataset as ds
import train
from lab_workflow import (base_config, experiment_plan, run_experiments, best_config,
                         compare_inference, lock_selection, run_final, export_results, smoke)
if Path(train.__file__).resolve().parent != ROOT / 'code':
    raise RuntimeError('Kernel đã import starter/train.py; restart kernel rồi chạy lại')

STAGE = globals().get('STAGE', 'setup')
PROFILE = 'kaggle'                 # local | kaggle
ALLOW_FINAL_TEST = globals().get('ALLOW_FINAL_TEST', False)          # bật khi đã chốt cấu hình trên val
SEEDS = (0, 1, 2)
SELECTED_METHOD = None            # None: chọn theo macro-F1 val, ECE rồi độ trễ
assert STAGE in ('setup', 'smoke', 'backbones', 'training', 'inference', 'final', 'export')
BASE = base_config(ROOT, PROFILE)
BASE.images_dir = str(KAGGLE_IMAGES_DIR)
BASE.labels_dir = str(KAGGLE_LABELS_DIR)
print('Repo:', ROOT, '| Stage:', STAGE, '| Profile:', PROFILE)
print('python:', platform.python_version(), '| torch:', torch.__version__, '| timm:', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Ảnh:', BASE.images_dir, '| batch:', BASE.batch_size, '| workers:', BASE.num_workers)
(ROOT / 'runs').mkdir(exist_ok=True)
(ROOT / 'runs/environment.json').write_text(json.dumps({
    'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__,
    'pandas': pd.__version__, 'profile': PROFILE, 'seeds': SEEDS,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
}, indent=2), encoding='utf-8')

### Dữ liệu và kiểm tra

Notebook đã tìm đường dẫn images/CSV trong Input ở trên. Cell kế tiếp kiểm tra toàn bộ
17.509 ảnh theo fold 0: số ảnh, giao giữa các tập, union và file thiếu.
Dataset hiện có một nhãn lệch giữa labels.csv và train_subset0.csv cho
`20170714-110407-3.jpg`; giữ CSV nguyên bản theo đề bài, ghi nhận trong báo cáo.


In [ ]:
train_df, val_df, test_df = ds.load_split(BASE.labels_dir, fold=0)
split_report = ds.check_split(train_df, val_df, test_df, BASE.images_dir, strict_ratio=True)
(ROOT / 'runs/eda.json').write_text(json.dumps(split_report, indent=2, ensure_ascii=False), encoding='utf-8')
counts = pd.DataFrame({name: frame['Label'].value_counts().reindex(range(9), fill_value=0)
                      for name, frame in [('train', train_df), ('val', val_df), ('test', test_df)]})
counts.index = ds.CLASS_NAMES
display(counts)
ax = counts.plot.bar(figsize=(12, 4), rot=30)
ax.set_ylabel('Số ảnh'); ax.set_title('Fold 0 — phân bố lớp'); plt.tight_layout()
eda_dir = ROOT / 'curves/_eda'; eda_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(eda_dir / 'class_distribution.png', dpi=140); plt.show()

In [ ]:
from PIL import Image
fig, axes = plt.subplots(9, 3, figsize=(9, 23))
image_dir = ds.resolve_images_dir(BASE.images_dir)
for label, name in enumerate(ds.CLASS_NAMES):
    examples = train_df[train_df.Label == label].head(3)
    for ax, (_, row) in zip(axes[label], examples.iterrows()):
        with Image.open(image_dir / row.Filename) as image:
            ax.imshow(image.convert('RGB'))
        ax.set_title(f'{name} | {row.Filename}', fontsize=8); ax.axis('off')
plt.tight_layout(); plt.savefig(eda_dir / 'examples.png', dpi=120); plt.show()

## 1. Kiểm tra pipeline và chạy thử

Self-test kiểm tra loss, Mixup/CutMix, BN, temperature, nhóm tham số và hợp đồng dự đoán.
Stage `smoke` chạy 1 epoch MobileNetV3 scratch trên 2 ảnh/lớp train và val ở 64×64;
không dùng test. Log nằm ở `runs/_smoke/`, biểu đồ ở `curves/_smoke/`.
Đây là kiểm tra đường đi dữ liệu và checkpoint, không phải thí nghiệm so sánh.

In [ ]:
if STAGE in ('setup', 'smoke'):
    subprocess.run([sys.executable, '-X', 'utf8', str(ROOT / 'code/selftest.py')], check=True, cwd=ROOT)
if STAGE == 'smoke':
    smoke_summary = smoke(ROOT, PROFILE, images_dir=BASE.images_dir, labels_dir=BASE.labels_dir)
    display(pd.DataFrame([smoke_summary]))

## 2. So sánh 5 backbone

ResNet50, ConvNeXt-Tiny, DeiT-Small, Swin-Tiny và MobileNetV3; cùng recipe, seed, 12 epoch.
Chọn backbone có macro-F1 val tốt nhất để đi tiếp. Xem cả thời gian và kích thước trước khi
viết kết luận. Trọng số ImageNet được tải lần đầu, cần Internet hoặc cache sẵn.

In [ ]:
backbone_configs = experiment_plan(BASE, 'backbones')
display(pd.DataFrame([dataclasses.asdict(c) for c in backbone_configs])[
    ['exp_id', 'backbone', 'init', 'epochs', 'batch_size', 'seed']])
if STAGE == 'backbones':
    backbone_results = run_experiments(backbone_configs)
    display(backbone_results.sort_values('val_macro_f1', ascending=False))

## 3. Công thức huấn luyện

Chỉ đổi một yếu tố so với T00: khởi tạo (finetune/scratch/frozen), augmentation
(basic/trivial/color), loss (CE/label smoothing/focal); bổ sung CutMix và Mixup.
T00 là baseline trên backbone thắng Bước 2. Có thể thêm cấu hình kết hợp bằng một exp_id mới.
Notebook không tự thay đổi các thí nghiệm đã hoàn tất.

In [ ]:
training_configs = None
if STAGE in ('training', 'inference', 'final'):
    backbone_winner = best_config(backbone_configs)
    recipe_base = dataclasses.replace(BASE, backbone=backbone_winner.backbone)
    training_configs = experiment_plan(recipe_base, 'training')
    display(pd.DataFrame([dataclasses.asdict(c) for c in training_configs])[
        ['exp_id', 'backbone', 'init', 'aug', 'loss', 'mix']])
    if STAGE == 'training':
        training_results = run_experiments(training_configs)
        display(training_results.sort_values('val_macro_f1', ascending=False))

## 4. So sánh suy luận trên val

I00: một view; I01: lật ngang, gộp xác suất; I02: five-crop (crop 7/8 rồi resize về
kích thước model); I03: lật ngang, gộp logit; I04: temperature scaling khớp trên val.
Tất cả đánh giá FP32. Độ trễ batch 1 đo chính các view và cách gộp trên GPU, warmup 10,
50 lần đo, CUDA synchronize; không gồm đọc/giải mã ảnh.

In [ ]:
if STAGE == 'inference':
    training_winner = best_config(training_configs)
    inference_results = compare_inference(training_winner)
    display(inference_results.sort_values(['val_macro_f1', 'val_ece'], ascending=[False, True]))
    export_results(ROOT)

## 5. Chung kết ≥3 seed

Sau khi xem kết quả val, đặt `STAGE='final'` và `ALLOW_FINAL_TEST=True`.
`SELECTED_METHOD=None` chọn macro-F1 val cao nhất, hòa thì ECE thấp nhất rồi p95 thấp nhất;
có thể đặt tên phương pháp trong bảng để chốt lựa chọn riêng.
Cấu hình được khóa ở `runs/selection.json`. Baseline T00 và F01 chạy cùng 3 seed.
File test đã có sẽ được giữ nguyên; không chạy lại test để chọn cấu hình.
Temperature được khớp riêng trên val của từng seed và áp dụng sang test.

In [ ]:
if STAGE == 'final':
    if not ALLOW_FINAL_TEST:
        print('Chốt cấu hình trên val rồi bật ALLOW_FINAL_TEST để chạy chung kết')
    else:
        training_winner = best_config(training_configs)
        idir = Path(BASE.out_dir) / 'inference' / training_winner.exp_id / f'seed{training_winner.seed}'
        candidates = [json.loads(p.read_text(encoding='utf-8')) for p in idir.glob('I*.json')]
        if len(candidates) < 5:
            raise RuntimeError('Chạy stage inference trước khi chốt chung kết')
        chosen = sorted(candidates, key=lambda r: (-r['val_macro_f1'], r['val_ece'], r['latency']['p95']))[0]
        method = SELECTED_METHOD or chosen['method']
        selection_path = lock_selection(training_winner, method)
        run_final(selection_path, training_configs[0], seeds=SEEDS)
        export_results(ROOT)

## 6. Đánh giá chuẩn và sản phẩm

`results.xlsx` chỉ lấy số từ log đã lưu; giai đoạn chưa chạy hiển thị chưa có kết quả.
`Final` dùng mean và std mẫu (`ddof=1`); `PerClass` lấy trực tiếp từ CSV dự đoán test.
Chạy `eval.py` gốc để đối chiếu, rồi viết báo cáo dựa trên số thật và dẫn nguồn tham khảo.

In [ ]:
if STAGE in ('final', 'export'):
    print('Workbook:', export_results(ROOT))
    if all(list((ROOT / 'predictions').glob(f'{tag}_seed*_test.csv')) for tag in ('F01', 'T00')):
        for tag in ('F01', 'T00'):
            subprocess.run([sys.executable, '-X', 'utf8', str(ROOT / 'eval.py'), 'score',
                '--pred', str(ROOT / f'predictions/{tag}_seed*_test.csv'),
                '--test-csv', str(Path(BASE.labels_dir) / 'test_subset0.csv'),
                '--labels', str(Path(BASE.labels_dir) / 'labels.csv'), '--tag', tag,
                '--out', str(ROOT / 'eval_out')], check=True)
        grade = [sys.executable, '-X', 'utf8', str(ROOT / 'eval.py'), 'grade',
                 '--final', str(ROOT / 'predictions/F01_seed*_test.csv'),
                 '--baseline', str(ROOT / 'predictions/T00_seed*_test.csv'),
                 '--test-csv', str(Path(BASE.labels_dir) / 'test_subset0.csv'),
                 '--labels', str(Path(BASE.labels_dir) / 'labels.csv'),
                 '--final-val', str(ROOT / 'predictions/F01_seed*_val.csv'),
                 '--out', str(ROOT / 'eval_out')]
        if list((ROOT / 'predictions').glob('F01uncal_seed*_test.csv')):
            grade += ['--uncal', str(ROOT / 'predictions/F01uncal_seed*_test.csv')]
        subprocess.run(grade, check=True)
    else:
        print('Chưa đủ dự đoán chung kết và baseline để chạy eval.py')

### Báo cáo và tái lập

Theo dàn ý GUIDE mục 6.3: thiết lập và EDA; bảng backbone; ablation; suy luận/độ trễ;
mean ± std test; phân tích Chinee Apple/Snake Weed; lựa chọn cuối và hạn chế.
Không dùng kết quả SMOKE để kết luận chất lượng model. Nộp `results.xlsx`, báo cáo,
code, notebook, `curves/` và `predictions/`; không commit ảnh, archive hoặc checkpoint.
Xem `code/README.md` để chạy CLI, test và lựa chọn profile.

Tải kết quả từ `/kaggle/working/deepweeds-lab/` sau khi hoàn tất. Khi tạo phiên Kaggle
mới, thư mục working của phiên trước có thể không còn. Muốn chạy tiếp stage sau,
hãy giữ kết quả của phiên trước và đưa checkpoint/log về cùng working trước khi chạy.
